In [ ]:
'''Biohub Lineage Forge

Research 3D lineage reconstruction with dual temporal models,
dual edge-feature TTA, and conservative DeepCenter division gating.

Research edition.'''

import os
BIOHUB_PRESET = 'harmonic_v3_division_wide'
BIOHUB_SCORE_AXIS = 'public 0.939 base + holdout-selected post-process configuration'

os.environ["BIOHUB_OUTPUT_FILTER_SHORT_TRACKS"] = "1"
os.environ["BIOHUB_DET_THRESHOLD"] = "0.965"
os.environ["BIOHUB_MOTION_RELINK_LEARNED_BONUS"] = '1.0'
os.environ["BIOHUB_ILP_APPEARANCE_WEIGHT"] = "0.0"
os.environ["BIOHUB_ILP_DISAPPEARANCE_WEIGHT"] = "2"
os.environ["BIOHUB_GAP_CLOSE_MAX_GAP"] = "2"
os.environ["BIOHUB_GAP_CLOSE_UM"] = "5.0"
os.environ["BIOHUB_GAP_DENSITY_ADAPTIVE"] = "1"
os.environ["BIOHUB_GAP_DENSITY_REFERENCE_UM"] = "6.5"
os.environ["BIOHUB_GAP_DENSITY_GAIN"] = "0.040"
os.environ["BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM"] = "0.125"
os.environ["BIOHUB_GAP_DENSITY_NEIGHBORS"] = "3"
os.environ["BIOHUB_OUTPUT_MIN_TRACK_LEN"] = "6"
os.environ["BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS"] = "1"
os.environ["BIOHUB_OUTPUT_GAP2_RECOVERY"] = "1"
os.environ["BIOHUB_SAFE_DIV_MAX_UM"] = "9.0"  


os.environ["BIOHUB_SAFE_DIV_SISTER_MAX_UM"] = "14.0"  



os.environ["BIOHUB_SAFE_DIV_SISTER_SYMMETRY_TAU"] = "0.6"  
os.environ["BIOHUB_SAFE_DIV_DIVERGE_UM"] = "2.25"  


os.environ["BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM"] = "10.0"
os.environ["BIOHUB_SAFE_DIV_FRAME_FRAC_CAP"] = "0.0076"
os.environ["BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP"] = "0.00375"

os.environ["BIOHUB_ILP_DIVISION_WEIGHT"] = "1.2"     
os.environ["BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE"] = "1"
os.environ["BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN"] = "4"
os.environ["BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB"] = "0.88"
os.environ["BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM"] = "3.0"
os.environ["BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC"] = "0.012"
os.environ["BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS"] = "120"
os.environ["BIOHUB_USE_DEEPCENTER_VETO"] = "1"
os.environ["BIOHUB_REQUIRE_DEEPCENTER_VETO"] = "1"
os.environ["BIOHUB_DEEPCENTER_EXPECTED_EPOCH"] = "2"
os.environ["BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM"] = "8.5"
os.environ["BIOHUB_DEEPCENTER_CHECKPOINT"] = "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/best.pt"
os.environ["BIOHUB_DEEPCENTER_GAP_VETO"] = "1"
os.environ["BIOHUB_DEEPCENTER_GAP_THRESHOLD"] = "0.25"
os.environ["BIOHUB_DEEPCENTER_SAFE_DIV_VETO"] = "1"
os.environ["BIOHUB_RUN_OUTPUT_DIAGNOSTICS"] = "0"
os.environ["BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT"] = "0.15"
os.environ["BIOHUB_BIDIRECTIONAL_FUSION_MODE"] = "harmonic_probability"
os.environ["BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION"] = "0.90"
os.environ["BIOHUB_DIAGNOSTIC_ARM"] = "harmonic_association_production"
os.environ["BIOHUB_VALIDATOR_N_PER_TYPE"] = "4"
os.environ["BIOHUB_PPSWEEP_SELECT_MARGIN"] = "0.001"
os.environ["BIOHUB_PPSWEEP_MAX_ADJ_LOSS"] = "0.0005"

os.environ["BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD"] = "0.20"
os.environ["BIOHUB_DEEPCENTER_TTA"] = "1"
print("BIOHUB_PRESET:", BIOHUB_PRESET)
print("BIOHUB_SCORE_AXIS:", BIOHUB_SCORE_AXIS)

In [ ]:

import json as _guard_json
import math as _guard_math
import os as _guard_os

_EXPECTED_NUMERIC = {
    "BIOHUB_DET_THRESHOLD": 0.965,
    "BIOHUB_ILP_APPEARANCE_WEIGHT": 0.0,
    "BIOHUB_ILP_DISAPPEARANCE_WEIGHT": 2,
    "BIOHUB_GAP_CLOSE_UM": 5.0,
    "BIOHUB_OUTPUT_MIN_TRACK_LEN": 6.0,
    "BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT": 0.15,
}

_EXPECTED_TEXT = {
    "BIOHUB_BIDIRECTIONAL_FUSION_MODE": "harmonic_probability",
    "BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION": "0.90",
}

_drift = {}
for _key, _want in _EXPECTED_NUMERIC.items():
    _raw = _guard_os.environ.get(_key)
    if _raw is None:
        _drift[_key] = "missing"
        continue
    _got = float(_raw)
    if not _guard_math.isclose(_got, _want, rel_tol=0.0, abs_tol=1e-12):
        _drift[_key] = {"expected": _want, "actual": _got}

for _key, _want in _EXPECTED_TEXT.items():
    _got = _guard_os.environ.get(_key)
    if _got != _want:
        _drift[_key] = {"expected": _want, "actual": _got}

if _drift:
    raise RuntimeError(
        "Configuration drift detected: " + _guard_json.dumps(_drift, sort_keys=True)
    )

print("Configuration guard: PASS")
print("Baseline: fixed-90 dual-seed clean pipeline (public LB 0.913)")
print("Single model-level change: harmonic mutual-support association fusion")
print("Reverse-time association weight: 0.200")


In [ ]:
from __future__ import annotations

import csv
import importlib.util
import json
import math
import os
import shutil
import subprocess
import tempfile
import zipfile
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
    Path(f"/kaggle/input/{COMPETITION}"),
]
COMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])

TEST_DIR = COMP_DIR / "test"

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING_DIR / "tracking_repo"
SUBMISSION_PATH = WORKING_DIR / "submission.csv"
RUN_STATS_PATH = WORKING_DIR / "run_stats.csv"

METHOD = "unet_transformer"
WEIGHTS_RELATIVE = f"weights/{METHOD}/split_0/edge_predictor_best.pth"
EXPERIMENT_TAG = "selected_101_dual_seed_near_balanced_center_confirmed_synthetic_gap"
TARGET_ARTIFACT_SLUG = os.environ.get("BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1")
PRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(
    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json",
))
ALLOW_ARTIFACT_FALLBACK = os.environ.get("BIOHUB_ALLOW_ARTIFACT_FALLBACK", "0") != "0"

DET_THRESHOLD = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.99"))
UNET_BATCH_SIZE = int(os.environ.get("BIOHUB_UNET_BATCH_SIZE", "4"))
USE_ILP = os.environ.get("BIOHUB_USE_ILP", "1") != "0"
ILP_EDGE_WEIGHT = float(os.environ.get("BIOHUB_ILP_EDGE_WEIGHT", "-1.0"))
ILP_APPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.1"))
ILP_DISAPPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "0.1"))
ILP_DIVISION_WEIGHT = float(os.environ.get("BIOHUB_ILP_DIVISION_WEIGHT", "1.0"))


SLICE = ""



ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"
RUN_OUTPUT_DIAGNOSTICS = os.environ.get("BIOHUB_RUN_OUTPUT_DIAGNOSTICS", "1") != "0"


OUTPUT_EDGE_MAX_UM = float(os.environ.get("BIOHUB_OUTPUT_EDGE_MAX_UM", "14.0"))
OUTPUT_ENFORCE_NEXT_FRAME = os.environ.get("BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME", "1") != "0"
OUTPUT_SINGLE_PARENT_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR", "1") != "0"
OUTPUT_SINGLE_CHILD_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR", "0") != "0"
OUTPUT_PRUNE_ISOLATED = os.environ.get("BIOHUB_OUTPUT_PRUNE_ISOLATED", "1") != "0"
OUTPUT_MOTION_RELINK = os.environ.get("BIOHUB_OUTPUT_MOTION_RELINK", "1") != "0"
MOTION_RELINK_TIGHT_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_TIGHT_UM", "6.0"))
MOTION_RELINK_RELAXED_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_RELAXED_UM", "10.0"))
MOTION_RELINK_VELOCITY_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT", "0.5"))
MOTION_RELINK_LEARNED_BONUS = float(os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_BONUS", "0.75"))
MOTION_RELINK_MAX_FRAME_NODES = int(os.environ.get("BIOHUB_MOTION_RELINK_MAX_FRAME_NODES", "2600"))

OUTPUT_DIVISION_GEOMETRY_FILTER = os.environ.get("BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER", "0") != "0"
DIV_PARENT_MAX_UM = float(os.environ.get("BIOHUB_DIV_PARENT_MAX_UM", "10.5"))
DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_DIV_SISTER_MAX_UM", "8.0"))
DIV_DROP_TO_SINGLE_IF_BAD = os.environ.get("BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD", "1") != "0"
OUTPUT_GAP_CLOSE = os.environ.get("BIOHUB_OUTPUT_GAP_CLOSE", "1") != "0"
GAP_CLOSE_MAX_GAP = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_GAP", "1"))
GAP_CLOSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_UM", "6.0"))
GAP_DENSITY_ADAPTIVE = os.environ.get("BIOHUB_GAP_DENSITY_ADAPTIVE", "0") != "0"
GAP_DENSITY_REFERENCE_UM = float(os.environ.get("BIOHUB_GAP_DENSITY_REFERENCE_UM", "6.5"))
GAP_DENSITY_GAIN = float(os.environ.get("BIOHUB_GAP_DENSITY_GAIN", "0.040"))
GAP_DENSITY_MAX_STEP_DELTA_UM = float(os.environ.get("BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM", "0.125"))
GAP_DENSITY_NEIGHBORS = int(os.environ.get("BIOHUB_GAP_DENSITY_NEIGHBORS", "3"))
GAP_CLOSE_REUSE_EXISTING = os.environ.get("BIOHUB_GAP_CLOSE_REUSE_EXISTING", "1") != "0"
GAP_CLOSE_REUSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_REUSE_UM", "3.2"))
GAP_CLOSE_MAX_ADDED_FRAC = float(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC", "0.05"))
GAP_CLOSE_MAX_ADDED_ABS = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_ABS", "2000"))
GAP_REFINE_SYNTHETIC = os.environ.get("BIOHUB_GAP_REFINE_SYNTHETIC", "1") != "0"
GAP_REFINE_WIN_Z = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_Z", "1"))
GAP_REFINE_WIN_YX = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_YX", "3"))
GAP_REFINE_MAX_SHIFT_UM = float(os.environ.get("BIOHUB_GAP_REFINE_MAX_SHIFT_UM", "3.2"))

OUTPUT_FILTER_SHORT_TRACKS = os.environ.get("BIOHUB_OUTPUT_FILTER_SHORT_TRACKS", "1") != "0"
OUTPUT_MIN_TRACK_LEN = int(os.environ.get("BIOHUB_OUTPUT_MIN_TRACK_LEN", "6"))
OUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get("BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS", "1") != "0"
ADAPTIVE_SHORT_TRACK_RESCUE = os.environ.get("BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE", "0") != "0"
SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC", "0.10"))
SHORT_TRACK_RESCUE_MIN_LEN = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN", "4"))
SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB", "0.82"))
SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM", "3.25"))
SHORT_TRACK_RESCUE_MAX_NODES_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC", "0.018"))
SHORT_TRACK_RESCUE_MAX_NODES_ABS = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS", "180"))

OUTPUT_LINEFIT_SMOOTH = os.environ.get("BIOHUB_OUTPUT_LINEFIT_SMOOTH", "1") != "0"
OUTPUT_LINEFIT_WEIGHT = float(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WEIGHT", "0.8"))
OUTPUT_LINEFIT_WINDOW = int(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WINDOW", "2"))

OUTPUT_GAP2_RECOVERY = os.environ.get("BIOHUB_OUTPUT_GAP2_RECOVERY", "0") != "0"
GAP2_MAX_TOTAL_UM = float(os.environ.get("BIOHUB_GAP2_MAX_TOTAL_UM", "10.2"))
GAP2_MAX_STEP_UM = float(os.environ.get("BIOHUB_GAP2_MAX_STEP_UM", "4.4"))
GAP2_MAX_LINKS_FRAC = float(os.environ.get("BIOHUB_GAP2_MAX_LINKS_FRAC", "0.0045"))
GAP2_MAX_LINKS_ABS = int(os.environ.get("BIOHUB_GAP2_MAX_LINKS_ABS", "180"))
GAP2_REQUIRE_CONTEXT = os.environ.get("BIOHUB_GAP2_REQUIRE_CONTEXT", "1") != "0"
GAP2_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_GAP2_FRAME_FRAC_CAP", "0.006"))

OUTPUT_SAFE_DIVISIONS = os.environ.get("BIOHUB_OUTPUT_SAFE_DIVISIONS", "1") != "0"
SAFE_DIV_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_MAX_UM", "4.7"))
SAFE_DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_MAX_UM", "7.2"))
SAFE_DIV_SISTER_SYMMETRY_TAU = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_SYMMETRY_TAU", "0.0"))
SAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM", "7.8"))
SAFE_DIV_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_FRAME_FRAC_CAP", "0.008"))
SAFE_DIV_GLOBAL_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP", "0.004"))


SAFE_DIV_DIVERGE_UM = float(os.environ.get("BIOHUB_SAFE_DIV_DIVERGE_UM", "2.25"))
SAFE_DIV_REQUIRE_DIVERGENCE = os.environ.get("BIOHUB_SAFE_DIV_REQUIRE_DIVERGENCE", "1") != "0"
SAFE_DIV_REQUIRE_MUTUAL_NN = os.environ.get("BIOHUB_SAFE_DIV_REQUIRE_MUTUAL_NN", "1") != "0"


USE_DEEPCENTER_VETO = os.environ.get("BIOHUB_USE_DEEPCENTER_VETO", "1") != "0"
REQUIRE_DEEPCENTER_VETO = os.environ.get("BIOHUB_REQUIRE_DEEPCENTER_VETO", "1") != "0"
DEEPCENTER_MANIFEST_DEFAULT = os.environ.get(
    "BIOHUB_DEEPCENTER_MANIFEST_DEFAULT",
    "/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/ARTIFACT_MANIFEST.json",
)
DEEPCENTER_CHECKPOINT_DEFAULT = os.environ.get(
    "BIOHUB_DEEPCENTER_CHECKPOINT_DEFAULT",
    "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/best.pt",
)
DEEPCENTER_RELATIVE = os.environ.get("BIOHUB_DEEPCENTER_RELATIVE", "weights/full_frame_center/best.pt")
DEEPCENTER_GAP_VETO = os.environ.get("BIOHUB_DEEPCENTER_GAP_VETO", "1") != "0"
DEEPCENTER_SAFE_DIV_VETO = os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_VETO", "1") != "0"
DEEPCENTER_GAP_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_GAP_THRESHOLD", "0.10"))
DEEPCENTER_EXPECTED_EPOCH = int(os.environ.get("BIOHUB_DEEPCENTER_EXPECTED_EPOCH", "0"))
DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM = float(os.environ.get("BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM", "0"))
DEEPCENTER_SAFE_DIV_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD", "0.12"))
DEEPCENTER_SCORE_WIN_Z = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_Z", "1"))
DEEPCENTER_SCORE_WIN_YX = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_YX", "2"))
DEEPCENTER_SCORE_CACHE_MAX_FRAMES = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_CACHE_MAX_FRAMES", "8"))

CONFIG_DISPLAY = {
    "experiment_tag": EXPERIMENT_TAG,
    "method": METHOD,
    "weights": WEIGHTS_RELATIVE,
    "target_artifact_slug": TARGET_ARTIFACT_SLUG,
    "primary_artifact_manifest": str(PRIMARY_ARTIFACT_MANIFEST),
    "allow_artifact_fallback": ALLOW_ARTIFACT_FALLBACK,
    "det_threshold": DET_THRESHOLD,
    "unet_batch_size": UNET_BATCH_SIZE,
    "use_ilp": USE_ILP,
    "ilp_edge_weight": ILP_EDGE_WEIGHT,
    "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,
    "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,
    "ilp_division_weight": ILP_DIVISION_WEIGHT,
    "slice": SLICE,
    "allow_pip_install": ALLOW_PIP_INSTALL,
    "output_edge_max_um": OUTPUT_EDGE_MAX_UM,
    "output_enforce_next_frame": OUTPUT_ENFORCE_NEXT_FRAME,
    "output_single_parent_repair": OUTPUT_SINGLE_PARENT_REPAIR,
    "output_single_child_repair": OUTPUT_SINGLE_CHILD_REPAIR,
    "output_prune_isolated": OUTPUT_PRUNE_ISOLATED,
    "output_motion_relink": OUTPUT_MOTION_RELINK,
    "motion_relink_tight_um": MOTION_RELINK_TIGHT_UM,
    "motion_relink_relaxed_um": MOTION_RELINK_RELAXED_UM,
    "motion_relink_velocity_weight": MOTION_RELINK_VELOCITY_WEIGHT,
    "motion_relink_learned_bonus": MOTION_RELINK_LEARNED_BONUS,
    "motion_relink_max_frame_nodes": MOTION_RELINK_MAX_FRAME_NODES,
    "output_division_geometry_filter": OUTPUT_DIVISION_GEOMETRY_FILTER,
    "div_parent_max_um": DIV_PARENT_MAX_UM,
    "div_sister_max_um": DIV_SISTER_MAX_UM,
    "div_drop_to_single_if_bad": DIV_DROP_TO_SINGLE_IF_BAD,
    "output_gap_close": OUTPUT_GAP_CLOSE,
    "gap_close_max_gap": GAP_CLOSE_MAX_GAP,
    "gap_close_effective_max_gap": min(GAP_CLOSE_MAX_GAP, 1),
    "gap_close_um": GAP_CLOSE_UM,
    "gap_density_adaptive": GAP_DENSITY_ADAPTIVE,
    "gap_density_reference_um": GAP_DENSITY_REFERENCE_UM,
    "gap_density_gain": GAP_DENSITY_GAIN,
    "gap_density_max_step_delta_um": GAP_DENSITY_MAX_STEP_DELTA_UM,
    "gap_density_neighbors": GAP_DENSITY_NEIGHBORS,
    "gap_close_reuse_existing": GAP_CLOSE_REUSE_EXISTING,
    "gap_close_reuse_um": GAP_CLOSE_REUSE_UM,
    "gap_close_max_added_frac": GAP_CLOSE_MAX_ADDED_FRAC,
    "gap_close_max_added_abs": GAP_CLOSE_MAX_ADDED_ABS,
    "gap_refine_synthetic": GAP_REFINE_SYNTHETIC,
    "gap_refine_win_z": GAP_REFINE_WIN_Z,
    "gap_refine_win_yx": GAP_REFINE_WIN_YX,
    "gap_refine_max_shift_um": GAP_REFINE_MAX_SHIFT_UM,
    "output_filter_short_tracks": OUTPUT_FILTER_SHORT_TRACKS,
    "output_min_track_len": OUTPUT_MIN_TRACK_LEN,
    "output_keep_division_components": OUTPUT_KEEP_DIVISION_COMPONENTS,
    "adaptive_short_track_rescue": ADAPTIVE_SHORT_TRACK_RESCUE,
    "short_track_rescue_trigger_removed_frac": SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC,
    "short_track_rescue_min_len": SHORT_TRACK_RESCUE_MIN_LEN,
    "short_track_rescue_min_mean_edge_prob": SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB,
    "short_track_rescue_max_mean_edge_dist_um": SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM,
    "short_track_rescue_max_nodes_frac": SHORT_TRACK_RESCUE_MAX_NODES_FRAC,
    "short_track_rescue_max_nodes_abs": SHORT_TRACK_RESCUE_MAX_NODES_ABS,
    "output_linefit_smooth": OUTPUT_LINEFIT_SMOOTH,
    "output_linefit_weight": OUTPUT_LINEFIT_WEIGHT,
    "output_linefit_window": OUTPUT_LINEFIT_WINDOW,
    "output_gap2_recovery": OUTPUT_GAP2_RECOVERY,
    "gap2_max_total_um": GAP2_MAX_TOTAL_UM,
    "gap2_max_step_um": GAP2_MAX_STEP_UM,
    "gap2_max_links_frac": GAP2_MAX_LINKS_FRAC,
    "gap2_max_links_abs": GAP2_MAX_LINKS_ABS,
    "gap2_require_context": GAP2_REQUIRE_CONTEXT,
    "gap2_frame_frac_cap": GAP2_FRAME_FRAC_CAP,
    "output_safe_divisions": OUTPUT_SAFE_DIVISIONS,
    "safe_div_max_um": SAFE_DIV_MAX_UM,
    "safe_div_sister_max_um": SAFE_DIV_SISTER_MAX_UM,
    "safe_div_existing_child_max_um": SAFE_DIV_EXISTING_CHILD_MAX_UM,
    "safe_div_frame_frac_cap": SAFE_DIV_FRAME_FRAC_CAP,
    "safe_div_global_frac_cap": SAFE_DIV_GLOBAL_FRAC_CAP,
    "use_deepcenter_add_only_gate": USE_DEEPCENTER_VETO,
    "deepcenter_gap_add_gate": DEEPCENTER_GAP_VETO,
    "deepcenter_safe_div_add_gate": DEEPCENTER_SAFE_DIV_VETO,
    "deepcenter_gap_threshold": DEEPCENTER_GAP_THRESHOLD,
    "deepcenter_expected_epoch": DEEPCENTER_EXPECTED_EPOCH,
    "deepcenter_gap_confirm_min_span_um": DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM,
    "deepcenter_safe_div_threshold": DEEPCENTER_SAFE_DIV_THRESHOLD,
    "deepcenter_checkpoint_default": DEEPCENTER_CHECKPOINT_DEFAULT,
}

print("Biohub learned UNet + node-transformer + ILP submission")
print("COMP_DIR:", COMP_DIR, "exists:", COMP_DIR.exists())
print("TEST_DIR:", TEST_DIR, "exists:", TEST_DIR.exists())
print(json.dumps(CONFIG_DISPLAY, indent=2, sort_keys=True))

In [ ]:
import re

os.environ.setdefault("POLARS_PREFER_PKG", "32")

PACKAGE_SPECS = {
    "tracksdata": ("tracksdata", "tracksdata"),
    "zarr": ("zarr", "zarr>=3.0.10,<4"),
    "pyscipopt": ("pyscipopt", "pyscipopt"),
    "geff": ("geff", "geff>=1.1.3.1.1"),
    "geff_spec": ("geff_spec", "geff-spec<1.2"),
    "ilpy": ("ilpy", "ilpy>=0.5.1"),
    "polars": ("polars", "polars>=1.36"),
    "blosc2": ("blosc2", "blosc2"),
    "dask": ("dask", "dask"),
    "imagecodecs": ("imagecodecs", "imagecodecs"),
    "skimage": ("skimage", "scikit-image>=0.24"),
    "pyarrow": ("pyarrow", "pyarrow"),
    "rustworkx": ("rustworkx", "rustworkx>=0.17.1"),
    "sqlalchemy": ("sqlalchemy", "sqlalchemy>=2"),
    "numcodecs": ("numcodecs", "numcodecs>=0.13,<0.16"),
    "donfig": ("donfig", "donfig>=0.8"),
    "google_crc32c": ("google_crc32c", "google-crc32c>=1.5"),
    "bidict": ("bidict", "bidict>=0.23.1"),
    "psygnal": ("psygnal", "psygnal>=0.14"),
    "rich": ("rich", "rich"),
    "networkx": ("networkx", "networkx>=3.2.1"),
    "pydantic": ("pydantic", "pydantic>=2.11"),
    "pydantic_core": ("pydantic_core", "pydantic-core"),
    "annotated_types": ("annotated_types", "annotated-types"),
    "typing_extensions": ("typing_extensions", "typing-extensions>=4.13"),
    "typing_inspection": ("typing_inspection", "typing-inspection"),
    "markdown_it": ("markdown_it", "markdown-it-py"),
    "pygments": ("pygments", "pygments"),
    "click": ("click", "click"),
    "cloudpickle": ("cloudpickle", "cloudpickle"),
    "fsspec": ("fsspec", "fsspec"),
    "partd": ("partd", "partd"),
    "locket": ("locket", "locket"),
    "toolz": ("toolz", "toolz"),
    "yaml": ("yaml", "pyyaml"),
    "ndindex": ("ndindex", "ndindex"),
    "msgpack": ("msgpack", "msgpack"),
    "numexpr": ("numexpr", "numexpr"),
    "deprecated": ("deprecated", "deprecated"),
    "wrapt": ("wrapt", "wrapt"),
    "imageio": ("imageio", "imageio"),
    "PIL": ("PIL", "pillow"),
    "tifffile": ("tifffile", "tifffile"),
    "lazy_loader": ("lazy_loader", "lazy-loader"),
    "tqdm": ("tqdm", "tqdm"),
}
EXTRA_SPECS_BY_NAME = {
    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],
    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],
    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],
    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],
    "polars": ["polars-runtime-32"],
    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],
    "partd": ["locket"],
    "blosc2": ["ndindex", "msgpack", "numexpr"],
    "numcodecs": ["deprecated", "msgpack", "wrapt"],
    "rich": ["markdown-it-py", "pygments"],
    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13", "typing-inspection"],
    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],
}
PIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]
REQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}
FALLBACK_ARTIFACT_SLUGS = ["biohub-tracking-support-pack-v1"]



ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"


def module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


def has_model_artifact(path: Path) -> bool:
    has_repo_dir = (path / "repo").exists()
    has_weights_dir = (path / "weights" / METHOD / "split_0" / "edge_predictor_best.pth").exists()
    has_repo_zip = (path / "repo.zip").exists()
    has_weights_zip = (path / "weights.zip").exists()
    return (has_repo_dir and has_weights_dir) or (has_repo_zip and has_weights_zip)


def artifact_manifest(path: Path) -> dict:
    manifest = path / "ARTIFACT_MANIFEST.json"
    if not manifest.exists():
        return {}
    try:
        return json.loads(manifest.read_text())
    except Exception:
        return {}


def artifact_matches_target(path: Path) -> bool:
    if ALLOW_ARTIFACT_FALLBACK:
        return True
    manifest = artifact_manifest(path)
    artifact_name = str(manifest.get("artifact_name", ""))
    path_text = str(path)
    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text


def candidate_roots_for_slug(slug: str) -> list[Path]:
    return [
        Path(f"/kaggle/input/datasets/pilkwang/{slug}"),
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/{slug}/{slug}"),
        Path(f"PublicNotebook/{slug}"),
    ]


def find_artifacts_root() -> Path:
    candidates: list[Path] = []
    for env_name in ["BIOHUB_MODEL_ARTIFACTS", "BIOHUB_ARTIFACTS"]:
        explicit = os.environ.get(env_name, "").strip()
        if explicit:
            candidates.append(Path(explicit))

    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)
    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))

    if ALLOW_ARTIFACT_FALLBACK:
        for slug in FALLBACK_ARTIFACT_SLUGS:
            candidates.extend(candidate_roots_for_slug(slug))

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if not child.is_dir():
                continue
            child_text = str(child)
            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:
                candidates.append(child)
                candidates.append(child / child.name)
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.append(grandchild)

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_model_artifact(candidate) and artifact_matches_target(candidate):
            return candidate
    checked = "\n".join(str(path) for path in candidates[:80])
    raise FileNotFoundError(
        "Could not find the required model artifact. "
        f"Expected slug: {TARGET_ARTIFACT_SLUG}\n"
        "Attach the newly uploaded support dataset, or set BIOHUB_MODEL_ARTIFACTS.\n"
        "To debug with an older artifact, set BIOHUB_ALLOW_ARTIFACT_FALLBACK=1.\n"
        "Checked:\n" + checked
    )


def _has_package_file(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    patterns = ("*.whl", "*.tar.gz", "*.zip")
    return any(any(path.glob(pattern)) for pattern in patterns)


def find_offline_package_dirs(artifacts: Path) -> list[Path]:
    candidates: list[Path] = [
        artifacts / "wheels",
        artifacts,
        Path("/kaggle/working"),
        Path("/kaggle/working/wheels"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if child.is_dir():
                candidates.extend([child / "wheels", child])
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.extend([grandchild / "wheels", grandchild])

    out: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if _has_package_file(candidate):
            out.append(candidate)
    return out


def purge_imported_modules(package_names: list[str]) -> None:
    roots = {"tracksdata"}
    for name in package_names:
        if name in PACKAGE_SPECS:
            module = PACKAGE_SPECS[name][0]
            roots.add(module.split(".")[0])
        if name == "polars":
            roots.add("polars")
    for root in roots:
        for module_name in list(sys.modules):
            if module_name == root or module_name.startswith(root + "."):
                sys.modules.pop(module_name, None)


def polars_runtime_ready() -> bool:
    try:
        import polars as _pl
        from polars._plr import PySeries as _PySeries

        _ = _PySeries
        return hasattr(_pl, "Float16") and _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64
    except Exception:
        return False


def packages_requiring_refresh() -> list[str]:
    refresh: list[str] = []
    if not module_missing("polars") and not polars_runtime_ready():
        refresh.append("polars")

    if not module_missing("zarr"):
        try:
            import zarr as _zarr
            version_text = str(getattr(_zarr, "__version__", "0"))
            major = int(version_text.split(".", 1)[0])
            if major < 3:
                refresh.append("zarr")
        except Exception:
            refresh.append("zarr")
    return refresh


def dependency_specs_for(missing: list[str]) -> list[str]:
    specs: list[str] = []
    seen: set[str] = set()

    def add(spec: str) -> None:
        key = spec.lower()
        if key not in seen:
            seen.add(key)
            specs.append(spec)

    for name in missing:
        if name in PACKAGE_SPECS:
            add(PACKAGE_SPECS[name][1])
        for spec in EXTRA_SPECS_BY_NAME.get(name, []):
            add(spec)
    return specs


def import_failures() -> dict[str, str]:
    failures: dict[str, str] = {}
    for name, module_name in REQUIRED_MODULES.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    return failures


def missing_names_from_failures(failures: dict[str, str]) -> list[str]:
    names: list[str] = []
    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}
    for message in failures.values():
        match = re.search(r"No module named ['\"]([^'\"]+)['\"]", message)
        if match:
            module = match.group(1).split(".")[0]
        else:
            match = re.search(r"module ['\"]([^'\"]+)['\"] has no attribute", message)
            if not match:
                continue
            module = match.group(1).split(".")[0]
        name = module_to_name.get(module)
        if name and name not in names:
            names.append(name)
    return names


def install_missing_dependencies(missing: list[str], artifacts: Path) -> None:
    specs = dependency_specs_for(missing)
    force_reinstall = bool({"polars", "zarr"} & set(missing))
    if not specs:
        return

    package_dirs = find_offline_package_dirs(artifacts)
    if package_dirs:
        offline_cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
        if force_reinstall:
            offline_cmd.append("--force-reinstall")
        for package_dir in package_dirs:
            offline_cmd.extend(["--find-links", str(package_dir)])
        offline_cmd.extend(specs)
        print("Installing missing packages from offline package dirs:", missing)
        print("Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.")
        print("Offline package dirs:", [str(path) for path in package_dirs])
        result = subprocess.run(offline_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("Offline dependency install succeeded.")
            return
        print("Offline dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    if ALLOW_PIP_INSTALL:
        online_cmd = [sys.executable, "-m", "pip", "install", "--no-deps"]
        if force_reinstall:
            online_cmd.append("--force-reinstall")
        online_cmd.extend(specs)
        print("Installing missing packages from PyPI:", missing)
        result = subprocess.run(online_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("PyPI dependency install succeeded.")
            return
        print("PyPI dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    command = "pip install tracksdata zarr>=3.0.10,<4 pyscipopt geff geff-spec ilpy polars blosc2 dask imagecodecs pyarrow rustworkx sqlalchemy donfig numcodecs"
    raise ImportError(
        "Missing required packages or dependency wheels: " + ", ".join(missing) + "\n"
        "Attach the support dataset with offline wheels. If supplying Kaggle dependency input instead, use:\n"
        + command + "\n"
        "Do not quote zarr>=3.0.10,<4 in Kaggle dependency input."
    )


def ensure_dependencies(artifacts: Path) -> None:
    for _ in range(5):
        refresh = packages_requiring_refresh()
        if refresh:
            install_missing_dependencies(refresh, artifacts)
            continue

        missing = [pkg for pkg, module in REQUIRED_MODULES.items() if module_missing(module)]
        if missing:
            install_missing_dependencies(missing, artifacts)
            continue

        failures = import_failures()
        if not failures:
            print("Required graph/Zarr/ILP packages import successfully.")
            return

        missing_from_import = missing_names_from_failures(failures)
        if missing_from_import:
            install_missing_dependencies(missing_from_import, artifacts)
            continue

        raise ImportError(
            "Required packages are present but failed to import. "
            "This may indicate a binary dependency mismatch in the live notebook kernel. "
            "Keep Kaggle dependency input empty and attach the wheels artifact.\n"
            + json.dumps(failures, indent=2)
        )

    failures = import_failures()
    raise ImportError(
        "Dependency recovery did not converge after repeated offline installs. "
        "The attached support artifact may be missing wheels.\n"
        + json.dumps(failures, indent=2)
    )


def remove_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)


def copy_or_extract_tree(src_dir: Path, src_zip: Path, dst: Path) -> None:
    remove_path(dst)
    if src_dir.exists() and src_dir.is_dir():
        shutil.copytree(src_dir, dst)
        return
    if src_zip.exists() and src_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src_zip) as zf:
            zf.extractall(dst)
        return
    raise FileNotFoundError(f"Missing source tree or zip: {src_dir} / {src_zip}")


def link_or_copy_tree(src: Path, dst: Path) -> None:
    remove_path(dst)
    try:
        os.symlink(src, dst, target_is_directory=True)
    except Exception:
        shutil.copytree(src, dst)


def materialize_inference_repo(artifacts: Path) -> None:
    copy_or_extract_tree(artifacts / "repo", artifacts / "repo.zip", REPO_DIR)

    weights_src = artifacts / "weights"
    weights_zip = artifacts / "weights.zip"
    weights_dst = REPO_DIR / "weights"
    if weights_src.exists() and weights_src.is_dir():
        link_or_copy_tree(weights_src, weights_dst)
    elif weights_zip.exists() and weights_zip.is_file():
        remove_path(weights_dst)
        weights_dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(weights_zip) as zf:
            zf.extractall(weights_dst)
    else:
        raise FileNotFoundError(f"Missing weights tree or zip under {artifacts}")

    required = [
        REPO_DIR / "scripts" / "predict_unet_transformer.py",
        REPO_DIR / WEIGHTS_RELATIVE,
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Materialized inference repo is incomplete:\n" + "\n".join(missing))
    print("Inference repo:", REPO_DIR)
    print("Weights:", REPO_DIR / WEIGHTS_RELATIVE)


ARTIFACTS = find_artifacts_root()
print("ARTIFACTS:", ARTIFACTS)
print("Has offline wheels:", (ARTIFACTS / "wheels").exists())
manifest_info = artifact_manifest(ARTIFACTS)
if manifest_info:
    print("Artifact name:", manifest_info.get("artifact_name"))
    print("Weight sha256:", manifest_info.get("model", {}).get("weight_sha256"))
    print("Weight path:", manifest_info.get("model", {}).get("weight_path"))
    _expected_primary_sha256 = "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771"
    _actual_primary_sha256 = str(manifest_info.get("model", {}).get("weight_sha256", ""))
    if _actual_primary_sha256 != _expected_primary_sha256:
        raise RuntimeError(
            "Primary model checksum mismatch: "
            f"expected {_expected_primary_sha256}, got {_actual_primary_sha256 or 'missing'}"
        )

ensure_dependencies(ARTIFACTS)
materialize_inference_repo(ARTIFACTS)




import hashlib as _integrity_hashlib

_support_expected_sha256 = {
    "scripts/augmentations.py": "13db09817bf492f8d0f710a0a4d09776320b262060167055090a303fc6057f4e",
    "scripts/dataspec.py": "e69bf952fb985477ac50ff8598a35020c95d20a035a09b81ab4056e655dd311f",
    "scripts/evaluate.py": "614813cc51c3581c6ccda4bb20725a19da8ecac4a27620654bfca58319cffa3c",
    "scripts/predict_unet_transformer.py": "c44e771ba5980b820f93091e03a303c25dfe8f3232e501f54dc9565731c234b9",
    "scripts/train_unet_transformer.py": "c4f6317736bb3bb1ec8f3f6e9a6d935a463e3f0f1f685481b2d13218d35dc9ea",
    "src/biohub_tracking/__init__.py": "26a18d8da84e40da73281a48ebc3017d847a2e57431ab63e8629d2109e6e8571",
    "src/biohub_tracking/division_metrics.py": "d1cf1e0a43009d02174f1699ce2aa28458a2220ac4b521731d3bcf31cf8c76be",
    "src/biohub_tracking/img_proc.py": "00e8ef0adc8b39f1aaaa547ea6197b906bf9e8c009e339d3e95f8f8dbf31be3f",
    "src/biohub_tracking/io.py": "efae135b088cecaab463d889f16c885ef6da3ad27b0747327d8ddc28d866b7bd",
    "src/biohub_tracking/metrics.py": "31baf45b54c78f68bab4f65dd8f4b38bca702abb644171c6df7c46cdeef55d83",
    "src/biohub_tracking/models/__init__.py": "ab7587ef79856bae50d24b62e5805092d0459ee1c586522b763f9ef70c093e1d",
    "src/biohub_tracking/models/simple_node_transformer.py": "b97209edeb03840e80d903e3e2a8c81c520641c8ef343f6ca2904d0f80db064e",
    "src/biohub_tracking/models/temporal_unet.py": "d809c35d42f504161074ddeaaa7aee5b407e5bca7f9b4e1d5f9b2ff345666cac"
}
_support_expected_manifest_sha256 = "978b626d1fd1e7397435a437dfe68691defe1572fc3c20e61012d7c9b52ed029"
_primary_expected_sha256 = "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771"
_deepcenter_expected_sha256 = "8040999a92f6b7bbd98fa8cf458141e045c0f9ad7c936bdb3b18e1f7edafe2a0"  


def _integrity_sha256_file(path: Path) -> str:
    digest = _integrity_hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


_support_materialized_paths = {
    path.relative_to(REPO_DIR).as_posix(): path
    for path in REPO_DIR.rglob("*.py")
}
_support_actual_names = set(_support_materialized_paths)
_support_expected_names = set(_support_expected_sha256)
if _support_actual_names != _support_expected_names:
    raise RuntimeError({
        "support_repo_python_files_missing": sorted(
            _support_expected_names - _support_actual_names
        ),
        "support_repo_python_files_extra": sorted(
            _support_actual_names - _support_expected_names
        ),
    })
_support_actual_sha256 = {
    relative: _integrity_sha256_file(_support_materialized_paths[relative])
    for relative in sorted(_support_materialized_paths)
}
if _support_actual_sha256 != _support_expected_sha256:
    raise RuntimeError({
        "support_repo_python_checksum_mismatch": {
            relative: {
                "expected": _support_expected_sha256[relative],
                "actual": _support_actual_sha256[relative],
            }
            for relative in sorted(_support_expected_sha256)
            if _support_actual_sha256[relative]
            != _support_expected_sha256[relative]
        }
    })
_support_manifest_bytes = "".join(
    f"{_support_actual_sha256[relative]}  {relative}\n"
    for relative in sorted(_support_actual_sha256)
).encode("utf-8")
_support_actual_manifest_sha256 = _integrity_hashlib.sha256(
    _support_manifest_bytes
).hexdigest()
if _support_actual_manifest_sha256 != _support_expected_manifest_sha256:
    raise RuntimeError(
        "Support repo manifest checksum mismatch: "
        f"expected {_support_expected_manifest_sha256}, "
        f"got {_support_actual_manifest_sha256}"
    )

_primary_materialized_path = REPO_DIR / WEIGHTS_RELATIVE
_primary_actual_sha256 = _integrity_sha256_file(_primary_materialized_path)
if _primary_actual_sha256 != _primary_expected_sha256:
    raise RuntimeError(
        "Materialized primary model checksum mismatch: "
        f"expected {_primary_expected_sha256}, got {_primary_actual_sha256}"
    )

_deepcenter_candidate_strings = [
    os.environ.get("BIOHUB_DEEPCENTER_CHECKPOINT", "").strip(),
    "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/"
    "full_frame_center/best.pt",
    "/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/"
    "weights/full_frame_center/best.pt",
]
_deepcenter_candidates = []
for _candidate_string in _deepcenter_candidate_strings:
    if not _candidate_string:
        continue
    _candidate_path = Path(_candidate_string)
    if _candidate_path not in _deepcenter_candidates:
        _deepcenter_candidates.append(_candidate_path)
_deepcenter_materialized_path = next(
    (path for path in _deepcenter_candidates if path.is_file()),
    None,
)
if _deepcenter_materialized_path is None:
    raise FileNotFoundError({
        "missing_deepcenter_checkpoint": [str(path) for path in _deepcenter_candidates]
    })
_deepcenter_actual_sha256 = _integrity_sha256_file(
    _deepcenter_materialized_path
)
if _deepcenter_actual_sha256 != _deepcenter_expected_sha256:
    raise RuntimeError(
        "DeepCenter checkpoint checksum mismatch: "
        f"expected {_deepcenter_expected_sha256}, "
        f"got {_deepcenter_actual_sha256}"
    )
os.environ["BIOHUB_DEEPCENTER_CHECKPOINT"] = str(
    _deepcenter_materialized_path
)

print("Support repo Python manifest SHA256:", _support_actual_manifest_sha256)
print("Primary materialized SHA256:", _primary_actual_sha256)
print("DeepCenter materialized SHA256:", _deepcenter_actual_sha256)



import hashlib as _hashlib

_secondary_manifest_explicit = Path(os.environ.get(
    "BIOHUB_SECONDARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-temporal-unet3d-seed314159-v1/ARTIFACT_MANIFEST.json",
))
_secondary_expected_sha256 = "9bac2fa0dadc4a6fc1899e0caf187f4b553e0a7cd90ba1261a68b35ffe9e305f"
_secondary_slug = "biohub-temporal-unet3d-seed314159-v1"


def _find_secondary_artifact_root() -> tuple[Path, dict]:
    candidates = [
        _secondary_manifest_explicit,
        Path(f"/kaggle/input/{_secondary_slug}/ARTIFACT_MANIFEST.json"),
        Path(f"/kaggle/input/datasets/pilkwang/{_secondary_slug}/ARTIFACT_MANIFEST.json"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend(input_root.rglob("ARTIFACT_MANIFEST.json"))

    seen = set()
    for manifest_path in candidates:
        manifest_path = manifest_path.expanduser()
        if manifest_path in seen or not manifest_path.is_file():
            continue
        seen.add(manifest_path)
        try:
            info = json.loads(manifest_path.read_text())
        except Exception:
            continue
        sha256 = str(info.get("model", {}).get("weight_sha256", ""))
        if sha256 == _secondary_expected_sha256:
            return manifest_path.parent, info
    raise FileNotFoundError(
        "Could not find the independent-seed artifact with weight SHA256 "
        + _secondary_expected_sha256
    )


SECONDARY_ARTIFACTS, secondary_manifest_info = _find_secondary_artifact_root()
SECONDARY_WEIGHTS_ROOT = WORKING_DIR / "secondary_seed_weights"
copy_or_extract_tree(
    SECONDARY_ARTIFACTS / "weights",
    SECONDARY_ARTIFACTS / "weights.zip",
    SECONDARY_WEIGHTS_ROOT,
)
SECONDARY_WEIGHTS_PATH = (
    SECONDARY_WEIGHTS_ROOT
    / "unet_transformer"
    / "split_0"
    / "edge_predictor_best.pth"
)
SECONDARY_CONFIG_PATH = SECONDARY_WEIGHTS_PATH.parent / "config.json"
for _required_secondary_path in (SECONDARY_WEIGHTS_PATH, SECONDARY_CONFIG_PATH):
    if not _required_secondary_path.is_file():
        raise FileNotFoundError(f"Missing secondary model file: {_required_secondary_path}")


def _sha256_file(path: Path) -> str:
    digest = _hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


_secondary_actual_sha256 = _sha256_file(SECONDARY_WEIGHTS_PATH)
if _secondary_actual_sha256 != _secondary_expected_sha256:
    raise RuntimeError(
        "Secondary model checksum mismatch: "
        f"expected {_secondary_expected_sha256}, got {_secondary_actual_sha256}"
    )

os.environ["BIOHUB_SECONDARY_WEIGHTS"] = str(SECONDARY_WEIGHTS_PATH)
os.environ["BIOHUB_SECONDARY_EDGE_WEIGHT"] = "0.15"
print("Secondary artifact:", SECONDARY_ARTIFACTS)
print("Secondary weight:", SECONDARY_WEIGHTS_PATH)
print("Secondary SHA256:", _secondary_actual_sha256)
print("Secondary edge-logit weight:", os.environ["BIOHUB_SECONDARY_EDGE_WEIGHT"])

os.environ["BIOHUB_SECONDARY_DETECTION_WEIGHT"] = "0.80"  
os.environ["BIOHUB_SECONDARY_LINK_MODE"] = "low_margin_consensus"
os.environ["BIOHUB_SECONDARY_MIX_TEMPERATURE"] = "1"
os.environ["BIOHUB_SECONDARY_LOW_MARGIN_MAX"] = "0.35"
os.environ["BIOHUB_DUAL_SEED_EDGE_THRESHOLD"] = "0.48"

_runtime_integrity_receipt = {
    "status": "complete_label_free_runtime_integrity",
    "verified_before_dynamic_source_patch": True,
    "support_repo_python_file_count": len(_support_actual_sha256),
    "support_repo_python_sha256": _support_actual_sha256,
    "support_repo_python_manifest_sha256": _support_actual_manifest_sha256,
    "checkpoint_sha256": {
        "primary": _primary_actual_sha256,
        "secondary": _secondary_actual_sha256,
        "deepcenter": _deepcenter_actual_sha256,
    },
    "materialized_paths": {
        "primary": str(_primary_materialized_path),
        "secondary": str(SECONDARY_WEIGHTS_PATH),
        "deepcenter": str(_deepcenter_materialized_path),
    },
    "ground_truth_accessed": False,
}
_runtime_integrity_receipt_path = (
    WORKING_DIR / "bidirectional_production_runtime_integrity.json"
)
_runtime_integrity_receipt_path.write_text(
    json.dumps(_runtime_integrity_receipt, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print("Runtime integrity receipt:", _runtime_integrity_receipt_path)


In [ ]:
import tracksdata as td
import numpy as np
import blosc2
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree

SUBMISSION_COLUMNS = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
CSV_COLUMNS = ["id", *SUBMISSION_COLUMNS]
VOXEL_SCALE_UM = (1.625, 0.40625, 0.40625)


def graph_from_geff(path: Path):
    graph = td.graph.IndexedRXGraph.from_geff(path)
    return graph[0] if isinstance(graph, tuple) else graph


def edge_distance_um(source: dict[str, object], target: dict[str, object]) -> float:
    dz = (float(source["z"]) - float(target["z"])) * VOXEL_SCALE_UM[0]
    dy = (float(source["y"]) - float(target["y"])) * VOXEL_SCALE_UM[1]
    dx = (float(source["x"]) - float(target["x"])) * VOXEL_SCALE_UM[2]
    return math.sqrt(dz * dz + dy * dy + dx * dx)


def point_distance_um(a: tuple[float, float, float], b: tuple[float, float, float]) -> float:
    dz = (a[0] - b[0]) * VOXEL_SCALE_UM[0]
    dy = (a[1] - b[1]) * VOXEL_SCALE_UM[1]
    dx = (a[2] - b[2]) * VOXEL_SCALE_UM[2]
    return math.sqrt(dz * dz + dy * dy + dx * dx)


def node_point(node: dict[str, object]) -> tuple[float, float, float]:
    return (float(node["z"]), float(node["y"]), float(node["x"]))


def edge_sort_key(edge: dict[str, object]) -> tuple[float, float]:
    prob = edge.get("edge_prob")
    prob_value = float(prob) if prob is not None else 0.0
    return prob_value, -float(edge["distance_um"])


def _next_node_id(nodes_by_id: dict[int, dict[str, object]]) -> int:
    return max(nodes_by_id) + 1 if nodes_by_id else 1



def read_test_frame(dataset: str, t: int, frame_cache: dict[int, np.ndarray]) -> np.ndarray:
    if t in frame_cache:
        return frame_cache[t]
    zarr_path = TEST_DIR / f"{dataset}.zarr"
    meta = json.loads((zarr_path / "0" / "zarr.json").read_text())
    shape = tuple(int(v) for v in meta["shape"])
    dtype = np.dtype(meta["data_type"])
    frame_shape = shape[1:]
    chunk_path = zarr_path / "0" / "c" / str(t) / "0" / "0" / "0"
    try:
        raw = chunk_path.read_bytes()
        arr = np.frombuffer(blosc2.decompress(raw), dtype=dtype)
        if arr.size == int(np.prod(frame_shape)):
            frame = arr.reshape(frame_shape).copy()
            frame_cache[t] = frame
            return frame
    except Exception:
        pass
    import zarr
    frame = np.asarray(zarr.open(zarr_path / "0", mode="r")[t])
    frame_cache[t] = frame
    return frame


def refine_synthetic_midpoint(
    dataset: str | None,
    t: int,
    midpoint: tuple[float, float, float],
    frame_cache: dict[int, np.ndarray],
    stats: dict[str, int],
) -> tuple[float, float, float]:
    if not GAP_REFINE_SYNTHETIC or dataset is None:
        return midpoint
    try:
        frame = read_test_frame(dataset, t, frame_cache)
        z, y, x = [int(round(v)) for v in midpoint]
        z0 = max(0, z - GAP_REFINE_WIN_Z)
        z1 = min(frame.shape[0], z + GAP_REFINE_WIN_Z + 1)
        y0 = max(0, y - GAP_REFINE_WIN_YX)
        y1 = min(frame.shape[1], y + GAP_REFINE_WIN_YX + 1)
        x0 = max(0, x - GAP_REFINE_WIN_YX)
        x1 = min(frame.shape[2], x + GAP_REFINE_WIN_YX + 1)
        patch = frame[z0:z1, y0:y1, x0:x1].astype(np.float64)
        if patch.size == 0:
            stats["gap_refine_failed"] += 1
            return midpoint
        baseline = float(np.percentile(patch, 20.0))
        weights = np.maximum(patch - baseline, 0.0)
        total = float(weights.sum())
        if total <= 0:
            stats["gap_refine_failed"] += 1
            return midpoint
        zz = np.arange(z0, z1, dtype=np.float64)[:, None, None]
        yy = np.arange(y0, y1, dtype=np.float64)[None, :, None]
        xx = np.arange(x0, x1, dtype=np.float64)[None, None, :]
        refined = (
            float((weights * zz).sum() / total),
            float((weights * yy).sum() / total),
            float((weights * xx).sum() / total),
        )
        if point_distance_um(refined, midpoint) > GAP_REFINE_MAX_SHIFT_UM:
            stats["gap_refine_rejected_shift"] += 1
            return midpoint
        stats["gap_refined_synthetic"] += 1
        return refined
    except Exception:
        stats["gap_refine_failed"] += 1
        return midpoint



def _dc_pool_frame_xy(volume: np.ndarray, factor: int) -> np.ndarray:
    if factor <= 1:
        return volume.astype(np.float32, copy=False)
    z, y, x = volume.shape
    y2 = (y // factor) * factor
    x2 = (x // factor) * factor
    cropped = volume[:, :y2, :x2].astype(np.float32, copy=False)
    return cropped.reshape(z, y2 // factor, factor, x2 // factor, factor).mean(axis=(2, 4))


def _dc_normalize_dynamic_range(volume: np.ndarray, cfg: object) -> np.ndarray:
    vol = np.asarray(volume, dtype=np.float32)
    lo = float(np.percentile(vol, float(getattr(cfg, "norm_lo_pct", 50.0))))
    hi = float(np.percentile(vol, float(getattr(cfg, "norm_hi_pct", 99.5))))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(vol, dtype=np.float32)
    ratio = (vol - lo) / (hi - lo)
    return np.clip(
        ratio,
        float(getattr(cfg, "norm_clip_lo", -0.5)),
        float(getattr(cfg, "norm_clip_hi", 6.0)),
    ).astype(np.float32)


def _dc_manifest_weight_paths(manifest_path: Path) -> list[Path]:
    if not manifest_path.exists():
        return []
    try:
        manifest = json.loads(manifest_path.read_text())
    except Exception as exc:
        print("Could not read DeepCenter manifest:", manifest_path, type(exc).__name__, exc)
        return []
    root = manifest_path.parent
    sections: list[dict[str, object]] = []
    for section in [
        manifest.get("model", {}),
        manifest.get("models", {}).get("full_frame_center", {}) if isinstance(manifest.get("models", {}), dict) else {},
        manifest.get("full_frame_center", {}),
    ]:
        if isinstance(section, dict):
            sections.append(section)
    candidates: list[Path] = []
    for section in sections:
        for key in ("weight_path", "path"):
            rel = section.get(key)
            if isinstance(rel, str) and rel:
                candidates.append(root / rel)
        for key in ("last_checkpoint", "best_checkpoint"):
            item = section.get(key)
            if isinstance(item, dict):
                rel = item.get("path")
                if isinstance(rel, str) and rel:
                    candidates.append(root / rel)
    for name in ("checkpoint_last.pt", "best.pt", "last.pt"):
        candidates.append(root / "weights" / "full_frame_center" / name)
        candidates.append(root / name)
    candidates.append(root / DEEPCENTER_RELATIVE)
    return candidates


def _dc_checkpoint_candidates() -> list[Path]:
    candidates: list[Path] = []
    explicit = os.environ.get("BIOHUB_DEEPCENTER_CHECKPOINT", DEEPCENTER_CHECKPOINT_DEFAULT).strip()
    if explicit:
        candidates.append(Path(explicit))
    manifest_explicit = os.environ.get("BIOHUB_DEEPCENTER_MANIFEST", DEEPCENTER_MANIFEST_DEFAULT).strip()
    if manifest_explicit:
        candidates.extend(_dc_manifest_weight_paths(Path(manifest_explicit)))

    input_root = Path("/kaggle/input")
    preferred_dirs = [
        Path("/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1"),
        Path("/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1"),
    ]
    for directory in preferred_dirs:
        candidates.extend(_dc_manifest_weight_paths(directory / "ARTIFACT_MANIFEST.json"))
        for name in ("checkpoint_last.pt", "best.pt", "last.pt"):
            candidates.append(directory / "weights" / "full_frame_center" / name)
            candidates.append(directory / name)
    if input_root.exists():
        for name in ("checkpoint_last.pt", "best.pt", "last.pt"):
            candidates.extend(sorted(input_root.glob(f"**/full_frame_center/**/{name}")))

    seen: set[Path] = set()
    out: list[Path] = []
    for path in candidates:
        path = path.expanduser()
        try:
            key = path.resolve() if path.exists() else path
        except Exception:
            key = path
        if key in seen:
            continue
        seen.add(key)
        out.append(path)
    return out


try:
    import torch
except Exception as _dc_torch_error:
    torch = None


if torch is not None:
    class _DCConvBlock3d(torch.nn.Module):
        def __init__(self, in_channels: int, out_channels: int) -> None:
            super().__init__()
            groups = min(8, out_channels)
            self.block = torch.nn.Sequential(
                torch.nn.Conv3d(in_channels, out_channels, 3, padding=1, bias=False),
                torch.nn.GroupNorm(groups, out_channels),
                torch.nn.SiLU(inplace=True),
                torch.nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False),
                torch.nn.GroupNorm(groups, out_channels),
                torch.nn.SiLU(inplace=True),
            )

        def forward(self, x):
            return self.block(x)


    class _DCDeepCenterUNet3D(torch.nn.Module):
        def __init__(self, in_channels: int = 1, base_channels: int = 24) -> None:
            super().__init__()
            c = int(base_channels)
            self.enc1 = _DCConvBlock3d(in_channels, c)
            self.down1 = torch.nn.MaxPool3d(2, 2)
            self.enc2 = _DCConvBlock3d(c, c * 2)
            self.down2 = torch.nn.MaxPool3d(2, 2)
            self.enc3 = _DCConvBlock3d(c * 2, c * 4)
            self.down3 = torch.nn.MaxPool3d(2, 2)
            self.bottleneck = _DCConvBlock3d(c * 4, c * 8)
            self.up3 = torch.nn.ConvTranspose3d(c * 8, c * 4, 2, 2)
            self.dec3 = _DCConvBlock3d(c * 8, c * 4)
            self.up2 = torch.nn.ConvTranspose3d(c * 4, c * 2, 2, 2)
            self.dec2 = _DCConvBlock3d(c * 4, c * 2)
            self.up1 = torch.nn.ConvTranspose3d(c * 2, c, 2, 2)
            self.dec1 = _DCConvBlock3d(c * 2, c)
            self.head = torch.nn.Conv3d(c, 1, 1)

        def forward(self, x):
            e1 = self.enc1(x)
            e2 = self.enc2(self.down1(e1))
            e3 = self.enc3(self.down2(e2))
            b = self.bottleneck(self.down3(e3))
            d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
            d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
            d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
            return self.head(d1)
else:
    _DCConvBlock3d = None
    _DCDeepCenterUNet3D = None

def load_deepcenter_veto_detector() -> dict[str, object] | None:
    if not USE_DEEPCENTER_VETO:
        print("DeepCenter add-only repair gate disabled by configuration.")
        return None
    if torch is None:
        if REQUIRE_DEEPCENTER_VETO:
            raise ImportError("torch is required for DeepCenter add-only repair gate")
        print("DeepCenter add-only repair gate skipped because torch is unavailable.")
        return None
    from types import SimpleNamespace

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    load_errors: list[str] = []
    for checkpoint_path in _dc_checkpoint_candidates():
        if not checkpoint_path.exists():
            continue
        try:
            print("Trying DeepCenter add-only gate checkpoint:", checkpoint_path)
            checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
            if not isinstance(checkpoint, dict) or "model_state" not in checkpoint:
                raise ValueError("checkpoint has no model_state")
            checkpoint_epoch = int(checkpoint.get("epoch", -1))
            if DEEPCENTER_EXPECTED_EPOCH > 0 and checkpoint_epoch != DEEPCENTER_EXPECTED_EPOCH:
                raise ValueError(
                    f"expected DeepCenter epoch {DEEPCENTER_EXPECTED_EPOCH}, got {checkpoint_epoch}"
                )
            cfg = SimpleNamespace(**checkpoint.get("config", {}))
            model = _DCDeepCenterUNet3D(base_channels=int(getattr(cfg, "base_channels", 24)))
            model.load_state_dict(checkpoint["model_state"])
            model.to(device)
            model.eval()
            print("Loaded DeepCenter add-only gate checkpoint:", checkpoint_path)
            print("DeepCenter checkpoint epoch:", checkpoint.get("epoch"), "best_score:", checkpoint.get("best_score"))
            return {
                "model": model,
                "cfg": cfg,
                "device": device,
                "path": checkpoint_path,
                "torch": torch,
            }
        except Exception as exc:
            load_errors.append(f"{checkpoint_path}: {type(exc).__name__}: {exc}")
            print("Skipping incompatible DeepCenter checkpoint:", checkpoint_path, "|", type(exc).__name__, exc)
    message = "No usable DeepCenter checkpoint found for add-only repair gate."
    if REQUIRE_DEEPCENTER_VETO:
        checked = "\n".join(str(p) for p in _dc_checkpoint_candidates()[:80])
        errors = "\n".join(load_errors[-20:])
        raise FileNotFoundError(message + "\nChecked:\n" + checked + ("\nLoad errors:\n" + errors if errors else ""))
    print(message)
    return None


def _dc_cache_trim(cache: dict[tuple[str, int], np.ndarray]) -> None:
    limit = max(1, int(DEEPCENTER_SCORE_CACHE_MAX_FRAMES))
    while len(cache) > limit:
        cache.pop(next(iter(cache)))


def deepcenter_heatmap_for_frame(
    dataset: str,
    t: int,
    detector_bundle: dict[str, object] | None,
    frame_cache: dict[int, np.ndarray],
    heatmap_cache: dict[tuple[str, int], np.ndarray],
) -> np.ndarray | None:
    if detector_bundle is None:
        return None
    key = (dataset, int(t))
    cached = heatmap_cache.get(key)
    if cached is not None:
        return cached
    model = detector_bundle["model"]
    cfg = detector_bundle["cfg"]
    device = detector_bundle["device"]
    torch_mod = detector_bundle["torch"]
    pool_factor = int(getattr(cfg, "pool_factor", 4))
    volume = read_test_frame(dataset, int(t), frame_cache)
    pooled = _dc_pool_frame_xy(volume, pool_factor)
    image = _dc_normalize_dynamic_range(pooled, cfg)
    with torch_mod.no_grad():
        tensor = torch_mod.from_numpy(image[None, None, ...]).to(device=device, dtype=torch_mod.float32)
        logits = model(tensor)
        
        
        
        
        
        if os.environ.get("BIOHUB_DEEPCENTER_TTA", "0") != "0":
            acc = logits.clone(); nv = 1
            for dims in [(-1,), (-2,), (-2, -1)]:
                acc = acc + model(tensor.flip(dims)).flip(dims); nv += 1
            if tensor.shape[-1] == tensor.shape[-2]:
                for k in (1, 3):
                    acc = acc + torch_mod.rot90(model(torch_mod.rot90(tensor, k, dims=(-2, -1))), -k, dims=(-2, -1)); nv += 1
                acc = acc + model(tensor.transpose(-1, -2)).transpose(-1, -2); nv += 1
                at = torch_mod.rot90(tensor, 1, dims=(-2, -1)).transpose(-1, -2)
                acc = acc + torch_mod.rot90(model(at).transpose(-1, -2), -1, dims=(-2, -1)); nv += 1
            delta = float((acc / nv - logits).abs().mean())
            if delta == 0.0:
                raise RuntimeError("DEEPCENTER_TTA_NO_OP: averaged veto logits identical to the single view")
            if not getattr(deepcenter_heatmap_for_frame, "_tta_announced", False):
                print("DEEPCENTER_TTA_ACTIVE views=", nv, "mean_abs_logit_delta=", round(delta, 6), flush=True)
                deepcenter_heatmap_for_frame._tta_announced = True
            logits = acc / nv
        heatmap = torch_mod.sigmoid(logits)[0, 0].detach().cpu().numpy().astype(np.float32, copy=False)
    heatmap_cache[key] = heatmap
    _dc_cache_trim(heatmap_cache)
    return heatmap


def deepcenter_score_point(
    dataset: str | None,
    t: int,
    point: tuple[float, float, float],
    detector_bundle: dict[str, object] | None,
    frame_cache: dict[int, np.ndarray],
    heatmap_cache: dict[tuple[str, int], np.ndarray],
) -> float | None:
    if not USE_DEEPCENTER_VETO or detector_bundle is None or dataset is None:
        return None
    heatmap = deepcenter_heatmap_for_frame(dataset, int(t), detector_bundle, frame_cache, heatmap_cache)
    if heatmap is None or heatmap.size == 0:
        return None
    cfg = detector_bundle["cfg"]
    pool_factor = int(getattr(cfg, "pool_factor", 4))
    z = int(round(float(point[0])))
    y = int(round(float(point[1]) / max(pool_factor, 1)))
    x = int(round(float(point[2]) / max(pool_factor, 1)))
    z0, z1 = max(0, z - DEEPCENTER_SCORE_WIN_Z), min(heatmap.shape[0], z + DEEPCENTER_SCORE_WIN_Z + 1)
    y0, y1 = max(0, y - DEEPCENTER_SCORE_WIN_YX), min(heatmap.shape[1], y + DEEPCENTER_SCORE_WIN_YX + 1)
    x0, x1 = max(0, x - DEEPCENTER_SCORE_WIN_YX), min(heatmap.shape[2], x + DEEPCENTER_SCORE_WIN_YX + 1)
    patch = heatmap[z0:z1, y0:y1, x0:x1]
    if patch.size == 0:
        return None
    score = float(np.max(patch))
    return score if np.isfinite(score) else None


def deepcenter_accept_repair_point(
    dataset: str | None,
    t: int,
    point: tuple[float, float, float],
    detector_bundle: dict[str, object] | None,
    frame_cache: dict[int, np.ndarray],
    heatmap_cache: dict[tuple[str, int], np.ndarray],
    stats: dict[str, int],
    prefix: str,
    threshold: float,
) -> bool:
    if not USE_DEEPCENTER_VETO:
        return True
    if detector_bundle is None or dataset is None:
        stats[f"deepcenter_{prefix}_missing"] += 1
        return True
    stats[f"deepcenter_{prefix}_checked"] += 1
    score = deepcenter_score_point(dataset, int(t), point, detector_bundle, frame_cache, heatmap_cache)
    if score is None:
        stats[f"deepcenter_{prefix}_missing"] += 1
        return True
    if score < float(threshold):
        stats[f"deepcenter_{prefix}_rejected"] += 1
        return False
    stats[f"deepcenter_{prefix}_accepted"] += 1
    return True

def _position_um(node: dict[str, object]) -> np.ndarray:
    return np.array(
        [float(node["z"]) * VOXEL_SCALE_UM[0], float(node["y"]) * VOXEL_SCALE_UM[1], float(node["x"]) * VOXEL_SCALE_UM[2]],
        dtype=np.float64,
    )


def motion_relink_edges(
    nodes_by_id: dict[int, dict[str, object]],
    stats: dict[str, int],
    learned_edge_probs: dict[tuple[int, int], float] | None = None,
) -> list[dict[str, object]]:
    if not OUTPUT_MOTION_RELINK or not nodes_by_id:
        return []

    learned_edge_probs = learned_edge_probs or {}

    def learned_prob(source_id: int, target_id: int) -> float:
        value = learned_edge_probs.get((source_id, target_id), 0.0)
        try:
            value = float(value)
        except (TypeError, ValueError):
            return 0.0
        if not np.isfinite(value):
            return 0.0
        if value < 0.0 or value > 1.0:
            value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))
        return float(np.clip(value, 0.0, 1.0))

    ids_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        ids_by_t.setdefault(int(node["t"]), []).append(node_id)
    for ids in ids_by_t.values():
        ids.sort()

    frame_sizes = [len(ids) for ids in ids_by_t.values()]
    if frame_sizes and max(frame_sizes) > MOTION_RELINK_MAX_FRAME_NODES:
        stats["motion_relink_skipped_large_frame"] = 1
        return []

    position_um = {node_id: _position_um(node) for node_id, node in nodes_by_id.items()}
    predecessor_position_um: dict[int, np.ndarray] = {}
    selected_edges: list[dict[str, object]] = []

    def assign_pass(
        source_ids: list[int],
        target_ids: list[int],
        gate_um: float,
    ) -> list[tuple[int, int, float, float, float]]:
        if not source_ids or not target_ids:
            return []
        big = gate_um * 1000.0 + 1.0
        cost = np.full((len(source_ids), len(target_ids)), big, dtype=np.float64)
        raw_dist = np.full_like(cost, np.inf)
        motion_dist = np.full_like(cost, np.inf)
        prob_matrix = np.zeros_like(cost)
        for i, source_id in enumerate(source_ids):
            source_pos = position_um[source_id]
            prev_pos = predecessor_position_um.get(source_id)
            if prev_pos is None:
                predicted = source_pos
            else:
                predicted = source_pos + MOTION_RELINK_VELOCITY_WEIGHT * (source_pos - prev_pos)
            for j, target_id in enumerate(target_ids):
                target_pos = position_um[target_id]
                raw = float(np.linalg.norm(target_pos - source_pos))
                if raw > gate_um:
                    continue
                motion = float(np.linalg.norm(target_pos - predicted))
                prob = learned_prob(source_id, target_id)
                raw_dist[i, j] = raw
                motion_dist[i, j] = motion
                prob_matrix[i, j] = prob
                cost[i, j] = motion + 0.05 * raw - MOTION_RELINK_LEARNED_BONUS * prob
        row_ind, col_ind = linear_sum_assignment(cost)
        matches: list[tuple[int, int, float, float, float]] = []
        for r, c in zip(row_ind, col_ind):
            if cost[r, c] >= big:
                continue
            matches.append((
                source_ids[int(r)],
                target_ids[int(c)],
                float(raw_dist[r, c]),
                float(motion_dist[r, c]),
                float(prob_matrix[r, c]),
            ))
        return matches

    times = sorted(ids_by_t)
    for t in times:
        source_ids = ids_by_t.get(t, [])
        target_ids = ids_by_t.get(t + 1, [])
        if not source_ids or not target_ids:
            continue
        unmatched_sources = set(source_ids)
        unmatched_targets = set(target_ids)
        frame_matches: list[tuple[int, int, float, float, str, float]] = []
        for pass_name, gate_um in (("tight", MOTION_RELINK_TIGHT_UM), ("relaxed", MOTION_RELINK_RELAXED_UM)):
            pass_sources = [node_id for node_id in source_ids if node_id in unmatched_sources]
            pass_targets = [node_id for node_id in target_ids if node_id in unmatched_targets]
            matches = assign_pass(pass_sources, pass_targets, gate_um)
            for source_id, target_id, raw, motion, prob in matches:
                if source_id not in unmatched_sources or target_id not in unmatched_targets:
                    continue
                unmatched_sources.remove(source_id)
                unmatched_targets.remove(target_id)
                frame_matches.append((source_id, target_id, raw, motion, pass_name, prob))
                if pass_name == "tight":
                    stats["motion_relink_tight_edges"] += 1
                else:
                    stats["motion_relink_relaxed_edges"] += 1
        for source_id, target_id, raw, motion, pass_name, prob in frame_matches:
            selected_edges.append({
                "source_id": source_id,
                "target_id": target_id,
                "edge_prob": prob,
                "distance_um": raw,
                "motion_distance_um": motion,
                "motion_relinked": 1,
                "motion_pass": pass_name,
            })
            predecessor_position_um[target_id] = position_um[source_id]
        stats["motion_relink_frames"] += 1

    stats["motion_relink_edges"] = len(selected_edges)
    return selected_edges

def close_single_frame_gaps(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
    dataset: str | None = None,
    deepcenter_bundle: dict[str, object] | None = None,
    frame_cache: dict[int, np.ndarray] | None = None,
    deepcenter_cache: dict[tuple[str, int], np.ndarray] | None = None,
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:
    if not OUTPUT_GAP_CLOSE or GAP_CLOSE_MAX_GAP < 1 or not edges:
        return nodes_by_id, edges

    outgoing = {int(edge["source_id"]) for edge in edges}
    incoming = {int(edge["target_id"]) for edge in edges}
    incident = outgoing | incoming

    ends_by_t: dict[int, list[int]] = {}
    starts_by_t: dict[int, list[int]] = {}
    isolated_by_t: dict[int, list[int]] = {}
    all_ids_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        t = int(node["t"])
        all_ids_by_t.setdefault(t, []).append(node_id)
        if node_id not in outgoing:
            ends_by_t.setdefault(t, []).append(node_id)
        if node_id not in incoming:
            starts_by_t.setdefault(t, []).append(node_id)
        if node_id not in incident:
            isolated_by_t.setdefault(t, []).append(node_id)

    max_synthetic = min(
        GAP_CLOSE_MAX_ADDED_ABS,
        max(1, int(round(len(nodes_by_id) * GAP_CLOSE_MAX_ADDED_FRAC))) if GAP_CLOSE_MAX_ADDED_FRAC > 0 else 0,
    )
    next_id = _next_node_id(nodes_by_id)
    frame_cache = frame_cache if frame_cache is not None else {}
    deepcenter_cache = deepcenter_cache if deepcenter_cache is not None else {}
    used_starts: set[int] = set()
    used_isolated: set[int] = set()
    synthetic_added = 0
    new_edges: list[dict[str, object]] = []

    density_cache: dict[int, dict[int, float]] = {}

    def frame_local_spacing(t: int) -> dict[int, float]:
        cached = density_cache.get(t)
        if cached is not None:
            return cached

        frame_ids = all_ids_by_t.get(t, [])
        if len(frame_ids) <= 1:
            result = {
                node_id: GAP_DENSITY_REFERENCE_UM
                for node_id in frame_ids
            }
            density_cache[t] = result
            return result

        positions = np.stack(
            [_position_um(nodes_by_id[node_id]) for node_id in frame_ids]
        )
        tree = cKDTree(positions)
        query_k = min(
            len(frame_ids),
            max(2, GAP_DENSITY_NEIGHBORS + 1),
        )
        distances, _ = tree.query(positions, k=query_k)
        if distances.ndim == 1:
            distances = distances[:, None]

        result: dict[int, float] = {}
        for idx, node_id in enumerate(frame_ids):
            neighbour_distances = distances[idx, 1:]
            neighbour_distances = neighbour_distances[
                np.isfinite(neighbour_distances)
            ]
            spacing = (
                float(np.median(neighbour_distances))
                if neighbour_distances.size
                else GAP_DENSITY_REFERENCE_UM
            )
            result[node_id] = spacing

        density_cache[t] = result
        stats["gap_density_nodes_scored"] += len(result)
        return result

    effective_gap_max = min(GAP_CLOSE_MAX_GAP, 1)
    stats["gap_close_effective_max_gap"] = effective_gap_max
    for gap in range(1, effective_gap_max + 1):
        for t, end_ids in sorted(ends_by_t.items()):
            start_ids = [sid for sid in starts_by_t.get(t + gap + 1, []) if sid not in used_starts]
            if not end_ids or not start_ids:
                continue

            end_points = [node_point(nodes_by_id[eid]) for eid in end_ids]
            start_points = [node_point(nodes_by_id[sid]) for sid in start_ids]
            threshold_um = GAP_CLOSE_UM * (gap + 1)
            d = np.zeros(
                (len(end_ids), len(start_ids)),
                dtype=np.float64,
            )
            adaptive_threshold = np.full_like(d, threshold_um)

            source_spacing = frame_local_spacing(t)
            target_spacing = frame_local_spacing(t + gap + 1)

            for i, ep in enumerate(end_points):
                for j, sp in enumerate(start_points):
                    d[i, j] = point_distance_um(ep, sp)

                    if GAP_DENSITY_ADAPTIVE:
                        local_spacing = 0.5 * (
                            source_spacing.get(
                                end_ids[i],
                                GAP_DENSITY_REFERENCE_UM,
                            )
                            + target_spacing.get(
                                start_ids[j],
                                GAP_DENSITY_REFERENCE_UM,
                            )
                        )
                        step_delta = float(
                            np.clip(
                                GAP_DENSITY_GAIN
                                * (
                                    local_spacing
                                    - GAP_DENSITY_REFERENCE_UM
                                ),
                                -GAP_DENSITY_MAX_STEP_DELTA_UM,
                                GAP_DENSITY_MAX_STEP_DELTA_UM,
                            )
                        )
                        adaptive_threshold[i, j] = (
                            threshold_um + step_delta * (gap + 1)
                        )
                        stats[
                            "gap_density_step_delta_milli_sum"
                        ] += int(round(1000.0 * step_delta))

            base_allowed = d <= threshold_um
            adaptive_allowed = d <= adaptive_threshold

            stats["gap_density_candidates_expanded"] += int(
                (adaptive_allowed & ~base_allowed).sum()
            )
            stats["gap_density_candidates_restricted"] += int(
                (base_allowed & ~adaptive_allowed).sum()
            )
            stats["gap_candidates"] += int(adaptive_allowed.sum())

            if not np.isfinite(d).any():
                continue

            max_threshold = float(np.max(adaptive_threshold))
            big = max_threshold * 1000.0 + 1.0
            cost = np.where(adaptive_allowed, d, big)
            row_ind, col_ind = linear_sum_assignment(cost)

            for r, c in zip(row_ind, col_ind):
                if not adaptive_allowed[r, c]:
                    continue
                if not base_allowed[r, c]:
                    stats[
                        "gap_density_selected_outside_base"
                    ] += 1
                source_id = end_ids[int(r)]
                target_id = start_ids[int(c)]
                if source_id in outgoing or target_id in used_starts:
                    continue

                source = nodes_by_id[source_id]
                target = nodes_by_id[target_id]
                mid_t = int(source["t"]) + gap
                mid_point = (
                    (float(source["z"]) + float(target["z"])) / 2.0,
                    (float(source["y"]) + float(target["y"])) / 2.0,
                    (float(source["x"]) + float(target["x"])) / 2.0,
                )

                middle_id: int | None = None
                middle_reused = False
                if GAP_CLOSE_REUSE_EXISTING:
                    candidates = [nid for nid in isolated_by_t.get(mid_t, []) if nid not in used_isolated]
                    if candidates:
                        distances = [point_distance_um(node_point(nodes_by_id[nid]), mid_point) for nid in candidates]
                        best_idx = int(np.argmin(distances))
                        if distances[best_idx] <= GAP_CLOSE_REUSE_UM:
                            middle_id = candidates[best_idx]
                            middle_reused = True

                if middle_id is None:
                    if synthetic_added >= max_synthetic:
                        stats["gap_skipped_node_cap"] += 1
                        continue
                    middle_id = next_id
                    next_id += 1
                    refined_point = refine_synthetic_midpoint(dataset, mid_t, mid_point, frame_cache, stats)
                    nodes_by_id[middle_id] = {
                        "node_id": middle_id,
                        "t": mid_t,
                        "z": refined_point[0],
                        "y": refined_point[1],
                        "x": refined_point[2],
                        "gap_synthetic": 1,
                    }
                    synthetic_added += 1
                    stats["gap_inserted_synthetic"] += 1

                middle = nodes_by_id[middle_id]
                gap_span_um = float(d[r, c])
                marginal_gap = gap_span_um >= DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM
                synthetic_middle = int(middle.get("gap_synthetic", 0)) == 1
                requires_center_confirmation = (
                    DEEPCENTER_GAP_VETO and marginal_gap and synthetic_middle
                )
                if DEEPCENTER_GAP_VETO and not marginal_gap:
                    stats["deepcenter_gap_bypassed_strong_motion"] += 1
                elif DEEPCENTER_GAP_VETO and not synthetic_middle:
                    stats["deepcenter_gap_bypassed_observed_node"] += 1
                if requires_center_confirmation and not deepcenter_accept_repair_point(
                    dataset,
                    mid_t,
                    node_point(middle),
                    deepcenter_bundle,
                    frame_cache,
                    deepcenter_cache,
                    stats,
                    "gap",
                    DEEPCENTER_GAP_THRESHOLD,
                ):
                    if int(middle.get("gap_synthetic", 0)) == 1:
                        nodes_by_id.pop(middle_id, None)
                        synthetic_added = max(0, synthetic_added - 1)
                        stats["gap_inserted_synthetic"] = max(0, stats["gap_inserted_synthetic"] - 1)
                    continue
                if middle_reused:
                    used_isolated.add(middle_id)
                    stats["gap_reused_existing"] += 1

                e1 = {
                    "source_id": source_id,
                    "target_id": middle_id,
                    "edge_prob": None,
                    "distance_um": edge_distance_um(source, middle),
                    "gap_closed": 1,
                }
                e2 = {
                    "source_id": middle_id,
                    "target_id": target_id,
                    "edge_prob": None,
                    "distance_um": edge_distance_um(middle, target),
                    "gap_closed": 1,
                }
                new_edges.extend([e1, e2])
                outgoing.add(source_id)
                incoming.add(middle_id)
                outgoing.add(middle_id)
                incoming.add(target_id)
                used_starts.add(target_id)
                stats["gap_pairs_selected"] += 1
                stats["gap_added_edges"] += 2

    if new_edges:
        edges = [*edges, *new_edges]
    stats["gap_added_nodes"] = stats["gap_inserted_synthetic"]
    return nodes_by_id, edges


def _single_successor_map(edges: list[dict[str, object]]) -> dict[int, int]:
    by_source: dict[int, list[int]] = {}
    for edge in edges:
        by_source.setdefault(int(edge["source_id"]), []).append(int(edge["target_id"]))
    return {source: targets[0] for source, targets in by_source.items() if len(targets) == 1}


def _single_predecessor_map(edges: list[dict[str, object]]) -> dict[int, int]:
    by_target: dict[int, list[int]] = {}
    for edge in edges:
        by_target.setdefault(int(edge["target_id"]), []).append(int(edge["source_id"]))
    return {target: sources[0] for target, sources in by_target.items() if len(sources) == 1}


def recover_strict_gap2(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
    dataset: str | None = None,
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:
    if not OUTPUT_GAP2_RECOVERY or not edges or not nodes_by_id:
        return nodes_by_id, edges

    outgoing = {int(edge["source_id"]) for edge in edges}
    incoming = {int(edge["target_id"]) for edge in edges}
    predecessor = _single_predecessor_map(edges)
    successor = _single_successor_map(edges)

    ends_by_t: dict[int, list[int]] = {}
    starts_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        t = int(node["t"])
        if node_id not in outgoing:
            ends_by_t.setdefault(t, []).append(node_id)
        if node_id not in incoming:
            starts_by_t.setdefault(t, []).append(node_id)

    cap = min(GAP2_MAX_LINKS_ABS, max(1, int(round(len(edges) * GAP2_MAX_LINKS_FRAC))))
    proposals: list[tuple[float, int, int, int, float]] = []

    def pos_um(node_id: int) -> np.ndarray:
        node = nodes_by_id[node_id]
        return np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64) * np.array(VOXEL_SCALE_UM)

    for t, end_ids in sorted(ends_by_t.items()):
        start_ids = starts_by_t.get(t + 3, [])
        if not end_ids or not start_ids:
            continue
        for end_id in end_ids:
            end_pos = pos_um(end_id)
            for start_id in start_ids:
                start_pos = pos_um(start_id)
                dist = float(np.linalg.norm(start_pos - end_pos))
                if dist > GAP2_MAX_TOTAL_UM or dist / 3.0 > GAP2_MAX_STEP_UM:
                    continue
                step = (start_pos - end_pos) / 3.0
                context_penalty = 0.0
                if GAP2_REQUIRE_CONTEXT:
                    ok_context = False
                    prev_id = predecessor.get(end_id)
                    if prev_id is not None:
                        prev_step = end_pos - pos_um(prev_id)
                        prev_norm = float(np.linalg.norm(prev_step))
                        step_norm = float(np.linalg.norm(step))
                        if prev_norm <= 0.01 or step_norm <= 0.01:
                            ok_context = True
                        else:
                            cos = float(np.dot(prev_step, step) / (prev_norm * step_norm + 1e-9))
                            if cos > -0.25 and np.linalg.norm(prev_step - step) <= 6.0:
                                ok_context = True
                            context_penalty += max(0.0, 0.25 - cos)
                    next_id = successor.get(start_id)
                    if next_id is not None:
                        next_step = pos_um(next_id) - start_pos
                        next_norm = float(np.linalg.norm(next_step))
                        step_norm = float(np.linalg.norm(step))
                        if next_norm <= 0.01 or step_norm <= 0.01:
                            ok_context = True
                        else:
                            cos = float(np.dot(next_step, step) / (next_norm * step_norm + 1e-9))
                            if cos > -0.25 and np.linalg.norm(next_step - step) <= 6.0:
                                ok_context = True
                            context_penalty += max(0.0, 0.25 - cos)
                    if not ok_context:
                        continue
                proposals.append((dist + 2.0 * context_penalty, end_id, start_id, t, dist))

    proposals.sort(key=lambda item: item[0])
    stats["gap2_candidates"] = len(proposals)
    if not proposals:
        return nodes_by_id, edges

    selected: list[tuple[float, int, int, int, float]] = []
    used_ends: set[int] = set()
    used_starts: set[int] = set()
    per_frame_count: dict[int, int] = {}
    for proposal in proposals:
        if len(selected) >= cap:
            stats["gap2_skipped_cap"] += 1
            break
        _, end_id, start_id, t, _ = proposal
        if end_id in used_ends or start_id in used_starts:
            continue
        frame_cap = max(1, int(round(len(ends_by_t.get(t, [])) * GAP2_FRAME_FRAC_CAP)))
        if per_frame_count.get(t, 0) >= frame_cap:
            continue
        selected.append(proposal)
        used_ends.add(end_id)
        used_starts.add(start_id)
        per_frame_count[t] = per_frame_count.get(t, 0) + 1

    if not selected:
        return nodes_by_id, edges

    next_node_id = _next_node_id(nodes_by_id)
    frame_cache: dict[int, np.ndarray] = {}
    new_edges: list[dict[str, object]] = []
    for _, end_id, start_id, t, _ in selected:
        source = nodes_by_id[end_id]
        target = nodes_by_id[start_id]
        previous_id = end_id
        inserted_ids: list[int] = []
        for k in (1, 2):
            frac = k / 3.0
            mid_t = int(source["t"]) + k
            midpoint = (
                float(source["z"]) + (float(target["z"]) - float(source["z"])) * frac,
                float(source["y"]) + (float(target["y"]) - float(source["y"])) * frac,
                float(source["x"]) + (float(target["x"]) - float(source["x"])) * frac,
            )
            refined_point = refine_synthetic_midpoint(dataset, mid_t, midpoint, frame_cache, stats)
            node_id = next_node_id
            next_node_id += 1
            nodes_by_id[node_id] = {
                "node_id": node_id,
                "t": mid_t,
                "z": refined_point[0],
                "y": refined_point[1],
                "x": refined_point[2],
            }
            inserted_ids.append(node_id)
            current = nodes_by_id[node_id]
            new_edges.append({
                "source_id": previous_id,
                "target_id": node_id,
                "edge_prob": None,
                "distance_um": edge_distance_um(nodes_by_id[previous_id], current),
                "gap2_recovered": 1,
            })
            previous_id = node_id
        new_edges.append({
            "source_id": previous_id,
            "target_id": start_id,
            "edge_prob": None,
            "distance_um": edge_distance_um(nodes_by_id[previous_id], target),
            "gap2_recovered": 1,
        })
        stats["gap2_pairs_selected"] += 1
        stats["gap2_added_nodes"] += len(inserted_ids)
        stats["gap2_added_edges"] += 3

    return nodes_by_id, [*edges, *new_edges]


def add_safe_divisions_postlink(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
    dataset: str | None = None,
    deepcenter_bundle: dict[str, object] | None = None,
    frame_cache: dict[int, np.ndarray] | None = None,
    deepcenter_cache: dict[tuple[str, int], np.ndarray] | None = None,
) -> list[dict[str, object]]:
    if not OUTPUT_SAFE_DIVISIONS or not edges or not nodes_by_id:
        return edges
    frame_cache = frame_cache if frame_cache is not None else {}
    deepcenter_cache = deepcenter_cache if deepcenter_cache is not None else {}
 
    out_by_source: dict[int, list[dict[str, object]]] = {}
    incoming: set[int] = set()
    for edge in edges:
        out_by_source.setdefault(int(edge["source_id"]), []).append(edge)
        incoming.add(int(edge["target_id"]))
 
    ids_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        ids_by_t.setdefault(int(node["t"]), []).append(node_id)
 
    existing_edges = {(int(edge["source_id"]), int(edge["target_id"])) for edge in edges}
    global_cap = max(1, int(round(max(1, len(edges)) * SAFE_DIV_GLOBAL_FRAC_CAP)))
    added: list[dict[str, object]] = []
    used_targets: set[int] = set()
    used_sources: set[int] = set()  
 
    for t in sorted(ids_by_t):
        child_frame_ids = ids_by_t.get(t + 1, [])
        if not child_frame_ids:
            continue
        source_ids = [node_id for node_id in ids_by_t[t] if len(out_by_source.get(node_id, [])) == 1]
        candidate_ids = [node_id for node_id in child_frame_ids if node_id not in incoming and node_id not in used_targets]
        if not source_ids or not candidate_ids:
            continue
 
        
        
        
        
        
        candidate_tree = None
        if SAFE_DIV_REQUIRE_MUTUAL_NN:
            candidate_positions = np.stack([_position_um(nodes_by_id[cid]) for cid in candidate_ids])
            candidate_tree = cKDTree(candidate_positions)
 
        frame_cap = max(1, int(round(len(source_ids) * SAFE_DIV_FRAME_FRAC_CAP)))
        proposals: list[tuple[float, int, int, float, float]] = []
        for source_id in source_ids:
            source = nodes_by_id[source_id]
            existing_child_edge = out_by_source[source_id][0]
            existing_child_id = int(existing_child_edge["target_id"])
            existing_child = nodes_by_id.get(existing_child_id)
            if existing_child is None or int(existing_child["t"]) != t + 1:
                continue
            child_dist = edge_distance_um(source, existing_child)
            if child_dist > SAFE_DIV_EXISTING_CHILD_MAX_UM:
                continue
 
            
            
            
            
            mutual_nn_id = None
            if candidate_tree is not None:
                _, nn_idx = candidate_tree.query(_position_um(existing_child))
                mutual_nn_id = candidate_ids[int(nn_idx)]
 
            for candidate_id in candidate_ids:
                if (source_id, candidate_id) in existing_edges:
                    continue
                candidate = nodes_by_id[candidate_id]
                parent_dist = edge_distance_um(source, candidate)
                if parent_dist > SAFE_DIV_MAX_UM:
                    continue
                sister_dist = edge_distance_um(existing_child, candidate)
                if sister_dist > SAFE_DIV_SISTER_MAX_UM:
                    continue
 
                
                if SAFE_DIV_REQUIRE_MUTUAL_NN and candidate_id != mutual_nn_id:
                    stats["safe_division_mutual_nn_rejected"] += 1
                    continue
 
                
                
                
                
                if SAFE_DIV_REQUIRE_DIVERGENCE:
                    c1_succ = out_by_source.get(existing_child_id, [])
                    q_succ = out_by_source.get(candidate_id, [])
                    if len(c1_succ) != 1 or len(q_succ) != 1:
                        stats["safe_division_divergence_rejected"] += 1
                        continue
                    c1_grandchild = nodes_by_id.get(int(c1_succ[0]["target_id"]))
                    q_grandchild = nodes_by_id.get(int(q_succ[0]["target_id"]))
                    if (
                        c1_grandchild is None or q_grandchild is None
                        or int(c1_grandchild["t"]) != t + 2
                        or int(q_grandchild["t"]) != t + 2
                    ):
                        stats["safe_division_divergence_rejected"] += 1
                        continue
                    grandchild_dist = edge_distance_um(c1_grandchild, q_grandchild)
                    if grandchild_dist - sister_dist < SAFE_DIV_DIVERGE_UM:
                        stats["safe_division_divergence_rejected"] += 1
                        continue
 
                stats["safe_division_geometric_candidates"] += 1
                if DEEPCENTER_SAFE_DIV_VETO and not deepcenter_accept_repair_point(
                    dataset,
                    int(candidate["t"]),
                    node_point(candidate),
                    deepcenter_bundle,
                    frame_cache,
                    deepcenter_cache,
                    stats,
                    "safe_div",
                    DEEPCENTER_SAFE_DIV_THRESHOLD,
                ):
                    continue
                
                
                
                
                
                
                if SAFE_DIV_SISTER_SYMMETRY_TAU > 0.0:
                    _sym_denom = max((child_dist + parent_dist) / 2.0, 1e-6)
                    if abs(child_dist - parent_dist) / _sym_denom > SAFE_DIV_SISTER_SYMMETRY_TAU:
                        stats["safe_division_symmetry_rejected"] += 1
                        continue
                score = parent_dist + 0.15 * sister_dist
                proposals.append((score, source_id, candidate_id, parent_dist, sister_dist))
 
        stats["safe_division_candidates"] += len(proposals)
        if not proposals:
            continue
        proposals.sort(key=lambda item: item[0])
        added_this_frame = 0
        for _, source_id, candidate_id, parent_dist, _ in proposals:
            if len(added) >= global_cap:
                stats["safe_division_skipped_cap"] += 1
                break
            if added_this_frame >= frame_cap:
                break
            if candidate_id in used_targets or candidate_id in incoming:
                continue
            if source_id in used_sources:
                continue
            candidate = nodes_by_id[candidate_id]
            added.append({
                "source_id": source_id,
                "target_id": candidate_id,
                "edge_prob": None,
                "distance_um": parent_dist,
                "safe_division": 1,
            })
            used_targets.add(candidate_id)
            used_sources.add(source_id)
            added_this_frame += 1
 
    if added:
        stats["safe_divisions_added"] = len(added)
        return [*edges, *added]
    return edges
_sprint_original_safe_div=add_safe_divisions_postlink
def add_safe_divisions_postlink(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
    dataset: str | None = None,
    deepcenter_bundle: dict[str, object] | None = None,
    frame_cache: dict[int, np.ndarray] | None = None,
    deepcenter_cache: dict[tuple[str, int], np.ndarray] | None = None,
) -> list[dict[str, object]]:
    if not OUTPUT_SAFE_DIVISIONS or not edges or not nodes_by_id:
        return edges
    frame_cache = frame_cache if frame_cache is not None else {}
    deepcenter_cache = deepcenter_cache if deepcenter_cache is not None else {}
 
    out_by_source: dict[int, list[dict[str, object]]] = {}
    incoming: set[int] = set()
    for edge in edges:
        out_by_source.setdefault(int(edge["source_id"]), []).append(edge)
        incoming.add(int(edge["target_id"]))
 
    ids_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        ids_by_t.setdefault(int(node["t"]), []).append(node_id)
 
    existing_edges = {(int(edge["source_id"]), int(edge["target_id"])) for edge in edges}
    global_cap = max(1, int(round(max(1, len(edges)) * SAFE_DIV_GLOBAL_FRAC_CAP)))
    added: list[dict[str, object]] = []
    used_targets: set[int] = set()
    used_sources: set[int] = set()  
 
    for t in sorted(ids_by_t):
        child_frame_ids = ids_by_t.get(t + 1, [])
        if not child_frame_ids:
            continue
        source_ids = [node_id for node_id in ids_by_t[t] if len(out_by_source.get(node_id, [])) == 1]
        candidate_ids = [node_id for node_id in child_frame_ids if node_id not in incoming and node_id not in used_targets]
        if not source_ids or not candidate_ids:
            continue
 
        
        
        
        
        
        candidate_tree = None
        if SAFE_DIV_REQUIRE_MUTUAL_NN:
            candidate_positions = np.stack([_position_um(nodes_by_id[cid]) for cid in candidate_ids])
            candidate_tree = cKDTree(candidate_positions)
 
        frame_cap = max(1, int(round(len(source_ids) * SAFE_DIV_FRAME_FRAC_CAP)))
        proposals: list[tuple[float, int, int, float, float]] = []
        for source_id in source_ids:
            source = nodes_by_id[source_id]
            existing_child_edge = out_by_source[source_id][0]
            existing_child_id = int(existing_child_edge["target_id"])
            existing_child = nodes_by_id.get(existing_child_id)
            if existing_child is None or int(existing_child["t"]) != t + 1:
                continue
            child_dist = edge_distance_um(source, existing_child)
            if child_dist > SAFE_DIV_EXISTING_CHILD_MAX_UM:
                continue
 
            
            
            
            
            mutual_nn_id = None
            if candidate_tree is not None:
                _, nn_idx = candidate_tree.query(_position_um(existing_child))
                mutual_nn_id = candidate_ids[int(nn_idx)]
 
            for candidate_id in candidate_ids:
                if (source_id, candidate_id) in existing_edges:
                    continue
                candidate = nodes_by_id[candidate_id]
                parent_dist = edge_distance_um(source, candidate)
                if parent_dist > SAFE_DIV_MAX_UM:
                    continue
                sister_dist = edge_distance_um(existing_child, candidate)
                if sister_dist > SAFE_DIV_SISTER_MAX_UM:
                    continue
 
                
                if SAFE_DIV_REQUIRE_MUTUAL_NN and candidate_id != mutual_nn_id:
                    stats["safe_division_mutual_nn_rejected"] += 1
                    continue
 
                
                
                
                
                if SAFE_DIV_REQUIRE_DIVERGENCE:
                    c1_succ = out_by_source.get(existing_child_id, [])
                    q_succ = out_by_source.get(candidate_id, [])
                    if len(c1_succ) != 1 or len(q_succ) != 1:
                        stats["safe_division_divergence_rejected"] += 1
                        continue
                    c1_grandchild = nodes_by_id.get(int(c1_succ[0]["target_id"]))
                    q_grandchild = nodes_by_id.get(int(q_succ[0]["target_id"]))
                    if (
                        c1_grandchild is None or q_grandchild is None
                        or int(c1_grandchild["t"]) != t + 2
                        or int(q_grandchild["t"]) != t + 2
                    ):
                        stats["safe_division_divergence_rejected"] += 1
                        continue
                    grandchild_dist = edge_distance_um(c1_grandchild, q_grandchild)
                    if grandchild_dist - sister_dist < SAFE_DIV_DIVERGE_UM:
                        stats["safe_division_divergence_rejected"] += 1
                        continue
 
                stats["safe_division_geometric_candidates"] += 1
                if DEEPCENTER_SAFE_DIV_VETO and not deepcenter_accept_repair_point(
                    dataset,
                    int(candidate["t"]),
                    node_point(candidate),
                    deepcenter_bundle,
                    frame_cache,
                    deepcenter_cache,
                    stats,
                    "safe_div",
                    DEEPCENTER_SAFE_DIV_THRESHOLD,
                ):
                    continue
                
                
                
                
                
                
                if SAFE_DIV_SISTER_SYMMETRY_TAU > 0.0:
                    _sym_denom = max((child_dist + parent_dist) / 2.0, 1e-6)
                    if abs(child_dist - parent_dist) / _sym_denom > SAFE_DIV_SISTER_SYMMETRY_TAU:
                        stats["safe_division_symmetry_rejected"] += 1
                        continue
                score = parent_dist + 0.15 * sister_dist
                proposal = (score, source_id, candidate_id, parent_dist, sister_dist)
                if SPRINT_POLICY.admit((nodes_by_id, out_by_source), proposal, dataset, t, existing_child_id):
                    proposals.append(proposal)
 
        stats["safe_division_candidates"] += len(proposals)
        if not proposals:
            continue
        SPRINT_POLICY.order(proposals, dataset, t)
        added_this_frame = 0
        for _, source_id, candidate_id, parent_dist, _ in proposals:
            if len(added) >= global_cap:
                stats["safe_division_skipped_cap"] += 1
                break
            if added_this_frame >= frame_cap:
                break
            if candidate_id in used_targets or candidate_id in incoming:
                continue
            if source_id in used_sources:
                continue
            candidate = nodes_by_id[candidate_id]
            added.append({
                "source_id": source_id,
                "target_id": candidate_id,
                "edge_prob": None,
                "distance_um": parent_dist,
                "safe_division": 1,
            })
            SPRINT_POLICY.selected(dataset, t, source_id, candidate_id)
            used_targets.add(candidate_id)
            used_sources.add(source_id)
            added_this_frame += 1
 
    if added:
        stats["safe_divisions_added"] = len(added)
        return [*edges, *added]
    return edges
_sprint_patched_safe_div=add_safe_divisions_postlink



def filter_short_track_components(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:
    if not OUTPUT_FILTER_SHORT_TRACKS or OUTPUT_MIN_TRACK_LEN <= 1 or not edges:
        return nodes_by_id, edges

    parent = {node_id: node_id for node_id in nodes_by_id}

    def find(node_id: int) -> int:
        while parent[node_id] != node_id:
            parent[node_id] = parent[parent[node_id]]
            node_id = parent[node_id]
        return node_id

    def union(a: int, b: int) -> None:
        if a not in parent or b not in parent:
            return
        ra = find(a)
        rb = find(b)
        if ra != rb:
            parent[ra] = rb

    out_count: dict[int, int] = {}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        union(source_id, target_id)
        out_count[source_id] = out_count.get(source_id, 0) + 1

    components: dict[int, list[int]] = {}
    for node_id in nodes_by_id:
        components.setdefault(find(node_id), []).append(node_id)

    component_edges: dict[int, list[dict[str, object]]] = {root: [] for root in components}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        if source_id in parent and target_id in parent:
            component_edges.setdefault(find(source_id), []).append(edge)

    keep: set[int] = set()
    for root, members in components.items():
        has_division = any(out_count.get(node_id, 0) >= 2 for node_id in members)
        if len(members) >= OUTPUT_MIN_TRACK_LEN or (OUTPUT_KEEP_DIVISION_COMPONENTS and has_division):
            keep.update(members)

    if not keep:
        stats["short_track_filter_skipped_all"] += 1
        return nodes_by_id, edges

    removed_before_rescue = len(nodes_by_id) - len(keep)
    if removed_before_rescue <= 0:
        return nodes_by_id, edges

    if ADAPTIVE_SHORT_TRACK_RESCUE:
        removed_frac = removed_before_rescue / max(len(nodes_by_id), 1)
        if removed_frac >= SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC:
            budget = min(
                SHORT_TRACK_RESCUE_MAX_NODES_ABS,
                max(0, int(round(len(nodes_by_id) * SHORT_TRACK_RESCUE_MAX_NODES_FRAC))),
            )
            stats["short_track_rescue_triggered"] = 1
            stats["short_track_rescue_budget"] = budget
            proposals: list[tuple[float, int, float, int, list[int]]] = []
            for root, members in components.items():
                if set(members) & keep:
                    continue
                if len(members) < SHORT_TRACK_RESCUE_MIN_LEN or len(members) >= OUTPUT_MIN_TRACK_LEN:
                    continue
                c_edges = component_edges.get(root, [])
                if not c_edges:
                    continue
                probs: list[float] = []
                dists: list[float] = []
                for edge in c_edges:
                    try:
                        prob = float(edge.get("edge_prob", 0.0))
                    except (TypeError, ValueError):
                        prob = 0.0
                    if np.isfinite(prob):
                        probs.append(prob)
                    try:
                        dist = float(edge.get("distance_um", np.nan))
                    except (TypeError, ValueError):
                        dist = np.nan
                    if np.isfinite(dist):
                        dists.append(dist)
                mean_prob = float(np.mean(probs)) if probs else 0.0
                mean_dist = float(np.mean(dists)) if dists else float("inf")
                if mean_prob < SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB:
                    continue
                if mean_dist > SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM:
                    continue
                score = mean_prob - 0.02 * mean_dist + 0.004 * len(members)
                proposals.append((score, len(members), mean_prob, root, members))
            proposals.sort(reverse=True)
            rescued_nodes = 0
            rescued_components = 0
            for _, size, _, _, members in proposals:
                if budget <= 0 or rescued_nodes + size > budget:
                    continue
                keep.update(members)
                rescued_nodes += size
                rescued_components += 1
            stats["short_track_rescue_components"] = rescued_components
            stats["short_track_rescue_nodes"] = rescued_nodes

    removed_nodes = len(nodes_by_id) - len(keep)
    if removed_nodes <= 0:
        return nodes_by_id, edges

    kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in keep}
    kept_edges = [
        edge for edge in edges
        if int(edge["source_id"]) in kept_nodes and int(edge["target_id"]) in kept_nodes
    ]
    stats["short_track_components_removed"] = sum(1 for members in components.values() if not (set(members) & keep))
    stats["short_track_nodes_removed"] = removed_nodes
    stats["short_track_edges_removed"] = len(edges) - len(kept_edges)
    return kept_nodes, kept_edges


def linefit_smooth_output_graph(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
) -> dict[int, dict[str, object]]:
    """Smooth linear track interiors without changing graph topology."""
    if not OUTPUT_LINEFIT_SMOOTH or OUTPUT_LINEFIT_WEIGHT <= 0 or OUTPUT_LINEFIT_WINDOW <= 0 or not edges:
        return nodes_by_id

    predecessor: dict[int, list[int]] = {}
    successor: dict[int, list[int]] = {}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        source = nodes_by_id.get(source_id)
        target = nodes_by_id.get(target_id)
        if source is None or target is None:
            continue
        if int(target["t"]) != int(source["t"]) + 1:
            continue
        successor.setdefault(source_id, []).append(target_id)
        predecessor.setdefault(target_id, []).append(source_id)

    original_pos = {
        node_id: np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64)
        for node_id, node in nodes_by_id.items()
    }
    updated_pos: dict[int, np.ndarray] = {}
    weight = float(np.clip(OUTPUT_LINEFIT_WEIGHT, 0.0, 1.0))

    for node_id in sorted(nodes_by_id):
        neighbourhood: list[tuple[int, int]] = [(0, node_id)]

        current = node_id
        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):
            prev_ids = predecessor.get(current, [])
            if len(prev_ids) != 1:
                break
            current = prev_ids[0]
            if current not in original_pos:
                break
            neighbourhood.append((-step, current))

        current = node_id
        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):
            next_ids = successor.get(current, [])
            if len(next_ids) != 1:
                break
            current = next_ids[0]
            if current not in original_pos:
                break
            neighbourhood.append((step, current))

        if len(neighbourhood) < 3:
            stats["linefit_skipped_nodes"] += 1
            continue

        dts = np.array([delta for delta, _ in neighbourhood], dtype=np.float64)
        coords = np.stack([original_pos[nid] for _, nid in neighbourhood])
        fitted = np.array([np.polyval(np.polyfit(dts, coords[:, axis], 1), 0.0) for axis in range(3)], dtype=np.float64)
        if not np.isfinite(fitted).all():
            stats["linefit_skipped_nodes"] += 1
            continue
        updated_pos[node_id] = (1.0 - weight) * original_pos[node_id] + weight * fitted

    for node_id, pos in updated_pos.items():
        nodes_by_id[node_id]["z"] = float(pos[0])
        nodes_by_id[node_id]["y"] = float(pos[1])
        nodes_by_id[node_id]["x"] = float(pos[2])

    stats["linefit_smoothed_nodes"] = len(updated_pos)
    return nodes_by_id


def filter_output_graph(
    nodes_by_id: dict[int, dict[str, object]],
    raw_edges: list[dict[str, object]],
    dataset: str | None = None,
    deepcenter_bundle: dict[str, object] | None = None,
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]], dict[str, int]]:
    stats = {
        "raw_edges": len(raw_edges),
        "dropped_nonconsecutive_edges": 0,
        "dropped_long_edges": 0,
        "dropped_multi_parent_edges": 0,
        "dropped_multi_child_edges": 0,
        "dropped_division_edges": 0,
        "gap_candidates": 0,
        "gap_pairs_selected": 0,
        "gap_reused_existing": 0,
        "gap_inserted_synthetic": 0,
        "gap_added_nodes": 0,
        "gap_added_edges": 0,
        "gap_skipped_node_cap": 0,
        "gap_density_nodes_scored": 0,
        "gap_density_candidates_expanded": 0,
        "gap_density_candidates_restricted": 0,
        "gap_density_selected_outside_base": 0,
        "gap_density_step_delta_milli_sum": 0,
        "gap_refined_synthetic": 0,
        "gap_refine_failed": 0,
        "gap_refine_rejected_shift": 0,
        "pruned_isolated_nodes": 0,
        "motion_relink_edges": 0,
        "motion_relink_tight_edges": 0,
        "motion_relink_relaxed_edges": 0,
        "motion_relink_frames": 0,
        "motion_relink_replaced_raw_edges": 0,
        "motion_relink_fallback_raw": 0,
        "motion_relink_skipped_large_frame": 0,
        "gap2_candidates": 0,
        "gap2_pairs_selected": 0,
        "gap2_added_nodes": 0,
        "gap2_added_edges": 0,
        "gap2_skipped_cap": 0,
        "safe_division_candidates": 0,
        "safe_division_geometric_candidates": 0,  
        "safe_divisions_added": 0,
        "safe_division_skipped_cap": 0,
        "safe_division_mutual_nn_rejected": 0,
        "safe_division_divergence_rejected": 0,
        "safe_division_symmetry_rejected": 0,  
        "deepcenter_gap_checked": 0,
        "deepcenter_gap_bypassed_strong_motion": 0,
        "deepcenter_gap_bypassed_observed_node": 0,
        "deepcenter_gap_accepted": 0,
        "deepcenter_gap_rejected": 0,
        "deepcenter_gap_missing": 0,
        "deepcenter_safe_div_checked": 0,
        "deepcenter_safe_div_accepted": 0,
        "deepcenter_safe_div_rejected": 0,
        "deepcenter_safe_div_missing": 0,
        "short_track_components_removed": 0,
        "short_track_nodes_removed": 0,
        "short_track_edges_removed": 0,
        "short_track_filter_skipped_all": 0,
        "short_track_rescue_triggered": 0,
        "short_track_rescue_components": 0,
        "short_track_rescue_nodes": 0,
        "short_track_rescue_budget": 0,
        "linefit_smoothed_nodes": 0,
        "linefit_skipped_nodes": 0,
    }

    edges: list[dict[str, object]] = []
    for edge in raw_edges:
        source = nodes_by_id.get(int(edge["source_id"]))
        target = nodes_by_id.get(int(edge["target_id"]))
        if source is None or target is None:
            continue
        if OUTPUT_ENFORCE_NEXT_FRAME and int(target["t"]) != int(source["t"]) + 1:
            stats["dropped_nonconsecutive_edges"] += 1
            continue
        distance_um = edge_distance_um(source, target)
        edge["distance_um"] = distance_um
        if OUTPUT_EDGE_MAX_UM > 0 and distance_um > OUTPUT_EDGE_MAX_UM:
            stats["dropped_long_edges"] += 1
            continue
        edges.append(edge)

    if OUTPUT_MOTION_RELINK:
        learned_edge_probs: dict[tuple[int, int], float] = {}
        for edge in edges:
            prob = edge.get("edge_prob")
            if prob is None:
                continue
            try:
                prob = float(prob)
            except (TypeError, ValueError):
                continue
            if np.isfinite(prob):
                key = (int(edge["source_id"]), int(edge["target_id"]))
                learned_edge_probs[key] = max(learned_edge_probs.get(key, float("-inf")), prob)
        motion_edges = motion_relink_edges(nodes_by_id, stats, learned_edge_probs)
        if motion_edges:
            stats["motion_relink_replaced_raw_edges"] = len(edges)
            edges = motion_edges
        else:
            stats["motion_relink_fallback_raw"] = 1

    if OUTPUT_SINGLE_PARENT_REPAIR and edges:
        best_by_target: dict[int, dict[str, object]] = {}
        for edge in edges:
            target_id = int(edge["target_id"])
            prev = best_by_target.get(target_id)
            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):
                best_by_target[target_id] = edge
        kept_ids = {id(edge) for edge in best_by_target.values()}
        stats["dropped_multi_parent_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)
        edges = [edge for edge in edges if id(edge) in kept_ids]

    if OUTPUT_SINGLE_CHILD_REPAIR and edges:
        best_by_source: dict[int, dict[str, object]] = {}
        for edge in edges:
            source_id = int(edge["source_id"])
            prev = best_by_source.get(source_id)
            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):
                best_by_source[source_id] = edge
        kept_ids = {id(edge) for edge in best_by_source.values()}
        stats["dropped_multi_child_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)
        edges = [edge for edge in edges if id(edge) in kept_ids]

    print(f"  [{dataset}] after edge-filter+motion-relink: {len(nodes_by_id)} nodes, {len(edges)} edges")
    repair_frame_cache: dict[int, np.ndarray] = {}
    deepcenter_heatmap_cache: dict[tuple[str, int], np.ndarray] = {}
    nodes_by_id, edges = close_single_frame_gaps(
        nodes_by_id,
        edges,
        stats,
        dataset=dataset,
        deepcenter_bundle=deepcenter_bundle,
        frame_cache=repair_frame_cache,
        deepcenter_cache=deepcenter_heatmap_cache,
    )
    nodes_by_id, edges = recover_strict_gap2(nodes_by_id, edges, stats, dataset=dataset)
    print(f"  [{dataset}] after gap-closing (single-frame + gap2): {len(nodes_by_id)} nodes, {len(edges)} edges")
    edges = add_safe_divisions_postlink(
        nodes_by_id,
        edges,
        stats,
        dataset=dataset,
        deepcenter_bundle=deepcenter_bundle,
        frame_cache=repair_frame_cache,
        deepcenter_cache=deepcenter_heatmap_cache,
    )

    _geo_cands = stats['safe_division_geometric_candidates']
    _post_veto_cands = stats['safe_division_candidates']
    _rejected_by_dc = _geo_cands - _post_veto_cands
    print(
        f"  [{dataset}] after safe-division repair: {len(nodes_by_id)} nodes, {len(edges)} edges"
        f" (geometric_candidates={_geo_cands}, deepcenter_rejected={_rejected_by_dc},"
        f" post_veto_candidates={_post_veto_cands}, added={stats['safe_divisions_added']},"
        f" cap_skipped={stats['safe_division_skipped_cap']},"
        f" mutual_nn_rejected={stats['safe_division_mutual_nn_rejected']},"
        f" divergence_rejected={stats['safe_division_divergence_rejected']})"
    )
    if OUTPUT_DIVISION_GEOMETRY_FILTER and edges:
        by_source: dict[int, list[dict[str, object]]] = {}
        for edge in edges:
            by_source.setdefault(int(edge["source_id"]), []).append(edge)

        filtered: list[dict[str, object]] = []
        for source_id, source_edges in by_source.items():
            if len(source_edges) <= 1:
                filtered.extend(source_edges)
                continue

            ranked = sorted(source_edges, key=edge_sort_key, reverse=True)
            source = nodes_by_id[source_id]
            top1 = ranked[0]
            top2 = ranked[1]
            d1 = float(top1["distance_um"])
            d2 = float(top2["distance_um"])
            sister = edge_distance_um(nodes_by_id[int(top1["target_id"])], nodes_by_id[int(top2["target_id"])])
            valid_division = (
                max(d1, d2) <= DIV_PARENT_MAX_UM
                and sister <= DIV_SISTER_MAX_UM
                and int(nodes_by_id[int(top1["target_id"])] ["t"]) == int(source["t"]) + 1
                and int(nodes_by_id[int(top2["target_id"])] ["t"]) == int(source["t"]) + 1
            )
            if valid_division:
                filtered.extend([top1, top2])
                stats["dropped_division_edges"] += max(0, len(ranked) - 2)
            elif DIV_DROP_TO_SINGLE_IF_BAD:
                filtered.append(top1)
                stats["dropped_division_edges"] += len(ranked) - 1
            else:
                filtered.extend(ranked)
        edges = filtered

    if OUTPUT_PRUNE_ISOLATED:
        incident = {int(edge["source_id"]) for edge in edges} | {int(edge["target_id"]) for edge in edges}
        if incident:
            kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in incident}
            stats["pruned_isolated_nodes"] = len(nodes_by_id) - len(kept_nodes)
            nodes_by_id = kept_nodes
            edges = [edge for edge in edges if int(edge["source_id"]) in nodes_by_id and int(edge["target_id"]) in nodes_by_id]

    print(f"  [{dataset}] after division-geometry-filter+prune-isolated: {len(nodes_by_id)} nodes, {len(edges)} edges")
    nodes_by_id, edges = filter_short_track_components(nodes_by_id, edges, stats)
    print(f"  [{dataset}] after short-track filtering: {len(nodes_by_id)} nodes, {len(edges)} edges"
          f" (components_removed={stats['short_track_components_removed']})")
    nodes_by_id = linefit_smooth_output_graph(nodes_by_id, edges, stats)
    print(f"  [{dataset}] FINAL: {len(nodes_by_id)} nodes, {len(edges)} edges")

    return nodes_by_id, edges, stats


DEEPCENTER_VETO_DETECTOR = load_deepcenter_veto_detector()

def write_test_submission(tag: str = "base") -> None:
    
    
    geffs = sorted((REPO_DIR / "predictions").glob(f"*/{METHOD}/split_0/*.geff"))
    print(f"Found {len(geffs)} prediction graphs")
    if len(geffs) != len(test_stems):
        found = {path.stem for path in geffs}
        missing = sorted(set(test_stems) - found)
        raise RuntimeError(f"Expected {len(test_stems)} graphs, found {len(geffs)}. Missing: {missing[:10]}")

    stats_rows: list[dict[str, object]] = []
    seen_datasets: set[str] = set()
    row_id = 0
    total_nodes = 0
    total_edges = 0

    with SUBMISSION_PATH.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        writer.writeheader()

        for geff_path in geffs:
            dataset = geff_path.stem
            seen_datasets.add(dataset)
            graph = graph_from_geff(geff_path)

            nodes_by_id: dict[int, dict[str, object]] = {}
            for row in graph.node_attrs().iter_rows(named=True):
                node_id = int(row["node_id"])
                nodes_by_id[node_id] = {
                    "node_id": node_id,
                    "t": int(row["t"]),
                    "z": float(row["z"]),
                    "y": float(row["y"]),
                    "x": float(row["x"]),
                }

            raw_edges: list[dict[str, object]] = []
            for row in graph.edge_attrs().iter_rows(named=True):
                edge_prob = row.get("edge_prob") if hasattr(row, "get") else None
                raw_edges.append({
                    "source_id": int(row["source_id"]),
                    "target_id": int(row["target_id"]),
                    "edge_prob": None if edge_prob is None else float(edge_prob),
                })

            raw_node_count = len(nodes_by_id)
            nodes_by_id, edges, filter_stats = filter_output_graph(nodes_by_id, raw_edges, dataset=dataset, deepcenter_bundle=DEEPCENTER_VETO_DETECTOR)
            if not nodes_by_id:
                raise AssertionError(f"{dataset}: post-processing removed every node")

            for node_id in sorted(nodes_by_id):
                node = nodes_by_id[node_id]
                writer.writerow({
                    "id": row_id,
                    "dataset": dataset,
                    "row_type": "node",
                    "node_id": int(node["node_id"]),
                    "t": int(node["t"]),
                    "z": max(0, int(round(float(node["z"])))),
                    "y": max(0, int(round(float(node["y"])))),
                    "x": max(0, int(round(float(node["x"])))),
                    "source_id": -1,
                    "target_id": -1,
                })
                row_id += 1

            division_sources: dict[int, int] = {}
            for edge in edges:
                source_id = int(edge["source_id"])
                target_id = int(edge["target_id"])
                if source_id not in nodes_by_id or target_id not in nodes_by_id:
                    raise AssertionError(f"{dataset}: dangling edge after filtering")
                writer.writerow({
                    "id": row_id,
                    "dataset": dataset,
                    "row_type": "edge",
                    "node_id": -1,
                    "t": -1,
                    "z": -1,
                    "y": -1,
                    "x": -1,
                    "source_id": source_id,
                    "target_id": target_id,
                })
                row_id += 1
                division_sources[source_id] = division_sources.get(source_id, 0) + 1

            node_count = len(nodes_by_id)
            edge_count = len(edges)
            total_nodes += node_count
            total_edges += edge_count
            stats_rows.append({
                "dataset": dataset,
                "raw_nodes": raw_node_count,
                "nodes": node_count,
                "raw_edges": filter_stats["raw_edges"],
                "edges": edge_count,
                "division_like_sources": sum(1 for count in division_sources.values() if count >= 2),
                "edge_to_node_ratio": edge_count / max(node_count, 1),
                "gap_added_nodes_frac": filter_stats.get("gap_added_nodes", 0) / max(raw_node_count, 1),
                **filter_stats,
            })

    expected_datasets = set(test_stems)
    missing_datasets = sorted(expected_datasets - seen_datasets)
    extra_datasets = sorted(seen_datasets - expected_datasets)
    if missing_datasets or extra_datasets:
        raise AssertionError({"missing": missing_datasets[:10], "extra": extra_datasets[:10]})
    assert row_id == total_nodes + total_edges, "Internal row counter mismatch"
    assert total_nodes > 0, "No node rows produced"

    header = SUBMISSION_PATH.open().readline().strip().split(",")
    assert header == CSV_COLUMNS, f"Bad CSV header: {header}"

    stats = pd.DataFrame(stats_rows).sort_values("dataset").reset_index(drop=True)
    stats["predict_minutes_total"] = predict_seconds / 60.0
    stats["experiment_tag"] = f"{EXPERIMENT_TAG}:{tag}"
    stats.to_csv(RUN_STATS_PATH, index=False)

    print(f"Wrote {SUBMISSION_PATH} with {row_id:,} rows")
    print(f"Node rows: {total_nodes:,} | edge rows: {total_edges:,}")
    print(f"Wrote {RUN_STATS_PATH}")
    display(pd.read_csv(SUBMISSION_PATH, nrows=8))


import numpy as np
from scipy.special import expit
SCALE = np.array([1.625,0.40625,0.40625])

def point(row):
    return np.array([row["z"], row["y"], row["x"]], dtype=float) * SCALE

def pair_features(coords):
    """Physical-coordinate input [..., parent/d1/d2/g1/g2, z/y/x]."""
    a = np.asarray(coords, dtype=float)
    if a.shape[-2:] != (5, 3) or not np.isfinite(a).all():
        raise ValueError("INVALID_PAIR_COORDINATES")
    p, c1, c2, g1, g2 = np.moveaxis(a, -2, 0)
    v1, v2 = c1-p, c2-p
    norm = lambda v: np.linalg.norm(v, axis=-1)
    d1, d2 = norm(v1), norm(v2)
    sister, grand = norm(c1-c2), norm(g1-g2)
    speed1, speed2 = norm(g1-c1), norm(g2-c2)
    return np.stack([np.minimum(d1,d2), np.maximum(d1,d2), sister,
                     norm((c1+c2)*0.5-p), np.abs(d1-d2)/np.maximum((d1+d2)*0.5,1e-6),
                     np.sum(v1*v2,axis=-1)/np.maximum(d1*d2,1e-6), grand, grand-sister,
                     (speed1+speed2)*0.5, np.abs(speed1-speed2)], axis=-1)

def context(nodes, out, parent, c1, c2):
    t = int(nodes[parent]["t"])
    if c1 == c2 or any(int(nodes[c]["t"]) != t+1 for c in (c1,c2)):
        return None
    if any(len(out.get(c,[])) != 1 for c in (c1,c2)):
        return None
    g1,g2 = out[c1][0],out[c2][0]
    if g1 == g2 or any(int(nodes[g]["t"]) != t+2 for g in (g1,g2)):
        return None
    return np.stack([point(nodes[i]) for i in (parent,c1,c2,g1,g2)])

def design(x, mean, scale):
    z = np.clip((np.asarray(x)-mean)/scale,-8.0,8.0)
    return np.concatenate([np.ones((*z.shape[:-1],1)), z, z*z],axis=-1)

def predict(model, features):
    return expit(design(features,np.asarray(model["mean"]),np.asarray(model["scale"]))@np.asarray(model["coefficients"]))

"""Minimal additions to the original Forge proposal acceptance and ordering.
No fit, model update, or new eligibility rule. Explicit model selection.
"""
import math

class ProposalPolicy:
    def __init__(self, arm, score_pair):
        if arm not in ('A0','G1','R1'): raise ValueError(arm)
        self.arm=arm; self.score_pair=score_pair; self.rows=[]; self.frame_fallbacks=[]
        self.scores={}
    def admit(self, snapshot, proposal, dataset, t, existing_child):
        distance,parent,child,*_=proposal
        score=None if self.arm=='A0' else self.score_pair(snapshot,parent,existing_child,child,dataset)
        if score is not None and not math.isfinite(score): raise RuntimeError('NONFINITE_LEARNED_SCORE')
        accept=not(self.arm=='G1' and score is not None and score<0.95)
        row={'dataset':dataset,'frame':t,'parent':parent,'child1':existing_child,'child2':child,
             'arm':self.arm,'original_eligible':True,'learned_score':score,'abstain':score is None and self.arm!='A0',
             'accepted_before_sort':accept,'distance_priority':distance,'selected':False}
        self.rows.append(row);self.scores[(dataset,t,parent,child)]=score
        return accept
    def order(self, proposals, dataset, t):
        before=[(x[1],x[2]) for x in proposals]
        if self.arm=='R1':
            scores=[self.scores[(dataset,t,p[1],p[2])] for p in proposals]
            if any(v is None for v in scores):
                self.frame_fallbacks.append((dataset,t)); proposals.sort(key=lambda p:p[0])
            else:
                proposals.sort(key=lambda p:(-self.scores[(dataset,t,p[1],p[2])],p[0]))
        else:proposals.sort(key=lambda p:p[0])
        assert set(before)=={(x[1],x[2]) for x in proposals}
        for rank,p in enumerate(proposals):
            row=next(r for r in reversed(self.rows) if (r['dataset'],r['frame'],r['parent'],r['child2'])==(dataset,t,p[1],p[2]))
            row['original_order']=before.index((p[1],p[2]));row['rank']=rank
    def selected(self,dataset,t,parent,child):
        row=next(r for r in reversed(self.rows) if (r['dataset'],r['frame'],r['parent'],r['child2'])==(dataset,t,parent,child));row['selected']=True

def patch_safe_div(source):
    """Patch one audited pure function; caller never passes a training cell."""
    append='                proposals.append((score, source_id, candidate_id, parent_dist, sister_dist))'
    assert source.count(append)==1
    source=source.replace(append,'''                proposal = (score, source_id, candidate_id, parent_dist, sister_dist)
                if SPRINT_POLICY.admit((nodes_by_id, out_by_source), proposal, dataset, t, existing_child_id):
                    proposals.append(proposal)''')
    old='        proposals.sort(key=lambda item: item[0])';assert source.count(old)==1
    source=source.replace(old,'        SPRINT_POLICY.order(proposals, dataset, t)')
    old='            used_targets.add(candidate_id)';assert source.count(old)==1
    return source.replace(old,'            SPRINT_POLICY.selected(dataset, t, source_id, candidate_id)\n'+old)

SPRINT_ARM='G1'
VALIDATION_EMBRYO_MAP={'44b6_12dfb391': '44b6', '44b6_267148e4': '44b6', '44b6_2a2eff9f': '44b6', '44b6_341df25f': '44b6', '6bba_062c8d37': '6bba', '6bba_07e24132': '6bba', '6bba_085bf656': '6bba', '6bba_09961292': '6bba'}
SPRINT_PP_KEYS=['SAFE_DIV_MAX_UM', 'SAFE_DIV_SISTER_MAX_UM', 'SAFE_DIV_DIVERGE_UM', 'SAFE_DIV_SISTER_SYMMETRY_TAU', 'SAFE_DIV_EXISTING_CHILD_MAX_UM', 'SAFE_DIV_FRAME_FRAC_CAP', 'SAFE_DIV_GLOBAL_FRAC_CAP', 'DEEPCENTER_SAFE_DIV_THRESHOLD', 'DEEPCENTER_GAP_THRESHOLD', 'GAP_CLOSE_UM', 'OUTPUT_MIN_TRACK_LEN', 'SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB', 'MOTION_RELINK_TIGHT_UM', 'MOTION_RELINK_RELAXED_UM', 'GAP2_MAX_STEP_UM', 'GAP2_MAX_TOTAL_UM', 'MOTION_RELINK_LEARNED_BONUS', 'MOTION_RELINK_VELOCITY_WEIGHT', 'GAP_CLOSE_REUSE_UM', 'OUTPUT_EDGE_MAX_UM']
"""Frozen-model inference and ordinary runtime evidence; inserted into Forge cell 5.
SPRINT_ARM, VALIDATION_EMBRYO_MAP and original/patched functions supplied by builder.
"""
import copy as _sprint_copy
import hashlib as _sprint_hashlib
import json as _sprint_json
_SPRINT_WEIGHT_SHA='0a1f9b93bb529e70f4f7c2ba0907eea8b4cecd2befccc8ba1fb75e569edf77a0'
_sprint_paths=[p for p in Path('/kaggle/input').rglob('division_gate_weights.json') if _sprint_hashlib.sha256(p.read_bytes()).hexdigest()==_SPRINT_WEIGHT_SHA]
assert len(_sprint_paths)==1,'SPRINT_EXACT_WEIGHT_REQUIRED'
_SPRINT_WEIGHTS=_sprint_json.loads(_sprint_paths[0].read_text())
assert set(_SPRINT_WEIGHTS['held_out'])=={'44b6','6bba'}
_SPRINT_TRAIN_RECEIPT=_sprint_json.loads((_sprint_paths[0].parent/'division_training_receipt.json').read_text())
assert _SPRINT_TRAIN_RECEIPT['weights_sha256']==_SPRINT_WEIGHT_SHA
for _fold in _SPRINT_TRAIN_RECEIPT['folds']:assert _fold['held_out_embryo'] not in _fold['training_embryos']
SPRINT_CALLS=[]
def _sprint_model_key(dataset):
 # Original validator switches the actual source directory; embryo comes only from the explicit frozen map.
 if Path(TEST_DIR).resolve()==(Path(COMP_DIR)/'train').resolve():
  assert dataset in VALIDATION_EMBRYO_MAP,'UNMAPPED_VALIDATION_EMBRYO'
  return VALIDATION_EMBRYO_MAP[dataset]
 return 'final'
def _sprint_score(snapshot,parent,c1,c2,dataset):
 nodes,out_edges=snapshot;out={u:[int(e['target_id']) for e in es] for u,es in out_edges.items() if u in (c1,c2)}
 coords=context(nodes,out,parent,c1,c2)
 if coords is None:return None
 key=_sprint_model_key(dataset);model=_SPRINT_WEIGHTS['final'] if key=='final' else _SPRINT_WEIGHTS['held_out'][key]
 return float(predict(model,pair_features(coords)))
def _sprint_graph_hash(nodes,edges):
 return _sprint_hashlib.sha256(_sprint_json.dumps([list(nodes.items()),edges],allow_nan=False,separators=(',',':')).encode()).hexdigest()
def sprint_audited_safe_div(nodes,edges,stats,**kw):
 global SPRINT_POLICY
 dataset=kw['dataset'];SPRINT_POLICY=ProposalPolicy(SPRINT_ARM,_sprint_score)
 before=_sprint_graph_hash(nodes,edges)
 shadow_nodes=_sprint_copy.deepcopy(nodes);shadow_edges=_sprint_copy.deepcopy(edges);shadow_stats=dict(stats)
 result=_sprint_patched_safe_div(nodes,edges,stats,**kw)
 # Same snapshot original rule replay is evidence only; it never supplies output.
 old=_sprint_original_safe_div(shadow_nodes,shadow_edges,shadow_stats,**kw)
 assert _sprint_graph_hash(shadow_nodes,shadow_edges)==before,'SPRINT_SHADOW_INPUT_MUTATED'
 old_es={(int(e['source_id']),int(e['target_id'])) for e in old};new_es={(int(e['source_id']),int(e['target_id'])) for e in result}
 record={'dataset':dataset,'arm':SPRINT_ARM,'model':_sprint_model_key(dataset),'input_hash':before,'old_safe_hash':_sprint_graph_hash(shadow_nodes,old),'new_safe_hash':_sprint_graph_hash(nodes,result),'added_edges':sorted(new_es-old_es),'lost_edges':sorted(old_es-new_es),'classifier_calls':sum(r['learned_score'] is not None for r in SPRINT_POLICY.rows),'filtered':sum(not r['accepted_before_sort'] for r in SPRINT_POLICY.rows),'abstain':sum(r['abstain'] for r in SPRINT_POLICY.rows),'fallback_frames':SPRINT_POLICY.frame_fallbacks,'candidates':SPRINT_POLICY.rows,'resolved_config':{k:globals()[k] for k in SPRINT_PP_KEYS}}
 SPRINT_CALLS.append(record)
 with (WORKING_DIR/'sprint_production_calls.jsonl').open('a') as f:f.write(_sprint_json.dumps(record,allow_nan=False)+'\n')
 print('SPRINT_PRODUCTION_SAFE_DIV',dataset,record['model'],record['classifier_calls'],len(record['added_edges']),len(record['lost_edges']),flush=True)
 return result
add_safe_divisions_postlink=sprint_audited_safe_div



import types as _f1_types
_f1_module=_f1_types.ModuleType('flow_patch')
exec('"""F1: one-pass neighbour displacement prior, no labels or model fitting."""\nimport numpy as np\n\nCONFIG = dict(k=12, radius_um=40.0, exclude_um=1.5, min_global_seeds=4, iterations=1)\n\ndef neighbour_predictions(nodes, seed_edges, position_um):\n    """Seeds are tight edges from an independent unmodified G1 motion call."""\n    by_t = {}\n    for edge in seed_edges:\n        if edge[\'motion_pass\'] != \'tight\':\n            continue\n        s, d = int(edge[\'source_id\']), int(edge[\'target_id\'])\n        assert s in nodes and d in nodes, \'F1_SEED_ENDPOINT\'\n        t = int(nodes[s][\'t\'])\n        assert int(nodes[d][\'t\']) == t + 1, \'F1_SEED_TIME\'\n        pos = position_um(nodes[s])\n        delta = position_um(nodes[d]) - pos\n        assert np.isfinite(pos).all() and np.isfinite(delta).all(), \'F1_NONFINITE_SEED\'\n        by_t.setdefault(t, []).append((s, pos, delta))\n    times = {int(n[\'t\']) for n in nodes.values()}\n    predictions, rows = {}, []\n    for t in sorted(times):\n        if t+1 not in times:\n            continue\n        seeds = by_t.get(t, [])\n        ids = sorted(i for i,n in nodes.items() if int(n[\'t\']) == t)\n        counts, covered = {}, 0\n        for node_id in ids:\n            pos = position_um(nodes[node_id])\n            assert np.isfinite(pos).all(), \'F1_NONFINITE_POSITION\'\n            neighbours = []\n            if len(seeds) >= CONFIG[\'min_global_seeds\']:\n                for seed_id, seed_pos, delta in seeds:\n                    if seed_id == node_id:\n                        continue\n                    distance = float(np.linalg.norm(seed_pos-pos))\n                    if CONFIG[\'exclude_um\'] < distance <= CONFIG[\'radius_um\']:\n                        neighbours.append((distance, seed_id, delta))\n                neighbours.sort(key=lambda r:(r[0],r[1]))\n                neighbours = neighbours[:CONFIG[\'k\']]\n            counts[len(neighbours)] = counts.get(len(neighbours),0)+1\n            if neighbours:\n                predicted = pos + np.median(np.stack([r[2] for r in neighbours]), axis=0)\n                assert np.isfinite(predicted).all(), \'F1_NONFINITE_FLOW\'\n                predictions[node_id] = predicted\n                covered += 1\n        rows.append(dict(t=t,seeds=len(seeds),source_nodes=len(ids),flow_nodes=covered,\n                         global_seed_fallback=len(ids) if len(seeds)<4 else 0,\n                         no_local_fallback=len(ids)-covered if len(seeds)>=4 else 0,\n                         neighbour_counts=counts))\n    return predictions, rows\n\ndef patch_motion_source(original):\n    """Change only predicted position; preserve gates, costs and assignments."""\n    anchor=\'            for j, target_id in enumerate(target_ids):\'\n    assert original.count(anchor)==1\n    return original.replace(\'def motion_relink_edges(\', \'def _f1_motion_impl(\',1).replace(\n        anchor, \'            if source_id in F1_PREDICTIONS:\\n                predicted = F1_PREDICTIONS[source_id]\\n\'+anchor,1)\n\ndef install(scope, original_source):\n    original = scope[\'motion_relink_edges\']\n    exec(patch_motion_source(original_source), scope)\n    impl = scope[\'_f1_motion_impl\']\n    scope[\'F1_ENABLED\'] = True\n    scope[\'F1_CALLS\'] = []\n    def motion(nodes, stats, learned_edge_probs=None):\n        import collections, time\n        if not scope[\'F1_ENABLED\']:\n            return original(nodes, stats, learned_edge_probs)\n        start = time.monotonic()\n        seed_stats = collections.defaultdict(int)\n        seeds = original(nodes, seed_stats, learned_edge_probs)\n        pred, rows = neighbour_predictions(nodes, seeds, scope[\'_position_um\'])\n        scope[\'F1_PREDICTIONS\'] = pred\n        result = impl(nodes, stats, learned_edge_probs)\n        scope[\'F1_CALLS\'].append(dict(frames=rows,seconds=time.monotonic()-start,\n                                     seed_stats=dict(seed_stats),prediction_count=len(pred)))\n        return result\n    scope[\'motion_relink_edges\'] = motion\n    return original\n',_f1_module.__dict__)
F1_CONFIG=_f1_module.CONFIG
F1_ORIGINAL_MOTION=_f1_module.install(globals(),'def motion_relink_edges(\n    nodes_by_id: dict[int, dict[str, object]],\n    stats: dict[str, int],\n    learned_edge_probs: dict[tuple[int, int], float] | None = None,\n) -> list[dict[str, object]]:\n    if not OUTPUT_MOTION_RELINK or not nodes_by_id:\n        return []\n\n    learned_edge_probs = learned_edge_probs or {}\n\n    def learned_prob(source_id: int, target_id: int) -> float:\n        value = learned_edge_probs.get((source_id, target_id), 0.0)\n        try:\n            value = float(value)\n        except (TypeError, ValueError):\n            return 0.0\n        if not np.isfinite(value):\n            return 0.0\n        if value < 0.0 or value > 1.0:\n            value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))\n        return float(np.clip(value, 0.0, 1.0))\n\n    ids_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        ids_by_t.setdefault(int(node["t"]), []).append(node_id)\n    for ids in ids_by_t.values():\n        ids.sort()\n\n    frame_sizes = [len(ids) for ids in ids_by_t.values()]\n    if frame_sizes and max(frame_sizes) > MOTION_RELINK_MAX_FRAME_NODES:\n        stats["motion_relink_skipped_large_frame"] = 1\n        return []\n\n    position_um = {node_id: _position_um(node) for node_id, node in nodes_by_id.items()}\n    predecessor_position_um: dict[int, np.ndarray] = {}\n    selected_edges: list[dict[str, object]] = []\n\n    def assign_pass(\n        source_ids: list[int],\n        target_ids: list[int],\n        gate_um: float,\n    ) -> list[tuple[int, int, float, float, float]]:\n        if not source_ids or not target_ids:\n            return []\n        big = gate_um * 1000.0 + 1.0\n        cost = np.full((len(source_ids), len(target_ids)), big, dtype=np.float64)\n        raw_dist = np.full_like(cost, np.inf)\n        motion_dist = np.full_like(cost, np.inf)\n        prob_matrix = np.zeros_like(cost)\n        for i, source_id in enumerate(source_ids):\n            source_pos = position_um[source_id]\n            prev_pos = predecessor_position_um.get(source_id)\n            if prev_pos is None:\n                predicted = source_pos\n            else:\n                predicted = source_pos + MOTION_RELINK_VELOCITY_WEIGHT * (source_pos - prev_pos)\n            for j, target_id in enumerate(target_ids):\n                target_pos = position_um[target_id]\n                raw = float(np.linalg.norm(target_pos - source_pos))\n                if raw > gate_um:\n                    continue\n                motion = float(np.linalg.norm(target_pos - predicted))\n                prob = learned_prob(source_id, target_id)\n                raw_dist[i, j] = raw\n                motion_dist[i, j] = motion\n                prob_matrix[i, j] = prob\n                cost[i, j] = motion + 0.05 * raw - MOTION_RELINK_LEARNED_BONUS * prob\n        row_ind, col_ind = linear_sum_assignment(cost)\n        matches: list[tuple[int, int, float, float, float]] = []\n        for r, c in zip(row_ind, col_ind):\n            if cost[r, c] >= big:\n                continue\n            matches.append((\n                source_ids[int(r)],\n                target_ids[int(c)],\n                float(raw_dist[r, c]),\n                float(motion_dist[r, c]),\n                float(prob_matrix[r, c]),\n            ))\n        return matches\n\n    times = sorted(ids_by_t)\n    for t in times:\n        source_ids = ids_by_t.get(t, [])\n        target_ids = ids_by_t.get(t + 1, [])\n        if not source_ids or not target_ids:\n            continue\n        unmatched_sources = set(source_ids)\n        unmatched_targets = set(target_ids)\n        frame_matches: list[tuple[int, int, float, float, str, float]] = []\n        for pass_name, gate_um in (("tight", MOTION_RELINK_TIGHT_UM), ("relaxed", MOTION_RELINK_RELAXED_UM)):\n            pass_sources = [node_id for node_id in source_ids if node_id in unmatched_sources]\n            pass_targets = [node_id for node_id in target_ids if node_id in unmatched_targets]\n            matches = assign_pass(pass_sources, pass_targets, gate_um)\n            for source_id, target_id, raw, motion, prob in matches:\n                if source_id not in unmatched_sources or target_id not in unmatched_targets:\n                    continue\n                unmatched_sources.remove(source_id)\n                unmatched_targets.remove(target_id)\n                frame_matches.append((source_id, target_id, raw, motion, pass_name, prob))\n                if pass_name == "tight":\n                    stats["motion_relink_tight_edges"] += 1\n                else:\n                    stats["motion_relink_relaxed_edges"] += 1\n        for source_id, target_id, raw, motion, pass_name, prob in frame_matches:\n            selected_edges.append({\n                "source_id": source_id,\n                "target_id": target_id,\n                "edge_prob": prob,\n                "distance_um": raw,\n                "motion_distance_um": motion,\n                "motion_relinked": 1,\n                "motion_pass": pass_name,\n            })\n            predecessor_position_um[target_id] = position_um[source_id]\n        stats["motion_relink_frames"] += 1\n\n    stats["motion_relink_edges"] = len(selected_edges)\n    return selected_edges')




In [ ]:
TRAIN_DIR=COMP_DIR / "train"
VALIDATOR_MATCH_RADIUS_UM=7.0
VALIDATOR_NODE_COUNT_PENALTY_A=0.1
VALIDATOR_DIVISION_WEIGHT=0.1







from scipy.optimize import linear_sum_assignment


def match_nodes_bipartite(pred_nodes: dict, gt_nodes: dict, max_dist: float = 7.0):
    pred_by_t: dict[int, list[int]] = {}
    for pid, (t, *_r) in pred_nodes.items():
        pred_by_t.setdefault(int(t), []).append(pid)
    gt_by_t: dict[int, list[int]] = {}
    for gid, (t, *_r) in gt_nodes.items():
        gt_by_t.setdefault(int(t), []).append(gid)

    pred_to_gt: dict[int, int] = {}
    gt_to_pred: dict[int, int] = {}
    for t, p_ids in pred_by_t.items():
        g_ids = gt_by_t.get(t, [])
        if not g_ids:
            continue
        voxel_scale = np.array(VOXEL_SCALE_UM, dtype=float)
        p_pos = np.array([pred_nodes[p][1:] for p in p_ids], dtype=float) * voxel_scale
        g_pos = np.array([gt_nodes[g][1:] for g in g_ids], dtype=float) * voxel_scale
        diff = p_pos[:, None, :] - g_pos[None, :, :]
        cost = np.sqrt((diff ** 2).sum(axis=-1))
        BIG = 1e6
        cost_gated = np.where(cost <= max_dist, cost, BIG)
        row_ind, col_ind = linear_sum_assignment(cost_gated)
        for r, c in zip(row_ind, col_ind):
            if cost_gated[r, c] >= BIG:
                continue
            pred_to_gt[p_ids[r]] = g_ids[c]
            gt_to_pred[g_ids[c]] = p_ids[r]
    return pred_to_gt, gt_to_pred


def compute_edge_confusion(pred_edges, gt_edges, pred_to_gt, gt_to_pred):
    gt_edge_set = set(gt_edges)
    gt_outgoing: dict[int, set[int]] = {}
    gt_incoming_source: dict[int, int] = {}
    for s, t in gt_edge_set:
        gt_outgoing.setdefault(s, set()).add(t)
        gt_incoming_source[t] = s

    tp = 0
    fp = 0
    matched_gt_edges = set()
    for s, t in pred_edges:
        ms = pred_to_gt.get(s)
        mt = pred_to_gt.get(t)
        is_tp = ms is not None and mt is not None and mt in gt_outgoing.get(ms, ())
        if is_tp:
            tp += 1
            matched_gt_edges.add((ms, mt))
            continue
        is_fp = (mt is not None and mt in gt_incoming_source) or (
            ms is not None and bool(gt_outgoing.get(ms))
        )
        if is_fp:
            fp += 1
    fn = len(gt_edge_set - matched_gt_edges)
    return tp, fp, fn


def edge_jaccard(tp: int, fp: int, fn: int) -> float:
    denom = tp + fp + fn
    return tp / denom if denom else 0.0


def adjusted_jaccard(jaccard: float, t_pred: int, t_true, a: float = 0.1) -> float:
    if not t_true or t_true <= 0:
        return jaccard
    return max(0.0, jaccard * (1.0 - a * (t_pred - t_true) / t_true))


def weakly_connected_components(node_ids, edges):
    parent = {n: n for n in node_ids}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for s, t in edges:
        if s in parent and t in parent:
            union(s, t)
    return {n: find(n) for n in node_ids}


def compute_division_confusion(pred_nodes, pred_edges, gt_nodes, gt_edges, pred_to_gt, gt_to_pred):
    gt_out: dict[int, set[int]] = {}
    gt_in: dict[int, int] = {}
    for s, t in gt_edges:
        gt_out.setdefault(s, set()).add(t)
        gt_in[t] = s

    pred_out: dict[int, set[int]] = {}
    for s, t in pred_edges:
        pred_out.setdefault(s, set()).add(t)

    pred_node_ids = list(pred_nodes.keys())
    pred_edge_list = list(pred_edges)
    components = weakly_connected_components(pred_node_ids, pred_edge_list)
    fork_components = {
        components[n] for n, outs in pred_out.items() if len(outs) >= 2 and n in components
    }
    gt_division_sources = [s for s, outs in gt_out.items() if len(outs) >= 2]

    def lineage_descendants(root_child: int) -> set[int]:
        seen = {root_child}
        stack = [root_child]
        while stack:
            cur = stack.pop()
            for nxt in gt_out.get(cur, ()):
                if nxt not in seen:
                    seen.add(nxt)
                    stack.append(nxt)
        return seen

    tp = 0
    fn = 0
    tp_gt_sources: set[int] = set()

    for gsrc in gt_division_sources:
        children = sorted(gt_out[gsrc])
        if len(children) < 2:
            continue
        anchor_candidates = [gsrc]
        if gsrc in gt_in:
            anchor_candidates.append(gt_in[gsrc])
        anchor_pred_nodes = [gt_to_pred[a] for a in anchor_candidates if a in gt_to_pred]

        lineage_hit_components: list[set[int]] = []
        ok = True
        for child in children[:2]:
            lineage = lineage_descendants(child)
            hit_comp_ids = {
                components[p_id]
                for gt_id in lineage
                if (p_id := gt_to_pred.get(gt_id)) is not None and p_id in components
            }
            if not hit_comp_ids:
                ok = False
                break
            lineage_hit_components.append(hit_comp_ids)

        if not ok or not anchor_pred_nodes:
            fn += 1
            continue

        anchor_comp_ids = {components[p] for p in anchor_pred_nodes if p in components}
        if not anchor_comp_ids:
            fn += 1
            continue

        found = any(
            comp_id in lineage_hit_components[0]
            and comp_id in lineage_hit_components[1]
            and comp_id in fork_components
            for comp_id in anchor_comp_ids
        )
        if found:
            tp += 1
            tp_gt_sources.add(gsrc)
        else:
            fn += 1

    fp = 0
    for n, outs in pred_out.items():
        if len(outs) < 2:
            continue
        g = pred_to_gt.get(n)
        if g is None or g not in gt_out or g in tp_gt_sources:
            continue
        fp += 1

    return tp, fp, fn


def decompose_errors(pred_nodes, gt_nodes, pred_edges, gt_edges, pred_to_gt, gt_to_pred):
    """Splits error mass into detection vs. fragmentation vs. wrong-association,
    using the exact same pred_to_gt/gt_to_pred matching compute_edge_confusion
    uses. Division errors are already isolated by compute_division_confusion;
    this covers everything else -- the diagnostic breakdown for deciding
    whether further gains are in detection, linking, or fragmentation."""
    gt_edge_set = set(gt_edges)
    pred_edge_set = set(pred_edges)
    gt_outgoing: dict[int, set[int]] = {}
    for s, t in gt_edge_set:
        gt_outgoing.setdefault(s, set()).add(t)

    missed_gt_nodes = sum(1 for g in gt_nodes if g not in gt_to_pred)
    spurious_pred_nodes = sum(1 for p in pred_nodes if p not in pred_to_gt)

    recovered = fragmented = lost_to_detection = 0
    for gs, gtid in gt_edge_set:
        ps, pt = gt_to_pred.get(gs), gt_to_pred.get(gtid)
        if ps is None or pt is None:
            lost_to_detection += 1
        elif (ps, pt) in pred_edge_set:
            recovered += 1
        else:
            fragmented += 1

    wrong_association = 0
    for ps, pt in pred_edge_set:
        ms, mt = pred_to_gt.get(ps), pred_to_gt.get(pt)
        if ms is not None and mt is not None and mt not in gt_outgoing.get(ms, ()):
            wrong_association += 1

    return {
        "missed_gt_nodes": missed_gt_nodes,
        "spurious_pred_nodes": spurious_pred_nodes,
        "edges_recovered": recovered,
        "edges_fragmented": fragmented,
        "edges_lost_to_detection": lost_to_detection,
        "wrong_association_edges": wrong_association,
    }


def _find_key_recursive(obj, key):
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        for v in obj.values():
            found = _find_key_recursive(v, key)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for item in obj:
            found = _find_key_recursive(item, key)
            if found is not None:
                return found
    return None


def read_estimated_true_node_count(geff_path: Path):
    for candidate in (geff_path / "zarr.json", geff_path / ".zattrs"):
        if not candidate.exists():
            continue
        try:
            payload = json.loads(candidate.read_text())
        except Exception:
            continue
        found = _find_key_recursive(payload, "estimated_number_of_nodes")
        if found is not None:
            try:
                return float(found)
            except (TypeError, ValueError):
                continue
    return None


def graph_to_plain(graph):
    nodes: dict[int, tuple] = {}
    for row in graph.node_attrs().iter_rows(named=True):
        node_id = int(row["node_id"])
        nodes[node_id] = (int(row["t"]), float(row["z"]), float(row["y"]), float(row["x"]))
    edges: list[tuple[int, int]] = []
    for row in graph.edge_attrs().iter_rows(named=True):
        edges.append((int(row["source_id"]), int(row["target_id"])))
    return nodes, edges


def nodes_by_id_to_plain(nodes_by_id):
    return {nid: (int(n["t"]), float(n["z"]), float(n["y"]), float(n["x"])) for nid, n in nodes_by_id.items()}


def score_sample(pred_nodes_plain, pred_edges_plain, gt_nodes_plain, gt_edges_plain, t_true):
    p2g, g2p = match_nodes_bipartite(pred_nodes_plain, gt_nodes_plain, max_dist=VALIDATOR_MATCH_RADIUS_UM)
    tp, fp, fn = compute_edge_confusion(pred_edges_plain, gt_edges_plain, p2g, g2p)
    jac = edge_jaccard(tp, fp, fn)
    t_pred = len(pred_nodes_plain)
    adj = adjusted_jaccard(jac, t_pred, t_true, a=VALIDATOR_NODE_COUNT_PENALTY_A)
    div_tp, div_fp, div_fn = compute_division_confusion(
        pred_nodes_plain, pred_edges_plain, gt_nodes_plain, gt_edges_plain, p2g, g2p
    )
    errors = decompose_errors(pred_nodes_plain, gt_nodes_plain, pred_edges_plain, gt_edges_plain, p2g, g2p)
    div_jac = edge_jaccard(div_tp, div_fp, div_fn)
    row = {
        "edge_tp": tp, "edge_fp": fp, "edge_fn": fn, "edge_jaccard": jac,
        "t_pred": t_pred, "t_true": t_true, "adjusted_edge_jaccard": adj,
        "div_tp": div_tp, "div_fp": div_fp, "div_fn": div_fn, "div_jaccard": div_jac,
        "weight": tp + fp + fn,
    }
    row.update(errors)
    return row


def aggregate_official(sample_rows):
    total_w = sum(r["weight"] for r in sample_rows) or 1
    weighted_adj = sum(r["adjusted_edge_jaccard"] * r["weight"] for r in sample_rows) / total_w
    div_tp = sum(r["div_tp"] for r in sample_rows)
    div_fp = sum(r["div_fp"] for r in sample_rows)
    div_fn = sum(r["div_fn"] for r in sample_rows)
    div_jac = edge_jaccard(div_tp, div_fp, div_fn)
    return {
        "adjusted_edge_jaccard": weighted_adj,
        "division_jaccard": div_jac,
        "proxy_score": weighted_adj + VALIDATOR_DIVISION_WEIGHT * div_jac,
        "div_tp": div_tp, "div_fp": div_fp, "div_fn": div_fn,
        "missed_gt_nodes": sum(r["missed_gt_nodes"] for r in sample_rows),
        "spurious_pred_nodes": sum(r["spurious_pred_nodes"] for r in sample_rows),
        "edges_recovered": sum(r["edges_recovered"] for r in sample_rows),
        "edges_fragmented": sum(r["edges_fragmented"] for r in sample_rows),
        "edges_lost_to_detection": sum(r["edges_lost_to_detection"] for r in sample_rows),
        "wrong_association_edges": sum(r["wrong_association_edges"] for r in sample_rows),
    }

# TARGET950: evaluate the same held-out graphs with the fixed public scorer.
# This changes automatic PP selection only. It does not claim the private
# Kaggle deployment is byte-identical or that a higher local score is a gain.
import hashlib as _official_hashlib
import importlib.util as _official_importlib
import sys as _official_sys
_official_files = {'__init__.py': '', 'metrics.py': 'import warnings\nfrom typing import Literal, NamedTuple\n\nimport polars as pl\nimport tracksdata as td\n\n\nclass EvaluationResult(NamedTuple):\n    """Counts returned by :func:`evaluate`."""\n\n    edge_tp: int\n    edge_fp: int\n    edge_fn: int\n    division_tp: int\n    division_fp: int\n    division_fn: int\n    num_pred_nodes: int\n\n\nclass DatasetsResult(NamedTuple):\n    """Cumulative (micro-averaged) Jaccards plus the combined score."""\n\n    edge_jaccard: float\n    division_jaccard: float\n    score: float\n\n\n# Penalty coefficient for the adjusted edge Jaccard:\n#   J_adj = max(0, J · (1 - ADJUSTMENT_ALPHA · total_node_ratio))\nADJUSTMENT_ALPHA: float = 0.1\n\n# Weight of the division Jaccard in the combined run-level score:\n#   score = adj_edge_jaccard + SCORE_DIVISION_WEIGHT · division_jaccard\nSCORE_DIVISION_WEIGHT: float = 0.1\n\nCOUNT_COLUMNS: tuple[str, ...] = (\n    "edge_tp", "edge_fp", "edge_fn",\n    "division_tp", "division_fp", "division_fn",\n    "num_pred_nodes",\n)\nMETRIC_COLUMNS: tuple[str, ...] = COUNT_COLUMNS + (\n    "node_recall", "total_node_ratio", "edge_jaccard", "adj_edge_jaccard",\n)\n\n\ndef _jaccard(tp: int, fp: int, fn: int) -> float:\n    denom = tp + fp + fn\n    return tp / denom if denom > 0 else float("nan")\n\n\n# function is split for easier testing\ndef _evaluate_matched_graph(\n    graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n) -> pl.DataFrame:\n    edge_attrs = graph.edge_attrs(attr_keys=[td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK])\n    # Guard against duplicate edges (same source→target pair appearing multiple times).\n    # tracksdata\'s match() inner-join marks all duplicates as matched, which inflates\n    # the intersection count and can push scores above 1.0. Sort matched rows first\n    # so the dedup keeps the matched copy when duplicates disagree on the mask.\n    edge_attrs = edge_attrs.sort(\n        td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK, descending=True,\n    ).unique(\n        subset=[td.DEFAULT_ATTR_KEYS.EDGE_SOURCE, td.DEFAULT_ATTR_KEYS.EDGE_TARGET],\n        keep="first",\n    )\n    node_attrs = graph.node_attrs(\n        attr_keys=[td.DEFAULT_ATTR_KEYS.NODE_ID, td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID, td.DEFAULT_ATTR_KEYS.T]\n    )\n\n    # Drop edges that do not connect consecutive frames, i.e. keep only edges where\n    # t_target == t_source + 1. This removes backward-in-time edges (t_target <= t_source)\n    # and any edge spanning more than a single time step (t_target - t_source > 1).\n    node_times = node_attrs.select(td.DEFAULT_ATTR_KEYS.NODE_ID, td.DEFAULT_ATTR_KEYS.T)\n    edge_attrs = edge_attrs.join(\n        node_times.rename({td.DEFAULT_ATTR_KEYS.T: "_source_t"}),\n        left_on=td.DEFAULT_ATTR_KEYS.EDGE_SOURCE,\n        right_on=td.DEFAULT_ATTR_KEYS.NODE_ID,\n        how="left",\n    ).join(\n        node_times.rename({td.DEFAULT_ATTR_KEYS.T: "_target_t"}),\n        left_on=td.DEFAULT_ATTR_KEYS.EDGE_TARGET,\n        right_on=td.DEFAULT_ATTR_KEYS.NODE_ID,\n        how="left",\n    ).filter(\n        pl.col("_target_t") - pl.col("_source_t") == 1\n    ).drop("_source_t", "_target_t")\n\n    # Collapse merges: when several predicted nodes match the same ground-truth\n    # node, multiple predicted edges can map onto the same ground-truth edge\n    # (identical matched source/target pair). tracksdata marks all of them as\n    # matched, inflating the intersection. Keep only the edge with the lowest\n    # EDGE_ID per matched GT edge and discard the rest with a warning.\n    matched_ids = node_attrs.select(\n        td.DEFAULT_ATTR_KEYS.NODE_ID, td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID\n    )\n    edge_attrs = edge_attrs.join(\n        matched_ids.rename({td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID: "_matched_source"}),\n        left_on=td.DEFAULT_ATTR_KEYS.EDGE_SOURCE,\n        right_on=td.DEFAULT_ATTR_KEYS.NODE_ID,\n        how="left",\n    ).join(\n        matched_ids.rename({td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID: "_matched_target"}),\n        left_on=td.DEFAULT_ATTR_KEYS.EDGE_TARGET,\n        right_on=td.DEFAULT_ATTR_KEYS.NODE_ID,\n        how="left",\n    )\n    # Only edges whose endpoints both match a GT node can collapse onto a GT edge.\n    both_matched = (\n        pl.col("_matched_source").is_not_null()\n        & pl.col("_matched_target").is_not_null()\n        & (pl.col("_matched_source") != -1)\n        & (pl.col("_matched_target") != -1)\n    )\n    edge_attrs = edge_attrs.with_columns(\n        (\n            both_matched\n            & (\n                pl.col(td.DEFAULT_ATTR_KEYS.EDGE_ID)\n                != pl.col(td.DEFAULT_ATTR_KEYS.EDGE_ID)\n                .min()\n                .over("_matched_source", "_matched_target")\n            )\n        ).alias("_is_merge_dup")\n    )\n    n_merge_dropped = int(edge_attrs["_is_merge_dup"].sum())\n    if n_merge_dropped > 0:\n        warnings.warn(\n            f"Dropped {n_merge_dropped} merged edge(s) mapping onto the same "\n            "ground-truth edge; kept the lowest edge id per merge.",\n            stacklevel=2,\n        )\n    edge_attrs = edge_attrs.filter(~pl.col("_is_merge_dup")).drop(\n        "_matched_source", "_matched_target", "_is_merge_dup"\n    )\n\n    # Cap out-degree: a dividing cell has at most two children, so a predicted node\n    # with more than two outgoing edges is biologically invalid. Keep the two edges\n    # with the lowest EDGE_ID per source and drop the rest with a warning.\n    edge_attrs = edge_attrs.with_columns(\n        pl.col(td.DEFAULT_ATTR_KEYS.EDGE_ID)\n        .rank("ordinal")\n        .over(td.DEFAULT_ATTR_KEYS.EDGE_SOURCE)\n        .alias("_out_rank")\n    )\n    n_outdeg_dropped = int((edge_attrs["_out_rank"] > 2).sum())\n    if n_outdeg_dropped > 0:\n        warnings.warn(\n            f"Dropped {n_outdeg_dropped} outgoing edge(s) from nodes with more than "\n            "two children; kept the two lowest edge ids per source.",\n            stacklevel=2,\n        )\n    edge_attrs = edge_attrs.filter(pl.col("_out_rank") <= 2).drop("_out_rank")\n\n    # I\'m assuming valid ground-truth edges are always 100% correct if they have an edge.\n    # Therefore, we don\'t have cases where the cell divided, but not in the ground truth.\n    gt_node_ids = gt_graph.node_ids()\n    gt_node_attrs = pl.DataFrame(\n        {\n            td.DEFAULT_ATTR_KEYS.NODE_ID: gt_node_ids,\n            "out_degree": gt_graph.out_degree(gt_node_ids),\n            "in_degree": gt_graph.in_degree(gt_node_ids),\n        }\n    ).with_columns(\n        (pl.col("out_degree") > 0).alias("out_valid"),\n        (pl.col("in_degree") > 0).alias("in_valid"),\n    )\n\n    # merging ground truth graph into the predicted graph\n    node_attrs = node_attrs.join(\n        gt_node_attrs,\n        left_on=td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID,\n        right_on=td.DEFAULT_ATTR_KEYS.NODE_ID,\n        how="left",\n    ).with_columns(\n        pl.col("out_valid").fill_null(False),\n        pl.col("in_valid").fill_null(False),\n    )\n\n    # merge out valid into source and in valid into target\n    edge_attrs = edge_attrs.join(\n        node_attrs.select(td.DEFAULT_ATTR_KEYS.NODE_ID, "out_valid"),\n        left_on=td.DEFAULT_ATTR_KEYS.EDGE_SOURCE,\n        right_on=td.DEFAULT_ATTR_KEYS.NODE_ID,\n        how="left",\n    ).join(\n        node_attrs.select(td.DEFAULT_ATTR_KEYS.NODE_ID, "in_valid"),\n        left_on=td.DEFAULT_ATTR_KEYS.EDGE_TARGET,\n        right_on=td.DEFAULT_ATTR_KEYS.NODE_ID,\n        how="left",\n    )\n\n    edge_attrs = edge_attrs.with_columns(\n        (pl.col("out_valid") | pl.col("in_valid")).alias("pred_valid"),\n    )\n\n    # sanity check that `pred_valid` is a superset of all matched edges\n    assert edge_attrs.filter(td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK)["pred_valid"].all()\n\n    return edge_attrs\n\n\ndef _compute_score(\n    edge_attrs: pl.DataFrame,\n    gt_num_edges: int,\n    metric: Literal["jaccard", "dice"],\n) -> float:\n    intersection = int(edge_attrs[td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK].sum())\n    n_valid_pred_edges = int(edge_attrs["pred_valid"].sum())\n\n    if metric == "jaccard":\n        num = intersection\n        denom = gt_num_edges + n_valid_pred_edges - intersection\n    elif metric == "dice":\n        num = 2 * intersection\n        denom = gt_num_edges + n_valid_pred_edges\n    else:\n        raise ValueError(f"Invalid metric: {metric}")\n\n    return num / denom if denom > 0 else float("nan")\n\n\ndef _evaluate(\n    graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n    metric: Literal["jaccard", "dice"],\n    scale: tuple[float, ...] | None,\n    max_distance: float,\n) -> float:\n    if td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID in graph.node_attr_keys():\n        warnings.warn("Graph already matched, overwriting previous matching.")\n        # Reset matching attributes to defaults before re-matching\n        all_node_ids = graph.node_ids()\n        graph.update_node_attrs(\n            node_ids=all_node_ids,\n            attrs={\n                td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID: -1,\n                td.DEFAULT_ATTR_KEYS.MATCH_SCORE: 0.0,\n            },\n        )\n        all_edge_ids = graph.edge_ids()\n        if len(all_edge_ids) > 0:\n            graph.update_edge_attrs(\n                edge_ids=all_edge_ids,\n                attrs={td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK: False},\n            )\n\n    from tracksdata.metrics import DistanceMatching\n    matching = DistanceMatching(max_distance=max_distance, scale=scale)\n\n    if graph.num_edges() == 0 or graph.num_nodes() == 0:\n        warnings.warn("Predicted graph has no edges or no nodes, returning score 0.0.")\n        return 0.0\n\n    from tracksdata.options import get_options, set_options\n\n    prev_show_progress = get_options().show_progress\n    set_options(show_progress=False)\n    try:\n        with warnings.catch_warnings():\n            from scipy.sparse import SparseEfficiencyWarning\n            warnings.filterwarnings("ignore", category=SparseEfficiencyWarning)\n            graph.match(gt_graph, matching=matching)\n    finally:\n        set_options(show_progress=prev_show_progress)\n\n    edge_attrs = _evaluate_matched_graph(graph, gt_graph)\n\n    return _compute_score(edge_attrs, gt_graph.num_edges(), metric)\n\n\ndef evaluate(\n    graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n    scale: tuple[float, ...] | None = None,\n    max_distance: float = 7.0,\n) -> EvaluationResult:\n    """\n    Evaluate a predicted graph against a ground-truth graph using\n    centroid-distance node matching.\n\n    Computes edge TP/FP/FN, division TP/FP/FN (via\n    :func:`tracking_cellmot.division_metrics.evaluate_divisions`), and the\n    total number of predicted nodes (irrespective of matching).\n\n    Parameters\n    ----------\n    graph : tracksdata.graph.BaseGraph\n        The predicted graph. Matching attributes are written onto *graph*\n        as a side effect.\n    gt_graph : tracksdata.graph.BaseGraph\n        The ground truth graph.\n    scale : tuple[float, ...] | None, optional\n        Physical scale for each spatial dimension (e.g., (z, y, x)) to\n        account for anisotropy. If None, assumes isotropic data.\n    max_distance : float, optional\n        Maximum distance between centroids to be considered as a match.\n\n    Returns\n    -------\n    EvaluationResult\n    """\n    from .division_metrics import evaluate_divisions\n\n    # Match graph against gt_graph (in place); discard the returned score.\n    _evaluate(graph, gt_graph, "jaccard", scale, max_distance)\n\n    if graph.num_edges() == 0:\n        edge_tp = 0\n        edge_fp = 0\n        edge_fn = gt_graph.num_edges()\n    else:\n        edge_attrs = _evaluate_matched_graph(graph, gt_graph)\n        edge_tp = int(edge_attrs[td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK].sum())\n        edge_valid_pred = int(edge_attrs["pred_valid"].sum())\n        edge_fp = edge_valid_pred - edge_tp\n        edge_fn = gt_graph.num_edges() - edge_tp\n\n    div = evaluate_divisions(\n        graph, gt_graph, scale=scale, max_distance=max_distance,\n    )\n\n    return EvaluationResult(\n        edge_tp=edge_tp,\n        edge_fp=edge_fp,\n        edge_fn=edge_fn,\n        division_tp=div.tp,\n        division_fp=div.fp,\n        division_fn=div.fn,\n        num_pred_nodes=graph.num_nodes(),\n    )\n\n\ndef evaluate_datasets(\n    graph_pairs: list[tuple[td.graph.BaseGraph, td.graph.BaseGraph]],\n    scale: tuple[float, ...] | None = None,\n    max_distance: float = 7.0,\n) -> DatasetsResult:\n    """Run :func:`evaluate` on each (pred, gt) pair and return cumulative\n    (micro-averaged) edge and division Jaccard.\n\n    Per-pair TP/FP/FN counts are summed across the whole list before the\n    Jaccard is computed, so larger datasets dominate the score naturally.\n\n    Parameters\n    ----------\n    graph_pairs : list of (pred_graph, gt_graph)\n        Predicted / ground-truth graph pairs. Each *pred_graph* is mutated\n        in place by matching (same side effect as :func:`evaluate`).\n    scale : tuple[float, ...] | None, optional\n        Physical voxel scale used for centroid-distance matching.\n    max_distance : float, optional\n        Maximum centroid distance for a match.\n\n    Returns\n    -------\n    DatasetsResult\n        Named tuple with ``edge_jaccard``, ``division_jaccard``, and the\n        combined ``score = edge_jaccard + SCORE_DIVISION_WEIGHT *\n        division_jaccard``. If no divisions exist anywhere in the input\n        the division term is dropped and ``score = edge_jaccard``.\n    """\n    edge_tp = edge_fp = edge_fn = 0\n    div_tp = div_fp = div_fn = 0\n    for pred, gt in graph_pairs:\n        r = evaluate(pred, gt, scale=scale, max_distance=max_distance)\n        edge_tp += r.edge_tp\n        edge_fp += r.edge_fp\n        edge_fn += r.edge_fn\n        div_tp += r.division_tp\n        div_fp += r.division_fp\n        div_fn += r.division_fn\n\n    edge_jaccard = _jaccard(edge_tp, edge_fp, edge_fn)\n    has_divisions = (div_tp + div_fp + div_fn) > 0\n    division_jaccard = _jaccard(div_tp, div_fp, div_fn) if has_divisions else float("nan")\n    score = edge_jaccard + SCORE_DIVISION_WEIGHT * division_jaccard if has_divisions else edge_jaccard\n\n    return DatasetsResult(\n        edge_jaccard=edge_jaccard,\n        division_jaccard=division_jaccard,\n        score=score,\n    )\n\n\ndef _matched_node_ids(graph: td.graph.BaseGraph) -> pl.DataFrame:\n    """Return a DataFrame with NODE_ID and MATCHED_NODE_ID (as Int64) for *graph*."""\n    node_attrs = graph.node_attrs(\n        attr_keys=[td.DEFAULT_ATTR_KEYS.NODE_ID, td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID]\n    )\n    return node_attrs\n\n\ndef node_recall(\n    graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n) -> float:\n    """Fraction of GT nodes that were matched by a predicted node.\n\n    The predicted graph must already be matched (e.g. via :func:`evaluate` or\n    ``graph.match``).\n    """\n    node_attrs = _matched_node_ids(graph)\n    matched = node_attrs.filter(\n        pl.col(td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID).is_not_null()\n        & (pl.col(td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID) != -1)\n    )\n    n_matched_gt = matched[td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID].n_unique()\n    return n_matched_gt / gt_graph.num_nodes()\n\n\ndef per_sample_metrics(\n    er: EvaluationResult,\n    n_total: float,\n    node_recall: float,\n) -> dict:\n    """Derive per-sample metric columns from an :class:`EvaluationResult`.\n\n    Computes ``edge_jaccard``, ``total_node_ratio`` (``(N_pred − N_total) / N_total``),\n    and the adjusted edge Jaccard ``J_adj = max(0, J · (1 − α · total_node_ratio))``\n    with α = :data:`ADJUSTMENT_ALPHA`.\n\n    Parameters\n    ----------\n    er\n        Counts for one (pred, gt) pair — see :func:`evaluate`.\n    n_total\n        Target node count (e.g. from the GEFF ``estimated_number_of_nodes``\n        metadata extra). Pass ``float("nan")`` when unavailable; that makes\n        ``total_node_ratio`` and ``adj_edge_jaccard`` also NaN.\n    node_recall\n        Fraction of GT nodes matched by a predicted node.\n\n    Returns\n    -------\n    dict\n        One entry per key in :data:`METRIC_COLUMNS`.\n    """\n    if n_total > 0:\n        total_node_ratio = (er.num_pred_nodes - n_total) / n_total\n    else:\n        total_node_ratio = float("nan")\n\n    edge_denom = er.edge_tp + er.edge_fp + er.edge_fn\n    edge_jaccard = er.edge_tp / edge_denom if edge_denom > 0 else float("nan")\n    if edge_jaccard == edge_jaccard and total_node_ratio == total_node_ratio:\n        adj_edge_jaccard = max(\n            0.0, edge_jaccard * (1 - ADJUSTMENT_ALPHA * total_node_ratio),\n        )\n    else:\n        adj_edge_jaccard = float("nan")\n\n    return {\n        "edge_tp": er.edge_tp, "edge_fp": er.edge_fp, "edge_fn": er.edge_fn,\n        "division_tp": er.division_tp,\n        "division_fp": er.division_fp,\n        "division_fn": er.division_fn,\n        "num_pred_nodes": er.num_pred_nodes,\n        "node_recall": node_recall,\n        "total_node_ratio": total_node_ratio,\n        "edge_jaccard": edge_jaccard,\n        "adj_edge_jaccard": adj_edge_jaccard,\n    }\n\n\ndef nan_metrics_row() -> dict:\n    """Return a dict with every :data:`METRIC_COLUMNS` key set to NaN."""\n    return {col: float("nan") for col in METRIC_COLUMNS}\n\n\ndef summarise(rows: list[dict]) -> dict:\n    """Aggregate per-sample metric rows into a run-level summary.\n\n    - ``edge_jaccard`` / ``division_jaccard``: micro-averaged across valid rows\n      (TP/FP/FN summed, then Jaccard).\n    - ``adj_edge_jaccard``: per-sample adjusted Jaccard weight-averaged by\n      sample size ``w_i = TP_i + FP_i + FN_i``; rows with NaN are skipped.\n    - ``score``: ``adj_edge_jaccard + SCORE_DIVISION_WEIGHT · division_jaccard``.\n\n    Parameters\n    ----------\n    rows\n        Per-sample dicts as produced by :func:`per_sample_metrics`. Rows with\n        NaN ``edge_tp`` are treated as failed evaluations and skipped.\n    """\n    valid = [r for r in rows if r["edge_tp"] == r["edge_tp"]]\n    if not valid:\n        return {\n            "n": 0, "edge_jaccard": float("nan"),\n            "division_jaccard": float("nan"),\n            "division_tp": 0, "division_fp": 0, "division_fn": 0,\n            "node_recall": float("nan"),\n            "adj_edge_jaccard": float("nan"), "n_adj": 0,\n            "score": float("nan"),\n        }\n    totals = {c: sum(r[c] for r in valid) for c in COUNT_COLUMNS}\n\n    adj_rows = [r for r in valid if r["adj_edge_jaccard"] == r["adj_edge_jaccard"]]\n    weights = [r["edge_tp"] + r["edge_fp"] + r["edge_fn"] for r in adj_rows]\n    total_w = sum(weights)\n    if total_w > 0:\n        adj_edge_jaccard = sum(\n            w * r["adj_edge_jaccard"] for w, r in zip(weights, adj_rows)\n        ) / total_w\n    else:\n        adj_edge_jaccard = float("nan")\n\n    division_total = (\n        totals["division_tp"] + totals["division_fp"] + totals["division_fn"]\n    )\n    if division_total == 0:\n        warnings.warn(\n            "No divisions present across any sample in this split; "\n            "dropping division term from the combined score."\n        )\n        division_jaccard = float("nan")\n        score = adj_edge_jaccard\n    else:\n        division_jaccard = _jaccard(\n            totals["division_tp"], totals["division_fp"], totals["division_fn"],\n        )\n        score = adj_edge_jaccard + SCORE_DIVISION_WEIGHT * division_jaccard\n    return {\n        "n": len(valid),\n        "edge_jaccard": _jaccard(\n            totals["edge_tp"], totals["edge_fp"], totals["edge_fn"],\n        ),\n        "division_jaccard": division_jaccard,\n        "division_tp": totals["division_tp"],\n        "division_fp": totals["division_fp"],\n        "division_fn": totals["division_fn"],\n        "node_recall": sum(r["node_recall"] for r in valid) / len(valid),\n        "adj_edge_jaccard": adj_edge_jaccard,\n        "n_adj": len(adj_rows),\n        "score": score,\n    }\n', 'division_metrics.py': 'import warnings\nfrom typing import NamedTuple\n\nimport polars as pl\nimport tracksdata as td\n\n\nclass DivisionCounts(NamedTuple):\n    """Counts for division event evaluation."""\n\n    tp: int\n    fn: int\n    fp: int\n\n\nclass DivisionScores(NamedTuple):\n    """Result of :func:`score_divisions`.\n\n    Attributes\n    ----------\n    scores : dict[int, int]\n        Mapping from GT dividing-node ID to 1 (recovered) or 0 (not).\n    tp_forks : set[int]\n        Predicted dividing nodes paired to GT divisions.\n    fp_forks : set[int]\n        Predicted dividing nodes that were considered for a GT division\n        but did not become a true positive, including local-topology\n        rejects, bipartite leftovers, evaluable spurious forks, malformed\n        local branches, and forks whose branch evidence spans distinct GT\n        components.\n    """\n\n    scores: dict[int, int]\n    tp_forks: set[int]\n    fp_forks: set[int]\n\n\ndef _reset_matching_attrs(graph: td.graph.BaseGraph) -> None:\n    """Reset any pre-existing match attrs in place so a fresh ``.match()`` isn\'t\n    contaminated by stale values carried in from a previous matching pass."""\n    node_keys = graph.node_attr_keys()\n    if td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID in node_keys:\n        node_ids = graph.node_ids()\n        if len(node_ids) > 0:\n            reset: dict = {td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID: -1}\n            if td.DEFAULT_ATTR_KEYS.MATCH_SCORE in node_keys:\n                reset[td.DEFAULT_ATTR_KEYS.MATCH_SCORE] = 0.0\n            graph.update_node_attrs(node_ids=node_ids, attrs=reset)\n    if td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK in graph.edge_attr_keys():\n        edge_ids = graph.edge_ids()\n        if len(edge_ids) > 0:\n            graph.update_edge_attrs(\n                edge_ids=edge_ids,\n                attrs={td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK: False},\n            )\n\n\ndef extract_divisions(\n    graph: td.graph.BaseGraph,\n) -> dict[int, td.graph.BaseGraph]:\n    """Extract individual division events as separate subgraphs.\n\n    Each division event includes the parent of the dividing node, the\n    dividing node, its children, and the grandchildren::\n\n        parent → divider → child1 → grandchild1\n                         → child2 → grandchild2\n\n    Parameters\n    ----------\n    graph : td.graph.BaseGraph\n        The input tracking graph.\n\n    Returns\n    -------\n    dict[int, td.graph.BaseGraph]\n        Mapping from dividing node ID to a subgraph containing the\n        parent, divider, children, and grandchildren.\n    """\n    divisions: dict[int, td.graph.BaseGraph] = {}\n    for div_node in graph.dividing_nodes():\n        parents = graph.predecessors(div_node)\n        children = graph.successors(div_node)\n        grandchildren = [gc for child in children for gc in graph.successors(child)]\n        keep = [*parents, div_node, *children, *grandchildren]\n        divisions[div_node] = graph.filter(node_ids=keep).subgraph()\n    return divisions\n\n\ndef match_divisions(\n    pred_graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n    scale: tuple[float, ...] | None = None,\n    max_distance: float = 7.0,\n) -> dict[int, td.graph.BaseGraph]:\n    """Match the predicted graph against each GT division subgraph.\n\n    Extracts division events from *gt_graph* via :func:`extract_divisions`,\n    then runs ``pred_graph.match(gt_div, ...)`` for each one independently.\n    A fresh copy of *pred_graph* is used per division so matchings don\'t\n    interfere.\n\n    Parameters\n    ----------\n    pred_graph : td.graph.BaseGraph\n        The predicted tracking graph.\n    gt_graph : td.graph.BaseGraph\n        The ground-truth tracking graph.\n    scale : tuple[float, ...] | None\n        Physical voxel scale used for centroid-distance matching.\n    max_distance : float\n        Maximum centroid distance for a match.\n\n    Returns\n    -------\n    dict[int, td.graph.BaseGraph]\n        Mapping from GT dividing-node ID to the matched copy of\n        *pred_graph* for that division.\n    """\n    from tracksdata.metrics import DistanceMatching\n\n    matching = DistanceMatching(max_distance=max_distance, scale=scale)\n\n    gt_divisions = extract_divisions(gt_graph)\n    matched: dict[int, td.graph.BaseGraph] = {}\n\n    from tracksdata.options import get_options, set_options\n\n    prev_show_progress = get_options().show_progress\n    set_options(show_progress=False)\n    try:\n        for div_node, gt_div in gt_divisions.items():\n            pred_copy = pred_graph.copy()\n            _reset_matching_attrs(pred_copy)\n            with warnings.catch_warnings():\n                from scipy.sparse import SparseEfficiencyWarning\n\n                warnings.filterwarnings("ignore", category=SparseEfficiencyWarning)\n                pred_copy.match(gt_div, matching=matching)\n            matched[div_node] = pred_copy\n    finally:\n        set_options(show_progress=prev_show_progress)\n\n    return matched\n\n\ndef _match_full(\n    pred_graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n    scale: tuple[float, ...] | None,\n    max_distance: float,\n) -> td.graph.BaseGraph:\n    """Match the full pred graph against the full GT graph, return the matched copy."""\n    from tracksdata.metrics import DistanceMatching\n\n    matching = DistanceMatching(max_distance=max_distance, scale=scale)\n\n    pred_copy = pred_graph.copy()\n    _reset_matching_attrs(pred_copy)\n\n    from tracksdata.options import get_options, set_options\n\n    prev_show_progress = get_options().show_progress\n    set_options(show_progress=False)\n    try:\n        with warnings.catch_warnings():\n            from scipy.sparse import SparseEfficiencyWarning\n\n            warnings.filterwarnings("ignore", category=SparseEfficiencyWarning)\n            pred_copy.match(gt_graph, matching=matching)\n    finally:\n        set_options(show_progress=prev_show_progress)\n\n    return pred_copy\n\n\ndef _matched_node_attrs(graph: td.graph.BaseGraph) -> pl.DataFrame:\n    """Return pred/GT node-ID pairs for matched prediction nodes."""\n    node_attrs = graph.node_attrs(\n        attr_keys=[\n            td.DEFAULT_ATTR_KEYS.NODE_ID,\n            td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID,\n        ],\n    )\n    return node_attrs.filter(\n        pl.col(td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID).is_not_null()\n        & (pl.col(td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID) != -1)\n    )\n\n\ndef _matched_division_nodes(\n    matched_attrs: pl.DataFrame,\n    gt_div: td.graph.BaseGraph,\n    divider_id: int,\n) -> tuple[set[int], list[set[int]]] | None:\n    """Group matched pred nodes by their role in a GT division window.\n\n    The parent side contains the GT divider (the parent cell) and its\n    immediate predecessor (the grandparent). Each daughter side contains\n    one GT child and its immediate successors (the grandchildren).\n    """\n    if matched_attrs.is_empty():\n        return None\n\n    node_to_gt = dict(\n        zip(\n            matched_attrs[td.DEFAULT_ATTR_KEYS.NODE_ID].to_list(),\n            matched_attrs[td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID].to_list(),\n            strict=True,\n        )\n    )\n    gt_children = gt_div.successors(divider_id)\n    if len(gt_children) < 2:\n        return None\n\n    gt_parent_ids = {divider_id, *gt_div.predecessors(divider_id)}\n    parent_ids = {pred_id for pred_id, gt_id in node_to_gt.items() if gt_id in gt_parent_ids}\n    daughter_ids = [\n        {pred_id for pred_id, gt_id in node_to_gt.items() if gt_id in {child, *gt_div.successors(child)}}\n        for child in gt_children\n    ]\n    if not parent_ids or sum(bool(ids) for ids in daughter_ids) < 2:\n        return None\n    return parent_ids, daughter_ids\n\n\ndef _is_strongly_connected_division(\n    pred_graph: td.graph.BaseGraph,\n    pred_div: int,\n    parent_ids: set[int],\n    daughter_ids: list[set[int]],\n) -> bool:\n    """Check a predicted division\'s local directed topology.\n\n    The prediction window mirrors :func:`extract_divisions`: an immediate\n    predecessor (grandparent), *pred_div* (parent), its children, and their\n    children (grandchildren). The parent match must be the fork itself or\n    its immediate predecessor. Matches from at least two GT daughter\n    lineages must occur in two distinct predicted child lineages.\n\n    Parameters\n    ----------\n    pred_graph : td.graph.BaseGraph\n        The predicted tracking graph.\n    pred_div : int\n        Candidate predicted dividing node (the parent/fork).\n    parent_ids : set[int]\n        Prediction node IDs matched to the GT parent side (grandparent or\n        dividing parent).\n    daughter_ids : list[set[int]]\n        Prediction node IDs matched to each GT daughter lineage (child or\n        grandchild), grouped by lineage.\n\n    Returns\n    -------\n    bool\n        Whether the local prediction topology connects the parent side to\n        at least two distinct daughter lineages through *pred_div*.\n    """\n    pred_parent_ids = {pred_div, *pred_graph.predecessors(pred_div)}\n    if pred_parent_ids.isdisjoint(parent_ids):\n        return False\n\n    pred_lineages = [{child, *pred_graph.successors(child)} for child in pred_graph.successors(pred_div)]\n    lineage_edges = {\n        gt_lineage: {\n            pred_lineage for pred_lineage, pred_ids in enumerate(pred_lineages) if not matched_ids.isdisjoint(pred_ids)\n        }\n        for gt_lineage, matched_ids in enumerate(daughter_ids)\n    }\n    return len(_bipartite_max_matching(list(lineage_edges), lineage_edges)) >= 2\n\n\ndef _bipartite_max_matching(\n    left: list[int],\n    edges: dict[int, set[int]],\n) -> dict[int, int]:\n    """Maximum-cardinality bipartite matching via DFS augmenting paths.\n\n    *edges* maps each left-side vertex to the set of adjacent right-side\n    vertices. Returns only the matched pairs as a ``left → right`` dict.\n    """\n    match_r: dict[int, int] = {}\n    match_l: dict[int, int] = {}\n\n    def augment(u: int, seen: set[int]) -> bool:\n        for v in edges.get(u, ()):\n            if v in seen:\n                continue\n            seen.add(v)\n            if v not in match_r or augment(match_r[v], seen):\n                match_l[u] = v\n                match_r[v] = u\n                return True\n        return False\n\n    for u in left:\n        augment(u, set())\n\n    return match_l\n\n\ndef score_divisions(\n    pred_graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n    scale: tuple[float, ...] | None = None,\n    max_distance: float = 7.0,\n) -> DivisionScores:\n    """Score each GT division: 1 if the prediction recovers it, 0 otherwise.\n\n    For each GT division, the predicted graph is matched against its\n    parent/divider/children/grandchildren window. Candidate pred forks are\n    restricted to the matched parent-side nodes and their immediate\n    successors. A candidate is valid only when its local topology contains\n    a matched parent and matches from two GT daughter lineages on distinct\n    predicted child branches. A fork is rejected when two direct-child\n    branches have nearest matched evidence in distinct reliable GT components.\n    An unmatched child may use unambiguous grandchild evidence as a fallback;\n    matched children take precedence over downstream matches.\n\n    A maximum-cardinality bipartite matching is then computed so each pred\n    fork serves at most one GT division, and each GT division is paired\n    with at most one pred fork. A GT division scores 1 only if paired;\n    rejected candidates and valid candidates left unpaired are returned as\n    false-positive forks.\n\n    Parameters\n    ----------\n    pred_graph : td.graph.BaseGraph\n        The predicted tracking graph.\n    gt_graph : td.graph.BaseGraph\n        The ground-truth tracking graph.\n    scale : tuple[float, ...] | None\n        Physical voxel scale used for centroid-distance matching.\n    max_distance : float\n        Maximum centroid distance for a match.\n\n    Returns\n    -------\n    DivisionScores\n        The per-division scores and the predicted forks classified as true\n        positives or false positives. False-positive forks include local\n        topology rejects, cross-GT-component branches, locally merged branches,\n        evaluable spurious forks, and valid candidates left unmatched by the\n        bipartite pairing.\n    """\n    matched = match_divisions(\n        pred_graph,\n        gt_graph,\n        scale,\n        max_distance,\n    )\n    gt_divisions = extract_divisions(gt_graph)\n    pred_div_nodes = {\n        node_id for node_id in pred_graph.node_ids()\n        if pred_graph.out_degree(node_id) >= 2\n    }\n    evaluable_forks, cross_component_forks, malformed_forks = (\n        _pred_division_fork_sets(pred_graph, gt_graph, scale, max_distance)\n    )\n    invalid_forks = cross_component_forks | malformed_forks\n\n    candidates: dict[int, set[int]] = {}\n    considered: set[int] = set()\n    for div_node, matched_pred in matched.items():\n        matched_nodes = _matched_division_nodes(_matched_node_attrs(matched_pred), gt_divisions[div_node], div_node)\n        if matched_nodes is None:\n            candidates[div_node] = set()\n            continue\n\n        parent_ids, daughter_ids = matched_nodes\n        local_nodes = parent_ids | {\n            successor for parent_id in parent_ids for successor in matched_pred.successors(parent_id)\n        }\n        local_forks = local_nodes & pred_div_nodes\n        considered |= local_forks\n        candidates[div_node] = {\n            pred_div\n            for pred_div in local_forks - invalid_forks\n            if _is_strongly_connected_division(matched_pred, pred_div, parent_ids, daughter_ids)\n        }\n\n    pairing = _bipartite_max_matching(list(candidates), candidates)\n    scores = {div: int(div in pairing) for div in candidates}\n    tp_forks = set(pairing.values())\n    # Use a set union so forks supported by multiple FP rules are counted once.\n    # Invalid forks were excluded from the pairing above and therefore cannot\n    # also be true positives.\n    fp_forks = (considered | evaluable_forks | invalid_forks) - tp_forks\n    return DivisionScores(scores=scores, tp_forks=tp_forks, fp_forks=fp_forks)\n\n\ndef _gt_weak_component_ids(graph: td.graph.BaseGraph) -> dict[int, int]:\n    """Map each GT node to its weakly connected component ID."""\n    component_ids: dict[int, int] = {}\n    for seed in graph.node_ids():\n        if seed in component_ids:\n            continue\n        component_ids[seed] = seed\n        stack = [seed]\n        while stack:\n            current = stack.pop()\n            for neighbor in graph.successors(current) + graph.predecessors(current):\n                if neighbor not in component_ids:\n                    component_ids[neighbor] = seed\n                    stack.append(neighbor)\n    return component_ids\n\n\ndef _branch_component_evidence(\n    graph: td.graph.BaseGraph,\n    pred_div: int,\n    child: int,\n    pred_to_gt: dict[int, int],\n    gt_component: dict[int, int],\n) -> tuple[int | None, bool]:\n    """Return one GT component for a predicted child branch.\n\n    Direct-child evidence takes precedence over grandchildren so downstream\n    errors do not invalidate a correctly matched division. Grandchildren are\n    fallback evidence only when the child is unmatched. The boolean marks a\n    locally merged branch that cannot be assigned uniquely to this fork.\n    """\n    if set(graph.predecessors(child)) != {pred_div}:\n        return None, True\n    if child in pred_to_gt:\n        return gt_component[pred_to_gt[child]], False\n\n    grandchildren = graph.successors(child)\n    if any(set(graph.predecessors(node)) != {child} for node in grandchildren):\n        return None, True\n\n    components = {\n        gt_component[pred_to_gt[node]]\n        for node in grandchildren\n        if node in pred_to_gt\n    }\n    if len(components) == 1:\n        return next(iter(components)), False\n    return None, False\n\n\ndef _pred_division_fork_sets(\n    pred_graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n    scale: tuple[float, ...] | None,\n    max_distance: float,\n) -> tuple[set[int], set[int], set[int]]:\n    """Return evaluable, cross-component, and malformed predicted forks.\n\n    Cross-component evidence must come from distinct direct-child branches.\n    A matched child identifies its branch; otherwise an unambiguous matched\n    grandchild may identify it. Merged local branches are malformed.\n    """\n    matched_pred = _match_full(pred_graph, gt_graph, scale, max_distance)\n    matched_attrs = _matched_node_attrs(matched_pred)\n    pred_to_gt = dict(\n        zip(\n            matched_attrs[td.DEFAULT_ATTR_KEYS.NODE_ID].to_list(),\n            matched_attrs[td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID].to_list(),\n            strict=True,\n        )\n    )\n\n    pred_forks = {\n        node_id for node_id in matched_pred.node_ids()\n        if matched_pred.out_degree(node_id) >= 2\n    }\n    evaluable_forks = {\n        pred_id for pred_id in pred_forks\n        if pred_id in pred_to_gt and gt_graph.out_degree(pred_to_gt[pred_id]) >= 1\n    }\n\n    gt_component = _gt_weak_component_ids(gt_graph)\n    cross_component_forks: set[int] = set()\n    malformed_forks: set[int] = set()\n    for pred_id in pred_forks:\n        branch_evidence: list[int] = []\n        for child in matched_pred.successors(pred_id):\n            component, malformed = _branch_component_evidence(\n                matched_pred, pred_id, child, pred_to_gt, gt_component\n            )\n            if malformed:\n                malformed_forks.add(pred_id)\n                break\n            if component is not None:\n                branch_evidence.append(component)\n        else:\n            if len(set(branch_evidence)) >= 2:\n                cross_component_forks.add(pred_id)\n\n    return evaluable_forks, cross_component_forks, malformed_forks\n\n\ndef count_matched_pred_divisions(\n    pred_graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n    scale: tuple[float, ...] | None = None,\n    max_distance: float = 7.0,\n) -> int:\n    """Count predicted division nodes whose matched GT node is annotated.\n\n    Matches the full predicted graph against the full GT graph.  Among\n    predicted nodes that were matched to a GT node, counts how many are\n    dividing (out-degree >= 2) in the prediction *and* whose matched GT\n    node has at least one child.  A matched GT node with no children marks\n    the end of the annotation — we can\'t tell whether the cell actually\n    divided there, so such predicted divisions are excluded from the count\n    (and therefore from the FP tally).\n\n    Parameters\n    ----------\n    pred_graph : td.graph.BaseGraph\n        The predicted tracking graph.\n    gt_graph : td.graph.BaseGraph\n        The ground-truth tracking graph.\n    scale : tuple[float, ...] | None\n        Physical voxel scale used for centroid-distance matching.\n    max_distance : float\n        Maximum centroid distance for a match.\n\n    Returns\n    -------\n    int\n        Number of matched predicted division nodes.\n    """\n    evaluable_forks, _, _ = _pred_division_fork_sets(\n        pred_graph, gt_graph, scale, max_distance\n    )\n    return len(evaluable_forks)\n\n\ndef evaluate_divisions(\n    pred_graph: td.graph.BaseGraph,\n    gt_graph: td.graph.BaseGraph,\n    scale: tuple[float, ...] | None = None,\n    max_distance: float = 7.0,\n) -> DivisionCounts:\n    """Compute TP, FN, and FP counts for division events.\n\n    - **TP**: GT divisions correctly recovered in the prediction\n      (matched nodes connected and forking).\n    - **FN**: GT divisions not recovered.\n    - **FP**: Spurious predicted divisions, including forks matched to an\n      annotated GT node, local-topology rejects, bipartite leftovers, and\n      forks whose distinct child branches have nearest matched evidence in\n      distinct GT components, and forks with locally merged branches. Fork IDs\n      are unioned, so a fork supported by multiple rules counts once.\n\n    Parameters\n    ----------\n    pred_graph : td.graph.BaseGraph\n        The predicted tracking graph.\n    gt_graph : td.graph.BaseGraph\n        The ground-truth tracking graph.\n    scale : tuple[float, ...] | None\n        Physical voxel scale used for centroid-distance matching.\n    max_distance : float\n        Maximum centroid distance for a match.\n\n    Returns\n    -------\n    DivisionCounts\n        Named tuple with ``tp``, ``fn``, and ``fp`` fields.\n    """\n    result = score_divisions(\n        pred_graph,\n        gt_graph,\n        scale,\n        max_distance,\n    )\n    tp = sum(result.scores.values())\n    fn = len(result.scores) - tp\n    return DivisionCounts(tp=tp, fn=fn, fp=len(result.fp_forks))\n', 'LICENSE': 'BSD 3-Clause License\n\nCopyright (c) 2026, Thibaut Goldsborough\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are met:\n\n1. Redistributions of source code must retain the above copyright notice, this\n   list of conditions and the following disclaimer.\n\n2. Redistributions in binary form must reproduce the above copyright notice,\n   this list of conditions and the following disclaimer in the documentation\n   and/or other materials provided with the distribution.\n\n3. Neither the name of the copyright holder nor the names of its\n   contributors may be used to endorse or promote products derived from\n   this software without specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"\nAND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE\nIMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE\nFOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\nDAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\nSERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\nCAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,\nOR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE\nOF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n'}
_official_file_hashes = {'__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'metrics.py': 'cfdd596e3f8909cca14db0682889738b19ff75c3808b3773175aba9367ca7444', 'division_metrics.py': '0635c38621a38f1eb4b55a302b4a817a88e9094930dfc2dab16faeeee60f4dc9', 'LICENSE': '3910a8b578783928cbbf981c1b204591ab3368e70017dc695d6536f71d272796'}
_official_package_dir = WORKING_DIR / "official_scorer_075fc5"
_official_package_dir.mkdir(parents=True, exist_ok=True)
for _name, _contents in _official_files.items():
    _data = _contents.encode("utf-8")
    if _official_hashlib.sha256(_data).hexdigest() != _official_file_hashes[_name]:
        raise RuntimeError("OFFICIAL_SCORER_PAYLOAD_HASH_MISMATCH: " + _name)
    _target = _official_package_dir / _name
    _target.write_bytes(_data)
    if _official_hashlib.sha256(_target.read_bytes()).hexdigest() != _official_file_hashes[_name]:
        raise RuntimeError("OFFICIAL_SCORER_DISK_HASH_MISMATCH: " + _name)
_official_package_name = "biohub_official_075fc5"
_official_spec = _official_importlib.spec_from_file_location(
    _official_package_name, _official_package_dir / "__init__.py",
    submodule_search_locations=[str(_official_package_dir)],
)
_official_package = _official_importlib.module_from_spec(_official_spec)
_official_sys.modules[_official_package_name] = _official_package
_official_spec.loader.exec_module(_official_package)
_official_metrics_spec = _official_importlib.spec_from_file_location(
    _official_package_name + ".metrics", _official_package_dir / "metrics.py"
)
OFFICIAL_METRICS = _official_importlib.module_from_spec(_official_metrics_spec)
_official_sys.modules[_official_package_name + ".metrics"] = OFFICIAL_METRICS
_official_metrics_spec.loader.exec_module(OFFICIAL_METRICS)

"""Install the frozen official scoring chain after the existing cell 8.

The caller injects OFFICIAL_METRICS (the unmodified 075fc5 module), then calls
install_official_selector(globals()). No model, Notebook top level, data download
or external write is executed by this module. The old proxy remains diagnostic.
"""


def install_official_selector(scope):
    import hashlib
    import json
    import math
    import numbers
    import warnings

    import polars as pl
    import tracksdata as td

    metrics = scope["OFFICIAL_METRICS"]
    if "legacy_score_sample" in scope or "legacy_aggregate_official" in scope:
        raise RuntimeError("Official selector already installed; refusing double wrapping")
    legacy_score = scope["score_sample"]
    legacy_aggregate = scope["aggregate_official"]
    scale = tuple(float(x) for x in scope["VOXEL_SCALE_UM"])
    radius = float(scope["VALIDATOR_MATCH_RADIUS_UM"])
    if len(scale) != 3 or any(not math.isfinite(x) or x <= 0 for x in scale):
        raise ValueError("Official selector requires finite positive ZYX voxel scale")
    if not math.isfinite(radius) or radius <= 0:
        raise ValueError("Official selector requires a finite positive matching radius")
    if float(scope["VALIDATOR_NODE_COUNT_PENALTY_A"]) != metrics.ADJUSTMENT_ALPHA:
        raise ValueError("Node adjustment coefficient differs from frozen official source")
    if float(scope["VALIDATOR_DIVISION_WEIGHT"]) != metrics.SCORE_DIVISION_WEIGHT:
        raise ValueError("Division coefficient differs from frozen official source")
    provenance = "official_075fc5:evaluate->per_sample_metrics->summarise"
    legacy_provenance = "unchanged_forge947_cell8_proxy; independent_global_node_matching"
    error_fields = (
        "missed_gt_nodes", "spurious_pred_nodes", "edges_recovered",
        "edges_fragmented", "edges_lost_to_detection", "wrong_association_edges",
    )

    def input_graph_sha256(nodes, edges):
        # The lists retain input node insertion and edge order because matching
        # ties can depend on that order. Values are plain graph coordinates in
        # voxel units; physical scale and n_total are recorded separately.
        canonical = {
            "nodes": [[int(node_id), int(value[0]), *(float(v) for v in value[1:])]
                      for node_id, value in nodes.items()],
            "edges": [[int(source), int(target)] for source, target in edges],
        }
        data = json.dumps(canonical, sort_keys=True, separators=(",", ":"),
                          ensure_ascii=False, allow_nan=False).encode("utf-8")
        return hashlib.sha256(data).hexdigest()

    def new_graph(nodes, edges):
        """Map arbitrary integral input IDs to fresh dense backend IDs in bulk.

        Preserve node insertion and edge order, duplicate edges, isolated nodes,
        and nonconsecutive edges: the official evaluator owns their treatment.
        The backend mutates edge dictionaries, so every dictionary is new.
        """
        graph = td.graph.InMemoryGraph()
        for key in ("z", "y", "x"):
            graph.add_node_attr_key(key, pl.Float64, 0.0)
        node_ids = list(nodes)
        attributes = []
        for node_id in node_ids:
            if isinstance(node_id, bool) or not isinstance(node_id, numbers.Integral):
                raise ValueError("Node IDs must be integers")
            position = nodes[node_id]
            if len(position) != 4:
                raise ValueError("Node values must be (integer_frame, z, y, x)")
            t, z, y, x = position
            if isinstance(t, bool) or not isinstance(t, numbers.Real):
                raise ValueError("Frame must be a numeric integer")
            if not math.isfinite(t) or int(t) != t or not 0 <= t < 2**31:
                raise ValueError("Frame must be a nonnegative Int32 integer")
            coords = tuple(float(v) for v in (z, y, x))
            if not all(math.isfinite(v) for v in coords):
                raise ValueError("Nonfinite spatial coordinate")
            attributes.append(dict(t=int(t), z=coords[0], y=coords[1], x=coords[2]))
        internal_ids = graph.bulk_add_nodes(attributes)
        remap = dict(zip(node_ids, internal_ids, strict=True))
        edge_attributes = []
        for source, target in edges:
            if source not in remap or target not in remap:
                raise ValueError("Dangling edge in scorer input")
            edge_attributes.append(dict(source_id=remap[source], target_id=remap[target]))
        graph.bulk_add_edges(edge_attributes)
        return graph

    def score_sample(pred_nodes_plain, pred_edges_plain, gt_nodes_plain, gt_edges_plain, t_true):
        if t_true is None or isinstance(t_true, bool):
            raise ValueError("Missing or invalid estimated_number_of_nodes; no proxy fallback")
        n_total = float(t_true)
        if not math.isfinite(n_total) or n_total <= 0:
            raise ValueError("estimated_number_of_nodes must be finite and positive")
        if not gt_nodes_plain:
            raise ValueError("Empty GT is unsupported by the frozen official matching chain")
        # Copies isolate official in-place matching from cached input and proxy.
        pred_edges = list(pred_edges_plain)
        gt_edges = list(gt_edges_plain)
        pred = new_graph(pred_nodes_plain, pred_edges)
        gt = new_graph(gt_nodes_plain, gt_edges)
        pred_input_sha256 = input_graph_sha256(pred_nodes_plain, pred_edges)
        gt_input_sha256 = input_graph_sha256(gt_nodes_plain, gt_edges)
        er = metrics.evaluate(pred, gt, scale=scale, max_distance=radius)

        # Read-only attribution from the exact official matcher, preserving backend order.
        scope['F1_EDGE_STATUS'] = {}
        ordered_ids = list(pred_nodes_plain)
        id_back = dict(zip(list(pred.node_ids()), ordered_ids, strict=True))
        if pred.num_edges():
            for item in metrics._evaluate_matched_graph(pred,gt).iter_rows(named=True):
                pair = (id_back[int(item[td.DEFAULT_ATTR_KEYS.EDGE_SOURCE])],id_back[int(item[td.DEFAULT_ATTR_KEYS.EDGE_TARGET])])
                scope['F1_EDGE_STATUS'][pair] = 'TP' if item[td.DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK] else ('FP' if item['pred_valid'] else 'UNKNOWN')

        empty_edge_matching = pred.num_edges() == 0
        if empty_edge_matching:
            # evaluate() exits its edge matching early for an edgeless prediction.
            # Its division evaluation matches copies only; node_recall() would
            # otherwise raise KeyError for absent MATCHED_NODE_ID. Apply the same
            # official matcher to this temporary graph solely to obtain recall.
            from tracksdata.metrics import DistanceMatching
            from tracksdata.options import get_options, set_options
            from scipy.sparse import SparseEfficiencyWarning

            progress = get_options().show_progress
            set_options(show_progress=False)
            try:
                with warnings.catch_warnings():
                    warnings.filterwarnings("ignore", category=SparseEfficiencyWarning)
                    pred.match(gt, matching=DistanceMatching(scale=scale, max_distance=radius))
            finally:
                set_options(show_progress=progress)
        recall = metrics.node_recall(pred, gt)
        official_row = metrics.per_sample_metrics(er, n_total=n_total, node_recall=recall)
        legacy_row = legacy_score(pred_nodes_plain, pred_edges, gt_nodes_plain, gt_edges, n_total)
        division_total = er.division_tp + er.division_fp + er.division_fn
        row = dict(official_row)
        row.update({
            "adjusted_edge_jaccard": official_row["adj_edge_jaccard"],
            "div_tp": er.division_tp, "div_fp": er.division_fp, "div_fn": er.division_fn,
            "div_jaccard": er.division_tp / division_total if division_total else float("nan"),
            "t_pred": er.num_pred_nodes, "t_true": n_total,
            "weight": er.edge_tp + er.edge_fp + er.edge_fn,
            "metric_provenance": provenance,
            "error_decomposition_provenance": legacy_provenance,
            "official_edgeless_recall_matching": empty_edge_matching,
            "input_pred_graph_sha256": pred_input_sha256,
            "input_gt_graph_sha256": gt_input_sha256,
            "input_graph_hash_schema": "ordered_plain_graph_v1:node_id,t,z,y,x;ordered_source,target;canonical_json_utf8",
            "input_graph_provenance": "same_plain_graph_received_by_legacy_proxy_and_official_adapter",
            "input_n_total": n_total,
            "input_scale_zyx_um": scale,
            "input_match_radius_um": radius,
        })
        # Keep names used by cell9/10 diagnostics, but explicitly retain their
        # legacy provenance; these six fields never enter the official score.
        row.update({key: legacy_row[key] for key in error_fields})
        row.update({"legacy_proxy_" + key: value for key, value in legacy_row.items()})
        return row

    def aggregate_official(sample_rows):
        rows = list(sample_rows)
        if not rows:
            raise ValueError("No validator rows; refusing undefined official selector score")
        for row in rows:
            if row.get("metric_provenance") != provenance:
                raise ValueError("Selector received a row without official scorer provenance")
            if not math.isfinite(float(row["t_true"])) or row["t_true"] <= 0:
                raise ValueError("Invalid node count metadata in official aggregation")
        official = metrics.summarise(rows)
        # Official summarise may skip NaN rows. Preserve and expose those counts,
        # but a nonfinite final score may never participate in Python sorting.
        if not math.isfinite(official["score"]) or not math.isfinite(official["adj_edge_jaccard"]):
            raise ValueError("Official aggregate score is nonfinite; no proxy fallback")
        legacy_rows = [
            {key.removeprefix("legacy_proxy_"): value for key, value in row.items()
             if key.startswith("legacy_proxy_")}
            for row in rows
        ]
        legacy = legacy_aggregate(legacy_rows)
        summary = dict(official)
        summary.update({
            "adjusted_edge_jaccard": official["adj_edge_jaccard"],
            # Historical key required by the unchanged selector in cell10.
            "proxy_score": official["score"],
            "official_score": official["score"],
            "proxy_score_field_semantics": "official_score_alias_for_unchanged_cell10",
            "div_tp": official["division_tp"], "div_fp": official["division_fp"],
            "div_fn": official["division_fn"],
            "metric_provenance": provenance,
            "error_decomposition_provenance": legacy_provenance,
            "official_rows_skipped": len(rows) - official["n"],
            "official_adjusted_rows_used": official["n_adj"],
            "legacy_proxy_score": legacy["proxy_score"],
        })
        summary.update({key: legacy[key] for key in error_fields})
        summary.update({"legacy_proxy_" + key: value for key, value in legacy.items()
                        if key != "proxy_score"})
        return summary

    scope["legacy_score_sample"] = legacy_score
    scope["legacy_aggregate_official"] = legacy_aggregate
    scope["score_sample"] = score_sample
    scope["aggregate_official"] = aggregate_official
    scope["OFFICIAL_SELECTOR_PROVENANCE"] = provenance
    return {"metric_provenance": provenance, "legacy_proxy_preserved": True,
            "graph_builder": "fresh_InMemoryGraph_bulk_add_nodes_bulk_add_edges",
            "nonfinite_final_score": "FAIL_CLOSED", "missing_n_total": "FAIL_CLOSED"}

install_official_selector(globals())

print("TARGET950: selector uses fixed official score; legacy proxy fields are diagnostic only.")


In [ ]:
FIXED_STEMS=['44b6_12dfb391', '44b6_267148e4', '44b6_2a2eff9f', '44b6_341df25f', '6bba_062c8d37', '6bba_07e24132', '6bba_085bf656', '6bba_09961292']
FROZEN_CONFIG={'SAFE_DIV_MAX_UM': 9.0, 'SAFE_DIV_SISTER_MAX_UM': 14.0, 'SAFE_DIV_DIVERGE_UM': 2.25, 'SAFE_DIV_SISTER_SYMMETRY_TAU': 0.6, 'SAFE_DIV_EXISTING_CHILD_MAX_UM': 10.0, 'SAFE_DIV_FRAME_FRAC_CAP': 0.0076, 'SAFE_DIV_GLOBAL_FRAC_CAP': 0.00375, 'DEEPCENTER_SAFE_DIV_THRESHOLD': 0.2, 'DEEPCENTER_GAP_THRESHOLD': 0.25, 'GAP_CLOSE_UM': 5.0, 'OUTPUT_MIN_TRACK_LEN': 6, 'SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB': 0.88, 'MOTION_RELINK_TIGHT_UM': 5.5, 'MOTION_RELINK_RELAXED_UM': 10.0, 'GAP2_MAX_STEP_UM': 4.4, 'GAP2_MAX_TOTAL_UM': 10.2, 'MOTION_RELINK_LEARNED_BONUS': 1.0, 'MOTION_RELINK_VELOCITY_WEIGHT': 0.5, 'GAP_CLOSE_REUSE_UM': 3.2, 'OUTPUT_EDGE_MAX_UM': 14.0}
RAW_MANIFEST=[{'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/edges/ids/c/0/0', 'sha256': 'ada1fc02f6b8591288792c7cb21181aadf4873bb4c58e2edb480239ac32e7698'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/edges/ids/zarr.json', 'sha256': '502700e907eff3e19f67f74eb55c351db85651ab6475a36ff7384ba7d8abe7f2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/edges/props/edge_dist/values/c/0', 'sha256': '0871bd0a451dac2630b7db7c9ff994b8f6955dc2142321e0bea58cf0f3c9d60b'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/edges/props/edge_dist/values/zarr.json', 'sha256': 'b8cdaf957766396023c0b0db96a96d760756ebe893fad38174e06fbb55b4a46d'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/edges/props/edge_dist/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/edges/props/edge_prob/values/c/0', 'sha256': '878cd8e82f406fbdd56e55ba018e22f1d121a794fc5ae3ca8e80e5bd874e2820'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/edges/props/edge_prob/values/zarr.json', 'sha256': 'b8cdaf957766396023c0b0db96a96d760756ebe893fad38174e06fbb55b4a46d'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/edges/props/edge_prob/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/edges/props/solution/values/c/0', 'sha256': '9ca276f6b835ff69f897e823f124dd1bdffa86d0f9d7e518beebd028d87a5f16'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/edges/props/solution/values/zarr.json', 'sha256': '9e7aa1a7f51d399d9f710691c5b056ebe7d818d2533026452df181bed7430107'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/edges/props/solution/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/edges/props/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/edges/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/ids/c/0', 'sha256': '0c4b45143558c7a8a83e8ede163ba4287079d5ea80c12daf16e536f53198c242'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/ids/zarr.json', 'sha256': '45fda531e5905a715f39aba2b18001ffa81ea383a2e5d372b5c9796647573fa7'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/props/solution/values/c/0', 'sha256': '586852e6d4b67bdb43ffc72777522c02a9278839ac61677a93d5ac04979d9313'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/props/solution/values/zarr.json', 'sha256': 'cd0903d3612c3a843d5d6ee0ff97aabc315defc4405bc462c0b3516bcba87070'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/props/solution/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/props/t/values/c/0', 'sha256': '0382bea8c7a2dc5c0c850fffa87a7911072bdf65c0f07d400eeca912e6be2cae'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/props/t/values/zarr.json', 'sha256': '3ac8683ecbd4fc285a6be305891830a4057e17e3be01f9793c5c476dfa99c716'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/props/t/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/props/x/values/c/0', 'sha256': '882b6357a45b1f3ad1b3b02241f074ee2df1d6c65ac837ff5d5d5f1020d57b32'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/props/x/values/zarr.json', 'sha256': 'ae2050f97875e2d35a46f9500a63f9183837667e084eef1edcb968f49a19245e'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/props/x/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/props/y/values/c/0', 'sha256': '5fda7a2b4152c42e9d3be66a65c0d145c408d1fd79e3293fb567b7d30977f522'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/props/y/values/zarr.json', 'sha256': 'ae2050f97875e2d35a46f9500a63f9183837667e084eef1edcb968f49a19245e'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/props/y/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/props/z/values/c/0', 'sha256': '5ddcdbf1d81bae1caa78544f15ecb5bd0700333ad0d1ab61b217eb286c1a783c'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/props/z/values/zarr.json', 'sha256': 'ae2050f97875e2d35a46f9500a63f9183837667e084eef1edcb968f49a19245e'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/props/z/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/props/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/nodes/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_12dfb391.geff/zarr.json', 'sha256': '19fd4169dd15083cf59c68513a508014ecbd663087adfe784e36ba9f9ef76d87'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/edges/ids/c/0/0', 'sha256': 'ceb70ad6705cb01c2724dc5e2ea35f6487043a556255aecc29e0a32905ed4231'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/edges/ids/zarr.json', 'sha256': '7eeb80c9ea2bb7e4d97bcc98bd97b9abc900b24e5cdde13b6ede6d49358aca9a'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/edges/props/edge_dist/values/c/0', 'sha256': 'f626866a03627fa7986162475270539a8d0cf55e09662d0f198643b177e30bc0'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/edges/props/edge_dist/values/zarr.json', 'sha256': '5a081cc1c212d4bfb68c5cedc9f9be5281ca82d9a694c7f3aef27d96fe3730fc'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/edges/props/edge_dist/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/edges/props/edge_prob/values/c/0', 'sha256': 'be0b6eb6117f4ca68f4d43fd39544c8f895e6124742c28ee88c19d6acf152df6'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/edges/props/edge_prob/values/zarr.json', 'sha256': '5a081cc1c212d4bfb68c5cedc9f9be5281ca82d9a694c7f3aef27d96fe3730fc'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/edges/props/edge_prob/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/edges/props/solution/values/c/0', 'sha256': '33cdc3a5f3413ce323bcfcdb745b77975ece278e73ba1df94eb3f98e87f6aa11'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/edges/props/solution/values/zarr.json', 'sha256': 'bf87e4ee10e4caed71e5053848bbfb70021f6911f2367f14f2e5709c67f44e12'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/edges/props/solution/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/edges/props/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/edges/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/ids/c/0', 'sha256': 'a6b283dc030c0b5ee3a02cef7dbba1b1d2bcb84aa8bafb94e2611a1bf2b486c3'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/ids/zarr.json', 'sha256': 'c7da900387e8e9129fa7592e49bb66bf4c78d9717093c91d1e9d823a8b2773e2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/props/solution/values/c/0', 'sha256': 'ca2a62e37370329a3362c3e367058aa1d68935e928f2d02f909fd9296cad397c'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/props/solution/values/zarr.json', 'sha256': 'eecca7f70095827427cead320d2f1526eb8737341e99529b7b2d42fa683e4e61'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/props/solution/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/props/t/values/c/0', 'sha256': '256f57b12e2e4ef201b44417e72c0635b1d48f2437af5e6c3f1035f90f197566'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/props/t/values/zarr.json', 'sha256': 'd329d5af20e180bc5a4c8d09151a3d2afa4185737f71a954de37179712d32bdd'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/props/t/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/props/x/values/c/0', 'sha256': '6f0c16c4feb32a3e03e84874d0dd3fae9c3b07f6a1c9ca3ff75b3910513cbdf6'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/props/x/values/zarr.json', 'sha256': '86653cb8de4d6cfbd10a131ceff15812e969e423b666c67f831a27a0473de9a5'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/props/x/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/props/y/values/c/0', 'sha256': '6f4656b87eff9aa85f342b04f9d5ce2871c5abc0d07d620e31f2af9b5d1c8e63'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/props/y/values/zarr.json', 'sha256': '86653cb8de4d6cfbd10a131ceff15812e969e423b666c67f831a27a0473de9a5'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/props/y/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/props/z/values/c/0', 'sha256': 'da4d444c81f0fbf141fc2c667e6000064f09fcdfc6c5b19df38f380079dfb06b'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/props/z/values/zarr.json', 'sha256': '86653cb8de4d6cfbd10a131ceff15812e969e423b666c67f831a27a0473de9a5'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/props/z/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/props/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/nodes/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_267148e4.geff/zarr.json', 'sha256': '0cc4449fb7e0ec717bef574ab1a8571310c825337f6ebd5beeb956d1ca525234'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/edges/ids/c/0/0', 'sha256': '4aa294b1c88d3f94aa677c7a7bd640bfc4458494ad047444482f6ce241a097e6'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/edges/ids/zarr.json', 'sha256': 'a806a68d1f350b8ed2c6a1973ab5a7b0b03dd3f3e8a24c3e23122911d59e12d3'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/edges/props/edge_dist/values/c/0', 'sha256': '1b3cf6551dacb55d47bcba4cb16a02934bc6496024b3ed048c82ce56c6f06e46'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/edges/props/edge_dist/values/zarr.json', 'sha256': '029a8323e854bc1da8a600edc305fb14dccee68b84a7a18c9f5baaaf8c97defe'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/edges/props/edge_dist/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/edges/props/edge_prob/values/c/0', 'sha256': 'd04a36033f1187f25224ff368ca24530ee9034904f68d17ea1d298cfab32a958'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/edges/props/edge_prob/values/zarr.json', 'sha256': '029a8323e854bc1da8a600edc305fb14dccee68b84a7a18c9f5baaaf8c97defe'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/edges/props/edge_prob/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/edges/props/solution/values/c/0', 'sha256': '43215dfd42499a64eb3d1b21ef44de3bda9d5914bbcbcde32f2e953ebafcf0f2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/edges/props/solution/values/zarr.json', 'sha256': 'ecd608ec60b87f8562772ca7b068c097553496c416f66a6cc41369a2cdfaa117'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/edges/props/solution/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/edges/props/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/edges/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/ids/c/0', 'sha256': '1ecf79ac504b3a63a3cac9a3c893571c7f13a84641ed6b000b5ab3594a0752ba'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/ids/zarr.json', 'sha256': 'efa9ee36892080edfcc0fd19d2831c661cc6e23e39653757dad8e97e7066ae4f'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/props/solution/values/c/0', 'sha256': 'ef34f361e49d925beb070bdb281f33a5d2b98e29a34ee155153fa61502f0e857'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/props/solution/values/zarr.json', 'sha256': 'c365a809b587e57ca254a2bc9bc916e40db6ba8b5e14b597103778d2c5490898'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/props/solution/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/props/t/values/c/0', 'sha256': '10c02e5f53f7849459394a6314fa75a628e54b620d0ee39bbc6ff0c7087f9a0b'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/props/t/values/zarr.json', 'sha256': 'a3b37327b5a9d415d3fd11ce8b29295b15f7b5f51b5fd3169e275d717809f763'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/props/t/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/props/x/values/c/0', 'sha256': 'e6122d4e1552414c3df9131cf59c691520286ae92ae5226dafd0db82f71a7520'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/props/x/values/zarr.json', 'sha256': '6f4d20a42611b3e73d5559b376335db78c99bed492930276379090aa9c5f1f23'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/props/x/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/props/y/values/c/0', 'sha256': 'b4aee7621eba5634f358a12baa54847b06d7f73d1ad9063c801eeac780403f21'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/props/y/values/zarr.json', 'sha256': '6f4d20a42611b3e73d5559b376335db78c99bed492930276379090aa9c5f1f23'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/props/y/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/props/z/values/c/0', 'sha256': '98936f0828a155e2cea12866ae2a0b4e2173660e75fb7c62eb1bd5b927e8e4fb'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/props/z/values/zarr.json', 'sha256': '6f4d20a42611b3e73d5559b376335db78c99bed492930276379090aa9c5f1f23'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/props/z/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/props/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/nodes/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_2a2eff9f.geff/zarr.json', 'sha256': '19fd4169dd15083cf59c68513a508014ecbd663087adfe784e36ba9f9ef76d87'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/edges/ids/c/0/0', 'sha256': 'a5b1757251987e45234c16440f6e7c90da4714efa3e0e10e4db7bc58f30c27b1'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/edges/ids/zarr.json', 'sha256': '6623dd68cc7c49504eb33fb2a9ce28450847cb0b4cf4246241ff35730a38baeb'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/edges/props/edge_dist/values/c/0', 'sha256': 'd1bd8f79d1dc26bc22cca336429ccee1e66096a654fc9e8d12c1a7f23b40b486'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/edges/props/edge_dist/values/zarr.json', 'sha256': '16888ec99f2d753d3132cc8ec8cafac2015e3a1ef9f1176e47d041b20973af9d'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/edges/props/edge_dist/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/edges/props/edge_prob/values/c/0', 'sha256': 'b8cc52d1049d9a80cc7a98d70a9c15c05368fb08fd8c3d9d0a0fd630feb06434'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/edges/props/edge_prob/values/zarr.json', 'sha256': '16888ec99f2d753d3132cc8ec8cafac2015e3a1ef9f1176e47d041b20973af9d'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/edges/props/edge_prob/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/edges/props/solution/values/c/0', 'sha256': '48346315725144b648919a2ed8c1dbef35176f5d094f9d17cb57992bed7d17a7'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/edges/props/solution/values/zarr.json', 'sha256': 'e421c206bc8323121e7ebdada832e1cda1c5203d8b3dae4acfd20f79409110f0'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/edges/props/solution/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/edges/props/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/edges/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/ids/c/0', 'sha256': '65fcd59dfd54e36a0a25cdb0a410e4bf693580537d44016b30d1b63e358a359e'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/ids/zarr.json', 'sha256': 'e9940c188b08a1a21ce21da116f60b253129cd3a0240663b2f2aaf00187f1a6a'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/props/solution/values/c/0', 'sha256': '0fb6f4ae923e646d7f385211151196b131af3f07a18f35834dbfcb561e0a88d4'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/props/solution/values/zarr.json', 'sha256': 'a1f47a76e8da8495eee0a08ca56f5c53cfccb8eaafff676edde6bcf2be41a020'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/props/solution/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/props/t/values/c/0', 'sha256': '1e4b580b5bf875e03389c9d64ba24cf3ea61b788ab2bcdc07c28fef9cc213ff4'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/props/t/values/zarr.json', 'sha256': 'c234c136972b2d22b73b77f0598a364c2dd99ed32d94e4997de712d3e683733f'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/props/t/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/props/x/values/c/0', 'sha256': '8a9c12bba5827b994191800d8103ba5a0aadf6c683f2d812d6a65935af41592d'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/props/x/values/zarr.json', 'sha256': '7b394252681ac166679b4b7129cb25986781ca57728d9125745dffbda7da4331'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/props/x/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/props/y/values/c/0', 'sha256': '1e1ce75aebdc79c0295bffeacec35fffa2601f880af6a91d9007d55dcbfe6af9'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/props/y/values/zarr.json', 'sha256': '7b394252681ac166679b4b7129cb25986781ca57728d9125745dffbda7da4331'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/props/y/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/props/z/values/c/0', 'sha256': 'f112c5d96de40094317e280f49beb86eab875d5f356121c751dad3a60adcb0ad'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/props/z/values/zarr.json', 'sha256': '7b394252681ac166679b4b7129cb25986781ca57728d9125745dffbda7da4331'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/props/z/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/props/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/nodes/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/44b6_341df25f.geff/zarr.json', 'sha256': 'f0d0296482138552abe663ead76d41795973e8359cd82f53aa483e0a5a5a5022'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/edges/ids/c/0/0', 'sha256': '8f2c5b3f8eefe41c3143e59924f6ebff4c39ad72435cb5b54f75f59274488e9e'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/edges/ids/zarr.json', 'sha256': '11cdafda87213dca774cfd19864963ccf0e4ba188035f8d9517c36dbf8a6c1c7'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/edges/props/edge_dist/values/c/0', 'sha256': '24e24905ada3bfaf7fd33d85f55b94083836fdca7b8c6e258e409ee57f7b5480'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/edges/props/edge_dist/values/zarr.json', 'sha256': '26ca8f63cbb35183eafa69b8b4f40ec156db46a0edf67b4048fab6441c281471'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/edges/props/edge_dist/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/edges/props/edge_prob/values/c/0', 'sha256': 'dd69ec92bf419f85a08de4fe8670393b64893a0d8028a5834cafdb9acf89e441'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/edges/props/edge_prob/values/zarr.json', 'sha256': '26ca8f63cbb35183eafa69b8b4f40ec156db46a0edf67b4048fab6441c281471'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/edges/props/edge_prob/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/edges/props/solution/values/c/0', 'sha256': '97c1d13ae8558b4105a0d55f4a80e490124b2cc6e1f823a5d348a373ac2cd4db'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/edges/props/solution/values/zarr.json', 'sha256': '70c83c29b6986d79b93c59bcc7cc050d0db1803c826980823ce65421716ee6fb'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/edges/props/solution/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/edges/props/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/edges/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/ids/c/0', 'sha256': '4c5ac8193ff57c7de9543381153073179841f832613cf3bcec5a515d0ad45a48'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/ids/zarr.json', 'sha256': '039397ce2ab51de6fc727b68a9adc4c034815bb913e5cc07e6c7d8897662ec2d'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/props/solution/values/c/0', 'sha256': 'e79815efab9a0bc53edbc854f7e4e54553e3b0b593c86f279a934ec282c4e21c'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/props/solution/values/zarr.json', 'sha256': '1bd85f0299f54ca0c831661cd3588667a15d8dce3ac1e13f3909b430a6ec5dcb'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/props/solution/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/props/t/values/c/0', 'sha256': 'ce6908d5e1fbb898ae2b5e9118268e68e51930f319a06619aeebfcf03abe048a'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/props/t/values/zarr.json', 'sha256': '4b06b3bb3c82735c9568800e09caf3322d53c8207f09b04ca19739fb09810b58'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/props/t/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/props/x/values/c/0', 'sha256': '704fd9bfe43391184e41c6f4cb5a10e04d58f02787a8003a65098973af152961'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/props/x/values/zarr.json', 'sha256': '053d920d795fdf247a0177c6e88451693dc93d244948997afaaa8b21498c420e'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/props/x/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/props/y/values/c/0', 'sha256': '19225f079acdf19c1acc5ce4163cf982823f8e92447dea6693daa1a138214577'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/props/y/values/zarr.json', 'sha256': '053d920d795fdf247a0177c6e88451693dc93d244948997afaaa8b21498c420e'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/props/y/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/props/z/values/c/0', 'sha256': 'cbfa9b4a487e064d004cb69817641b6b0637c387bf39c7a74888c60ef209bed3'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/props/z/values/zarr.json', 'sha256': '053d920d795fdf247a0177c6e88451693dc93d244948997afaaa8b21498c420e'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/props/z/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/props/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/nodes/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_062c8d37.geff/zarr.json', 'sha256': 'cd82e8b818a4ef63b682abbc43bb765a34a905fd222e4fe8b819d46b9f6c6361'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/edges/ids/c/0/0', 'sha256': '8922e9a31b084f554ced3958d7f0162f7a82e0a011df33b42cd030d1e9c3f303'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/edges/ids/zarr.json', 'sha256': 'cca5ac620bbebd9e1d2da988cc25d4e17cb35891a5a947f5a8081b229b353618'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/edges/props/edge_dist/values/c/0', 'sha256': 'dfa6fb1331235a53a5f5b28aef74f7e8ecc2e65aad75a3355fff2a4e5f1371d5'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/edges/props/edge_dist/values/zarr.json', 'sha256': '8a67898676baa60fd90769348a73ae6deddab5c5766c908599abbaf457c3c6f5'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/edges/props/edge_dist/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/edges/props/edge_prob/values/c/0', 'sha256': 'e428c9cb00fe0f5cfea8d84e7cf8dc1b62944bc1059b1009a92e70ea7bdb4398'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/edges/props/edge_prob/values/zarr.json', 'sha256': '8a67898676baa60fd90769348a73ae6deddab5c5766c908599abbaf457c3c6f5'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/edges/props/edge_prob/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/edges/props/solution/values/c/0', 'sha256': 'a51ee44344cd97af8a11790c90b4c64b1a6f45584f99ff333b1df68126979231'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/edges/props/solution/values/zarr.json', 'sha256': '4e26f052cde87a0d1b8cd73a2f50f43854fdeecd1f77cc7802e94e62b1f400fc'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/edges/props/solution/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/edges/props/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/edges/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/ids/c/0', 'sha256': '80de288dfe26f8bdd755545039c0bbb10fe22b7ef5fc2bac457b6400bddf4ee1'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/ids/zarr.json', 'sha256': '5deb1c1960c2ebb80a1f7efc9db560c3a8bda6088f93387c6643833ccdc94241'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/props/solution/values/c/0', 'sha256': '8c54fee9e478b6dd4c47f4909cb450fa2393f2eed0d2f2faf936febd114a0309'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/props/solution/values/zarr.json', 'sha256': '77c44814c8c768c81e835abce2b72b95af5fc1f9d7e59bb93fad6d8490dfb9f0'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/props/solution/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/props/t/values/c/0', 'sha256': 'c8ec5dbb719d65f197092a88a0f530ea85d167e4c78560eb943bd48d1bfa44dc'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/props/t/values/zarr.json', 'sha256': 'e67a23eeae9f2fc35e1b7978249d9f11ca450c2b4ed60e38208ca268fab47e38'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/props/t/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/props/x/values/c/0', 'sha256': '27849dd7ae60dc6e8bc175b8b2e0f17b3c1545141dce4686a7212d49e720ba92'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/props/x/values/zarr.json', 'sha256': '0bcb508a1ee1dbd13554cc74078ae32733d556e1a02c69e7372afa03e3330c89'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/props/x/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/props/y/values/c/0', 'sha256': '757f0314aec09db144d886ba95316b109399c16e95c149f267ee4bc2223eb874'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/props/y/values/zarr.json', 'sha256': '0bcb508a1ee1dbd13554cc74078ae32733d556e1a02c69e7372afa03e3330c89'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/props/y/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/props/z/values/c/0', 'sha256': '78a1b907bbdf87079d1adcec0687dbdedc7643049a74f9d10cd2b2b4c24d7f32'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/props/z/values/zarr.json', 'sha256': '0bcb508a1ee1dbd13554cc74078ae32733d556e1a02c69e7372afa03e3330c89'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/props/z/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/props/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/nodes/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_07e24132.geff/zarr.json', 'sha256': 'b5fa3bd0fc032bd927fbc5c3e97787a1c29c698eba296f1e0f9e5d97e3c13c54'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/edges/ids/c/0/0', 'sha256': '07f9314945dd41a2229672e125bec5c702eabd55c28e1aad42ef021a1ca9129c'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/edges/ids/zarr.json', 'sha256': '8793e2ac95ad9079c0c1bb02bde2b65563e468ca592741e034d32b0ba834c4a8'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/edges/props/edge_dist/values/c/0', 'sha256': '64fc4f7eca7856c93d7b9b75906aefa0283ea689120fd05bafaee970450751ca'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/edges/props/edge_dist/values/zarr.json', 'sha256': '6097d6acc9836b663dda950d89200defece5172645b44375a4b940c5bd1e046e'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/edges/props/edge_dist/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/edges/props/edge_prob/values/c/0', 'sha256': '9870ae14dd8235946ea009e2299dd5df685d11a758df3e0dba636cff60dbaf72'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/edges/props/edge_prob/values/zarr.json', 'sha256': '6097d6acc9836b663dda950d89200defece5172645b44375a4b940c5bd1e046e'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/edges/props/edge_prob/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/edges/props/solution/values/c/0', 'sha256': '9e5be282bd9a2ac041cea97255a10d3bf8863264f39ac95ba735e91e7e87de39'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/edges/props/solution/values/zarr.json', 'sha256': '8a9884d62ac6cdbe32061d6a3f789c8c87c5bd5d1cfc1db2413f378a59ff8d38'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/edges/props/solution/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/edges/props/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/edges/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/ids/c/0', 'sha256': 'b0db694d856473b65bd33bf046f58652113303bce4eeec0c7ff270f7fcf2ccdb'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/ids/zarr.json', 'sha256': '521ecac5d2238ed9c6cb45d74a705ed8d278e6f2f00055d2a946734570357e98'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/props/solution/values/c/0', 'sha256': '3d0507ceb81c9477127d7ee790417d35bfaca0a8262a65bbe4ae6543bcdf164d'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/props/solution/values/zarr.json', 'sha256': '170717d92e76981eab5478e004ff1eddae10b351ad6b529ae2800645c75a94e3'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/props/solution/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/props/t/values/c/0', 'sha256': '88ec52621712b4ab6daa0f948b2339cd6ba920028434b284f638cf34575dd110'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/props/t/values/zarr.json', 'sha256': '07413590ea1318e0c263fa716bf291d06a08d0361c9560397efeae94c3e9c0a9'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/props/t/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/props/x/values/c/0', 'sha256': 'f8fc3278fdfe6aef5a006b882a0ccbaed0d0c3cf5cf40a5c7473b646e86f1253'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/props/x/values/zarr.json', 'sha256': 'd925f5f34c8a66552257b6030f16a3a944cbfa4e05745a19f54b90caf8b77dff'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/props/x/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/props/y/values/c/0', 'sha256': '94de7b06da00fc395101c629217dd581a06ff354ba3f9e53e4b9d909348cc9c6'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/props/y/values/zarr.json', 'sha256': 'd925f5f34c8a66552257b6030f16a3a944cbfa4e05745a19f54b90caf8b77dff'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/props/y/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/props/z/values/c/0', 'sha256': '5f65666287185f2805aae07844e005d9d870fcf27b3bd2f817d7cf06fb247e4c'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/props/z/values/zarr.json', 'sha256': 'd925f5f34c8a66552257b6030f16a3a944cbfa4e05745a19f54b90caf8b77dff'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/props/z/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/props/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/nodes/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_085bf656.geff/zarr.json', 'sha256': '19fd4169dd15083cf59c68513a508014ecbd663087adfe784e36ba9f9ef76d87'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/edges/ids/c/0/0', 'sha256': '4dfdcab0a0278131ba62f901e9339792e7e1fc20a22eec26c94949df2240aa00'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/edges/ids/zarr.json', 'sha256': '205a82434d0718fc6fdc6a3f86702052d79deccd16eafc21186e9793a4bc3bf6'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/edges/props/edge_dist/values/c/0', 'sha256': '96e2c3ec32663eaa4d15bb2e2716ff363f607a463d04609ac71f8b460726aa90'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/edges/props/edge_dist/values/zarr.json', 'sha256': 'bc0ec196108e209bcf94286e622ec23a0117ae9f25eaede6f9565cf2b0a42c16'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/edges/props/edge_dist/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/edges/props/edge_prob/values/c/0', 'sha256': '29f1272e96e0cfc3a2ef8f9e176c64499051f5fca3e0b44f35be135b057a6568'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/edges/props/edge_prob/values/zarr.json', 'sha256': 'bc0ec196108e209bcf94286e622ec23a0117ae9f25eaede6f9565cf2b0a42c16'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/edges/props/edge_prob/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/edges/props/solution/values/c/0', 'sha256': 'a5339ff6995ad175f539827c896be0350a7d38a7953f211c9afaaeb7e62beb39'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/edges/props/solution/values/zarr.json', 'sha256': 'b1431f16f3c9162338ea6c7f1dceb7dea2d0a397308f834b8637b961e429cfb8'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/edges/props/solution/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/edges/props/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/edges/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/ids/c/0', 'sha256': '6adfcc38813027929ae641200f2ecc56169e9df4cb61f07cf5cb455f47e94d8b'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/ids/zarr.json', 'sha256': 'c00bfe12a30ca4c2a97f568fc082360a911e8c755c1d1d552dc9523705f61f6b'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/props/solution/values/c/0', 'sha256': '6218ea92289babb63244db05d9829c7b74511b610152ea5b48f1ee52ad729df9'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/props/solution/values/zarr.json', 'sha256': 'd53bfe1e9ab95ee54bf8d7447b787d451620a804f9a3f629339e337c5baff821'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/props/solution/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/props/t/values/c/0', 'sha256': '1e5425874099bff0d927c5de787a837c14bc1c35b41f1fe63707962bcf828b28'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/props/t/values/zarr.json', 'sha256': 'c89ffd65c8fa7516c602982441d6725746ed5b633e060f22b6f4093ab61d732d'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/props/t/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/props/x/values/c/0', 'sha256': '921cd60effec637559be1b3faaf918033defbe16d5773f9a41e5ace743f27d69'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/props/x/values/zarr.json', 'sha256': '54283d123f5d358fd2fef87703f062f9bc58ba30605d08bdb736ff90e5f96961'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/props/x/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/props/y/values/c/0', 'sha256': '53ecca15c5fd72cbb2455a4e9c2881a73972f8aa9944534d60e6fbcca2f30550'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/props/y/values/zarr.json', 'sha256': '54283d123f5d358fd2fef87703f062f9bc58ba30605d08bdb736ff90e5f96961'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/props/y/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/props/z/values/c/0', 'sha256': '9378d093070459445001f8db1ee25d1018535858ef797194f601f8dabb668601'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/props/z/values/zarr.json', 'sha256': '54283d123f5d358fd2fef87703f062f9bc58ba30605d08bdb736ff90e5f96961'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/props/z/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/props/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/nodes/zarr.json', 'sha256': '36615682d5be9646729635067a75185212eb0c48d708ef06ad34c1f58dd587d2'}, {'name': 'tracking_repo/predictions/unknown/unet_transformer_val/split_0/6bba_09961292.geff/zarr.json', 'sha256': '1b01959ceae67ebdb93a6af40f1a7de5943f54b2e7eaab094d0bf81a6efdee11'}]
EXPECTED_RAW_HASHES={'44b6_12dfb391': '56492546075c6dd74b1db00fbbaa6ef08c21be91253cfbfb2104a50e26b38df1', '44b6_267148e4': '19a229b559de172092053e3854ec4ea20fc085c906ba3f9acba315d8663ed13c', '44b6_2a2eff9f': '48543a15900eee3d5e92591f3b37f596a5c10ef47712b8310aaed73d2a7d2d88', '44b6_341df25f': 'a30e4ca37b57986e1438e8900df8f3328ce0bbc61ac87c28ef72eedc616c4891', '6bba_062c8d37': '6e5c00aa56e4f90d4359f20cf2c3bbe068f64b91fef268b7406fa18c7fc43e19', '6bba_07e24132': 'f1fb452270c57d9ece3bbaef7725601d496b512f3ba382456c45ed949c4d82ab', '6bba_085bf656': 'db3436921f7e0731bb5aa54e63fb0b3648c27b7a1cd29246c69a445fee2843f0', '6bba_09961292': '7d0c4cb79a057c1ad3806ed8b27c884f8cede2d7f1df25d3de903e87070608bd'}
"""Kaggle-only paired postprocessing of fixed pre-motion upstream graphs."""
import copy, collections, hashlib, json, time, resource, importlib.metadata
assert Path('/kaggle/input').is_dir(), 'CLOUD_ONLY'
ART=WORKING_DIR/'f1_small';ART.mkdir(exist_ok=True)
CACHE=WORKING_DIR/'f1_cache';CACHE.mkdir(exist_ok=True)
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
def gh(n,e):return hashlib.sha256(json.dumps([list(n.items()),e],allow_nan=False,separators=(',',':')).encode()).hexdigest()
def clean(x):
 if isinstance(x,dict):return {str(k):clean(v) for k,v in x.items()}
 if isinstance(x,(list,tuple)):return [clean(v) for v in x]
 if isinstance(x,np.generic):return clean(x.item())
 if isinstance(x,float) and not math.isfinite(x):return None
 return x
def save(name,x):(ART/name).write_text(json.dumps(clean(x),ensure_ascii=False,indent=2,allow_nan=False)+'\n')
def es(e):return {(int(r['source_id']),int(r['target_id'])) for r in e}
def valid(n,e):
 inc=collections.Counter();out=collections.Counter()
 assert all(np.isfinite([r['z'],r['y'],r['x']]).all() and r['node_id']==i for i,r in n.items())
 assert len(es(e))==len(e),'DUPLICATE_EDGE'
 for s,t in es(e):
  assert s in n and t in n and int(n[t]['t'])==int(n[s]['t'])+1,'EDGE_ENDPOINT_OR_TIME'
  inc[t]+=1;out[s]+=1
 assert max(inc.values(),default=0)<=1 and max(out.values(),default=0)<=2,'GRAPH_DEGREE'

TEST_DIR=COMP_DIR/'train';TRAIN_DIR=TEST_DIR
globals().update(FROZEN_CONFIG)
assert _sprint_model_key(FIXED_STEMS[0])=='44b6'
assert _sprint_model_key(FIXED_STEMS[-1])=='6bba'
source_root=_sprint_paths[0].parent
assert torch.cuda.is_available(),'GPU_REQUIRED'
torch.cuda.reset_peak_memory_stats()
versions={n:importlib.metadata.version(n) for n in ['tracksdata','polars','numpy','scipy','torch','geff','zarr']}
schema={k:getattr(td.DEFAULT_ATTR_KEYS,k) for k in ['NODE_ID','EDGE_SOURCE','EDGE_TARGET','T']}
assert schema==dict(NODE_ID='node_id',EDGE_SOURCE='source_id',EDGE_TARGET='target_id',T='t'),schema
modules={n:sha(Path(__import__(n).__file__)) for n in ['tracksdata','polars','numpy','scipy']}
save('runtime_start.json',dict(versions=versions,module_hashes=modules,schema=schema,gpu=torch.cuda.get_device_name(),
    free_disk_bytes=shutil.disk_usage(WORKING_DIR).free,config=FROZEN_CONFIG,flow_config=F1_CONFIG,
    weight_sha256=_runtime_integrity_receipt['checkpoint_sha256'],division_sha256=_SPRINT_WEIGHT_SHA,
    scorer_hashes=_official_file_hashes,training_calls=0,upstream_inference_calls=0))

# Verify every pre-motion graph file against the already archived cloud output manifest.
checked=[]
for item in RAW_MANIFEST:
 p=source_root/item['name'];assert p.is_file(),('CACHE_FILE_MISSING',item['name'])
 assert sha(p)==item['sha256'],('CACHE_HASH_MISMATCH',item['name'])
 checked.append({'name':item['name'],'sha256':item['sha256']})
save('input_hashes.json',checked)

# DeepCenter heatmaps are shared on disk without keeping all frames in memory.
original_heatmap=deepcenter_heatmap_for_frame
heatmap_computes=heatmap_reads=0
def cached_heatmap(dataset,t,bundle,frames,cache):
 global heatmap_computes,heatmap_reads
 p=CACHE/f'{dataset}_{t}_deepcenter.npy'
 if p.exists():heatmap_reads+=1;return np.load(p,allow_pickle=False)
 h=original_heatmap(dataset,t,bundle,frames,cache)
 assert h is not None,'MISSING_HEATMAP'
 np.save(p,h,allow_pickle=False);heatmap_computes+=1;return h
deepcenter_heatmap_for_frame=cached_heatmap
original_refine=refine_synthetic_midpoint;refine_cache={}
def cached_refine(dataset,t,midpoint,frames,stats):
 key=(dataset,int(t),tuple(midpoint))
 if key not in refine_cache:
  before=dict(stats);value=original_refine(dataset,t,midpoint,frames,stats)
  refine_cache[key]=(value,{k:v-before.get(k,0) for k,v in stats.items() if v!=before.get(k,0)})
  return value
 value,delta=refine_cache[key]
 for k,v in delta.items():stats[k]=stats.get(k,0)+v
 return value
refine_synthetic_midpoint=cached_refine

ROWS={a:[] for a in ['B0','B0_repeat','F1_off','F1']};RECEIPTS=[];EVENTS=[];USAGE=[];MOTION={}
f1_motion=motion_relink_edges
def capture_motion(n,stats,probs=None):
 result=(F1_ORIGINAL_MOTION if ACTIVE_ARM in ['B0','B0_repeat'] else f1_motion)(n,stats,probs)
 MOTION[ACTIVE_ARM]=copy.deepcopy(result)
 return result
motion_relink_edges=capture_motion
for index,stem in enumerate(FIXED_STEMS):
 start=time.monotonic();refine_cache.clear()
 paths=list((source_root/'tracking_repo/predictions').rglob(stem+'.geff'))
 assert len(paths)==1,('CACHE_STAGE_MISSING',stem)
 raw=graph_from_geff(paths[0]);rn=raw.node_attrs();re=raw.edge_attrs()
 assert {'node_id','t','z','y','x'}<=set(rn.columns)
 assert {'source_id','target_id','edge_prob'}<=set(re.columns)
 nodes={int(r['node_id']):{k:int(r[k]) if k in ['node_id','t'] else float(r[k]) for k in ['node_id','t','z','y','x']} for r in rn.iter_rows(named=True)}
 edges=[dict(source_id=int(r['source_id']),target_id=int(r['target_id']),edge_prob=None if r['edge_prob'] is None else float(r['edge_prob'])) for r in re.iter_rows(named=True)]
 raw_hash=gh(nodes,edges);assert raw_hash==EXPECTED_RAW_HASHES[stem],('RAW_STAGE_MISMATCH',stem)
 gtpath=TRAIN_DIR/(stem+'.geff');gt=graph_from_geff(gtpath);gn,ge=graph_to_plain(gt)
 total=read_estimated_true_node_count(gtpath);assert total is not None
 outputs={};statuses={};hashes={};motion_hashes={}
 for arm in ROWS:
  ACTIVE_ARM=arm;F1_ENABLED=arm=='F1';F1_CALLS.clear()
  before=time.monotonic()
  n,e,stats=filter_output_graph(copy.deepcopy(nodes),copy.deepcopy(edges),dataset=stem,deepcenter_bundle=DEEPCENTER_VETO_DETECTOR)
  post_seconds=time.monotonic()-before;valid(n,e)
  assert gh(nodes,edges)==raw_hash,'SHARED_INPUT_MUTATED'
  score_start=time.monotonic()
  row=score_sample(nodes_by_id_to_plain(n),[(r['source_id'],r['target_id']) for r in e],gn,ge,total)
  row.update(stem=stem,embryo=stem.split('_')[0],arm=arm)
  row['score']=aggregate_official([row])['score']
  ROWS[arm].append(row);statuses[arm]=dict(F1_EDGE_STATUS);outputs[arm]=(n,e)
  hashes[arm]=gh(n,e);motion_hashes[arm]=gh(nodes,MOTION[arm])
  RECEIPTS.append(dict(stem=stem,arm=arm,raw_hash=raw_hash,final_hash=hashes[arm],motion_hash=motion_hashes[arm],
      post_seconds=post_seconds,score_seconds=time.monotonic()-score_start,stats=stats,
      nodes=len(n),edges=len(e),classifier_model=_sprint_model_key(stem)))
  if arm=='F1':
   for call in F1_CALLS:
    for usage in call['frames']:USAGE.append(dict(stem=stem,**usage))
  (CACHE/f'{stem}_{arm}_graph.json').write_text(json.dumps(dict(nodes=list(n.items()),edges=e),allow_nan=False))
 assert hashes['B0']==hashes['B0_repeat']==hashes['F1_off'],'REPEAT_OR_OFF_MISMATCH'
 assert abs(ROWS['B0'][-1]['score']-ROWS['B0_repeat'][-1]['score'])<=1e-10
 for stage in ['motion','final']:
  a=MOTION['B0'] if stage=='motion' else outputs['B0'][1]
  b=MOTION['F1'] if stage=='motion' else outputs['F1'][1]
  for kind,changed,arm in [('added',es(b)-es(a),'F1'),('removed',es(a)-es(b),'B0')]:
   for s,t in sorted(changed):EVENTS.append(dict(stem=stem,stage=stage,change=kind,source_id=s,target_id=t,
       attribution=statuses[arm].get((s,t),'UNKNOWN') if stage=='final' else 'UNKNOWN',
       attribution_scope='official final matched-edge mask and sparse pred_valid' if stage=='final' else 'motion stage not independently scored'))
 save('progress.json',dict(completed=index+1,total=8,stem=stem,elapsed_seconds=time.monotonic()-start))
 print('F1_SAMPLE_COMPLETE',stem,ROWS['B0'][-1]['score'],ROWS['F1'][-1]['score'],flush=True)

summary={a:aggregate_official(rows) for a,rows in ROWS.items()}
groups={a:{g:aggregate_official([r for r in rows if r['embryo']==g]) for g in ['44b6','6bba']} for a,rows in ROWS.items()}
delta={k:summary['F1'][k]-summary['B0'][k] for k in ['score','adj_edge_jaccard','division_jaccard','edge_tp','edge_fp','edge_fn','division_tp','division_fp','division_fn']}
group_delta={g:groups['F1'][g]['score']-groups['B0'][g]['score'] for g in ['44b6','6bba']}
changed=any(r['final_hash']!=next(z['final_hash'] for z in RECEIPTS if z['stem']==r['stem'] and z['arm']=='B0') for r in RECEIPTS if r['arm']=='F1')
dominated=delta['score'] < -1e-10 and all(d < -1e-10 for d in group_delta.values()) and delta['adj_edge_jaccard']<=1e-10 and delta['division_jaccard']<=1e-10
status='NO_EFFECT' if not changed else 'DIAGNOSTIC_DOMINATED' if dominated else 'DIAGNOSTIC_FAVORABLE' if delta['score']>1e-10 and all(d>=-1e-10 for d in group_delta.values()) else 'DIAGNOSTIC_MIXED_EXPLORATORY'
assert sum(r['flow_nodes'] for r in USAGE)>0,'SILENT_ALL_FALLBACK'
save('results.json',dict(status=status,summary=summary,by_embryo=groups,delta=delta,embryo_delta=group_delta,
    graph_changed=changed,production_allowed=status in ['DIAGNOSTIC_FAVORABLE','DIAGNOSTIC_MIXED_EXPLORATORY'],
    repeat_and_off_equal=True,samples=FIXED_STEMS,training_calls=0,upstream_inference_calls=0,
    heatmap_computes=heatmap_computes,heatmap_reads=heatmap_reads,
    peak_gpu_allocated_bytes=torch.cuda.max_memory_allocated(),peak_memory_kib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
    full_pipeline_independent_validation=False))
save('stages.json',RECEIPTS)
pd.DataFrame([r for rows in ROWS.values() for r in rows]).to_csv(ART/'per_view.csv',index=False)
pd.DataFrame(EVENTS).to_csv(ART/'edge_changes.csv',index=False)
pd.DataFrame(USAGE).to_csv(ART/'flow_usage.csv',index=False)
print('F1_DIAGNOSTIC_COMPLETE',status,delta,flush=True)
